# Análisis Bivariado

Se examinan las relaciones y asociaciones entre pares de variables, permitiendo identificar cómo se comporta una variable en función de otra y detectar patrones conjuntos que no son evidentes en el análisis univariado. A través de la comparación cruzada de categorías y la exploración de tendencias, este enfoque facilita la comprensión de las interacciones presentes en los datos, revelando dependencias, correlaciones y posibles relaciones de causa-efecto.

## Carga de Librerías

Se importan las librerías necesarias para el análisis bivariado de los datos. Se emplean `pandas` y `numpy` para la manipulación y cálculo de estadísticos; `matplotlib.pyplot`, `seaborn` y `plotly.graph_objects` para la generación de gráficos estáticos e interactivos; y `scikit_posthocs` para pruebas post-hoc en análisis no paramétricos. Adicionalmente, se incorporan `scipy.stats` para pruebas estadísticas complementarias y `statsmodels.stats.multicomp.pairwise_tukeyhsd` para análisis post-hoc paramétricos. Finalmente, se suprimen los mensajes de advertencia (`warnings`) con el fin de mantener una salida limpia y centrada en los resultados del análisis.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import scikit_posthocs as sp

from IPython.display import display
from IPython.display import HTML
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [3]:
pio.renderers.default = "notebook_connected"

## Carga del DataFrame

Se carga el archivo CSV que contiene los datos de comparendos electrónicos utilizando la función `read_csv` de la librería pandas, y se almacena en el DataFrame `df_comparendos_electronicos`.

In [4]:
df_comparendos_electronicos = pd.read_csv("C:/Users/david/Documents/seminario_investigativo/comparendos_electronicos.csv")

### Conversión de las Fechas a Formato datetime

Se convierte la columna `fecha_comparendo` al tipo `datetime` utilizando el formato especificado (`'%Y %b %d %I:%M:%S %p'`), normalizando la hora a medianoche con el método `.dt.normalize()` para eliminar la información horaria y trabajar únicamente con fechas, dado que todos los registros contienen la hora de 12:00:00 AM. Posteriormente, se imprime el tipo de dato resultante para verificar la correcta conversión.

In [5]:
df_comparendos_electronicos['fecha_comparendo'] = pd.to_datetime(df_comparendos_electronicos['fecha_comparendo'], format='%Y %b %d %I:%M:%S %p').dt.normalize()

print(df_comparendos_electronicos['fecha_comparendo'].dtype)

datetime64[ns]


### Corrección de Valores Nulos

Se rellenan los valores nulos de la columna `DESC_INFRACCION` con la descripción correspondiente al código C14, obtenida del sitio web oficial del Tránsito del Atlántico. Según esta fuente, la infracción C14 corresponde a **"Transitar por sitios restringidos o en horas prohibidas por la autoridad competente"**. Esta corrección se aplica exclusivamente a los 564 registros que presentaban valores nulos, asociados a las nuevas cámaras tipo Carril Bus implementadas en Barranquilla a partir del 17 de octubre de 2025.

**Fuente:** https://transitodelatlantico.gov.co/valor-de-multas-de-transito/

In [6]:
df_comparendos_electronicos['DESC_INFRACCION'] = df_comparendos_electronicos['DESC_INFRACCION'].fillna("Transitar por sitios restringidos o en horas prohibidas por la autoridad competente")  

print(f"Total de valores nulos en el DataFrame: {df_comparendos_electronicos.isnull().sum().sum()}")

Total de valores nulos en el DataFrame: 0


### Corrección de Cámaras Duplicadas

Se unifican los nombres de las cámaras que presentan dos notaciones diferentes para el mismo punto de control. Esta corrección permite evitar la duplicación de ubicaciones en los análisis y garantizar la consistencia de los datos.

In [7]:
df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 53 CON CALLE 104', 'Camara_y_direccion'] = 'CARRERA 53 ENTRE CALLE 104 Y 106'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 84 CON CARRERA 59', 'Camara_y_direccion'] = 'CALLE 84 ENTRE CARRERA 59 Y 59B'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CARRERA 45 CON CALLE 53', 'Camara_y_direccion'] = 'CALLE 53 CON CARRERA 45'

df_comparendos_electronicos.loc[df_comparendos_electronicos['Camara_y_direccion'] == 'CALLE 45B CARRERA 14', 'Camara_y_direccion'] = 'CALLE 45B CON CARRERA 14'

## Análisis del Código de Infracción Según el Tipo de Cámara

Se construye una tabla pivote que cruza los códigos de infracción con los tipos de cámara, utilizando la suma ponderada de `CANTIDAD_INFRACCIONES` para visualizar el volumen de infracciones en cada combinación. Para cada código de infracción, se calcula el porcentaje que representa cada tipo de cámara sobre el total de sus comparendos, permitiendo identificar la contribución relativa de cada tecnología. Los resultados se presentan mediante un mapa de calor interactivo, donde los códigos de infracción se ordenan de mayor a menor según su total de comparendos, y cada celda muestra tanto el valor absoluto como el porcentaje correspondiente. Esta visualización permite identificar qué tipos de cámara son más efectivos o están más especializados en la detección de cada infracción específica, revelando patrones de asignación tecnológica según el tipo de falta cometida.

In [8]:
tabla_heatmap = df_comparendos_electronicos.pivot_table(
    values='CANTIDAD_INFRACCIONES',
    index='COD_INFRACCION',
    columns='Tipo Camara',
    aggfunc='sum',
    fill_value=0
)

orden = tabla_heatmap.sum(axis=1).sort_values(ascending=True).index
tabla_heatmap = tabla_heatmap.loc[orden]

total_por_fila = tabla_heatmap.sum(axis=1)

porcentajes = tabla_heatmap.div(total_por_fila, axis=0) * 100

porcentajes = porcentajes.round(2)

texto_combinado = tabla_heatmap.applymap(lambda x: f'{int(x):,}') + '<br>(' + porcentajes.applymap(lambda x: f'{x:.2f}%') + ')'

fig = go.Figure(data=go.Heatmap(
    z=tabla_heatmap.values,
    x=tabla_heatmap.columns,
    y=tabla_heatmap.index,
    colorscale='Blues',
    text=texto_combinado.values,
    texttemplate='%{text}',
    hovertemplate='Código: %{y}<br>Tipo de Cámara: %{x}<br>Comparendos: %{z:,.0f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Distribución de Comparendos Electrónicos por Código de Infracción y Tipo de Cámara',
        x=0.5,
        xanchor='center',
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(
        text='Tipo de Cámara',
        font=dict(weight='bold')
    ),
    yaxis_title=dict(
        text='Código de Infracción',
        font=dict(weight='bold')
    ),
    template='plotly_white',
    width=1055, 
    height=500
)

fig.show()

- **C29 - Exceso de velocidad (284,755 infracciones)**: Detectado exclusivamente por **cámaras fijas**. Este patrón indica que el control de velocidad se realiza mediante equipos instalados en puntos estratégicos fijos, no con cámaras móviles o de carril bus.

- **C02 - Estacionamiento en sitios prohibidos (204,975 infracciones)**: Detectado exclusivamente por **cámaras fijas** y **cámaras móviles**. La distribución muestra una participación marginal de los equipos fijos (2), por lo que estos 2 casos serán analizados para verificar si se tratan de casos reales, pruebas realizadas en el sistema de cámaras o un error de digitación.

- **C03 - Bloquear calzada o intersección (65,230 infracciones)**: Detectado exclusivamente por **cámaras fijas**. Además, no se registran detecciones de esta infracción con cámaras móviles o de carril bus.

- **D04 - No detenerse ante un semáforo o una señal de «pare» (54,303 infracciones)**: Detectado exclusivamente por **cámaras fijas**. Dado que, el control de semáforos en rojo requiere equipos instalados permanentemente en las intersecciones.

- **C32 - No respetar paso de peatones (3,369 infracciones)**: Detectado exclusivamente por **cámaras fijas**. Presenta un volumen significativamente menor en comparación con las infracciones anteriores.

- **C14 - Transitar por sitios restringidos (1,730 infracciones)**: Detectado exclusivamente por **cámaras tipo Carril Bus**. Este patrón es consistente con la naturaleza de esta infracción, que se comete al transitar por carriles restringidos para buses, y explica por qué solo este tipo de cámara la detecta.

---

**Hallazgos:**

- **Las cámaras fijas son las más versátiles**: Detectan 5 de los 6 códigos de infracción presentes en la base de datos. Su capacidad para controlar múltiples tipos de infracciones las convierte en el pilar del sistema de vigilancia.

- **Las cámaras móviles tienen un rol especializado**: Se enfocan exclusivamente en la detección de estacionamiento prohibido (C02). Esto sugiere que las unidades móviles recorren la ciudad específicamente para controlar el mal parqueo.

- **Las cámaras Carril Bus tienen una función única**: Detectan únicamente la infracción C14 (transitar por sitios restringidos), lo cual es coherente con su ubicación en corredores exclusivos para buses. Su baja participación se explica por su implementación reciente.

- **No hay solapamiento para la mayoría de códigos**: Excepto por el C02, que es detectado por cámaras fijas y móviles, el resto de las infracciones son detectadas por un solo tipo de cámara, lo que indica una clara estrategia de asignación de tecnología según el tipo de infracción.

**Nota:** La dependencia casi exclusiva de cámaras fijas para la mayoría de las infracciones sugiere que cualquier falla o mantenimiento en este tipo de equipos afectaría significativamente la capacidad de detección del sistema. Por otro lado, la especialización de las cámaras móviles y Carril Bus las hace críticas para sus respectivos nichos de control.

### Corrección de Infracciones C02 Detectadas por Cámaras Fijas

Se filtran y muestran los registros correspondientes a la infracción C02 (estacionamiento en sitios prohibidos) que fueron detectados específicamente por cámaras fijas, con el objetivo de identificar las causas de estos casos atípicos y examinar sus características particulares. Dado que estos registros representan una inconsistencia en la base de datos (las cámaras fijas no están diseñadas para detectar estacionamiento prohibido), se procede a eliminarlos del DataFrame principal para garantizar la consistencia de los análisis posteriores. Posteriormente, se verifica que no queden registros residuales de C02 asociados a cámaras fijas, confirmando la correcta limpieza de los datos.

In [9]:
df_c02_fijo = df_comparendos_electronicos[(df_comparendos_electronicos['COD_INFRACCION'] == 'C02') & (df_comparendos_electronicos['Tipo Camara'] == 'Fijo')]

display(df_c02_fijo)

,fecha_comparendo,COD_INFRACCION,DESC_INFRACCION,TIPO_INFRACCION,SERVICIO_VEHICULO_INFRACTOR,CLASE_VEHICULO_INFRACTOR,CANTIDAD_INFRACCIONES,Tipo Camara,Camara_y_direccion
233307,2023-08-28,C02,Estacionar un vehículo en los sitios prohibidos,TRÁNSITO,PUBLICO,AUTOMOVIL,1,Fijo,CALLE 72 CON CARRERA 44
260358,2024-02-28,C02,Estacionar un vehículo en los sitios prohibidos,TRÁNSITO,PARTICULAR,AUTOMOVIL,1,Fijo,CALLE 82 CON CARRERA 51B


- **Ausencia de patrón común**: Los dos casos no presentan similitudes entre sí en cuanto a fecha (diferencia de 6 meses), servicio del vehículo (uno público, otro particular), ni ubicación (calles diferentes).

- **Volumen marginal**: Solo 2 casos sobre 204,975 infracciones C02 son detectados por cámaras fijas, una proporción estadísticamente insignificante.

- **Inconsistencia operativa**: Dado que las cámaras fijas están diseñadas principalmente para detectar exceso de velocidad (C29), paso de semáforos en rojo (D04) y otras infracciones de tránsito, no para estacionamiento prohibido, lo esperable sería que el C02 fuera detectado exclusivamente por cámaras móviles, tal como se observa en el resto del dataset.

**Nota:** La presencia de estos 2 casos muy probablemente corresponde a **errores de digitación o registro** en la base de datos, ya que no existe una justificación técnica clara para que una cámara fija detecte estacionamiento prohibido, pues este tipo de infracción requiere vigilancia móvil que recorra diferentes zonas. Por lo tanto, se procede a eliminar estos registros.

In [10]:
df_comparendos_electronicos = df_comparendos_electronicos[~((df_comparendos_electronicos['COD_INFRACCION'] == 'C02') & 
                                                             (df_comparendos_electronicos['Tipo Camara'] == 'Fijo'))]

df_c02_fijo_verificacion = df_comparendos_electronicos[(df_comparendos_electronicos['COD_INFRACCION'] == 'C02') & 
                                                        (df_comparendos_electronicos['Tipo Camara'] == 'Fijo')]

print(f"Registro de infracciones C02 por cámara fija: {len(df_c02_fijo_verificacion)}")

Registro de infracciones C02 por cámara fija: 0


## Análisis del Servicio del Vehículo Infractor Según el Tipo de Cámara

Se construye una tabla pivote que cruza el servicio del vehículo infractor con los tipos de cámara, agrupando las categorías con baja representación (OFICIAL y DIPLOMÁTICO) en una categoría denominada "OTROS" para facilitar la visualización. Se utiliza la suma ponderada de `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones en cada combinación. Para cada categoría de servicio, se calcula el porcentaje que representa cada tipo de cámara sobre el total de sus comparendos, permitiendo identificar la contribución relativa de cada tecnología. Los resultados se presentan mediante un mapa de calor interactivo, donde las categorías de servicio se ordenan de mayor a menor según su total de comparendos, y cada celda muestra tanto el valor absoluto como el porcentaje correspondiente. Esta visualización permite identificar qué tipos de cámara son más utilizados para detectar infracciones en cada categoría de servicio vehicular, revelando patrones de asignación tecnológica según el tipo de servicio del vehículo infractor.

In [11]:
df_comparendos_electronicos_copy = df_comparendos_electronicos.copy()

df_comparendos_electronicos_copy['SERVICIO_AGRUPADO'] = df_comparendos_electronicos_copy['SERVICIO_VEHICULO_INFRACTOR'].replace(['OFICIAL', 'DIPLOMATICO'], 'OTROS')

tabla_heatmap = df_comparendos_electronicos_copy.pivot_table(
    values='CANTIDAD_INFRACCIONES',
    index='SERVICIO_AGRUPADO',
    columns='Tipo Camara',
    aggfunc='sum',
    fill_value=0
)

orden = tabla_heatmap.sum(axis=1).sort_values(ascending=True).index
tabla_heatmap = tabla_heatmap.loc[orden]

total_por_fila = tabla_heatmap.sum(axis=1)
porcentajes = tabla_heatmap.div(total_por_fila, axis=0) * 100
porcentajes = porcentajes.round(2)
texto_combinado = tabla_heatmap.applymap(lambda x: f'{int(x):,}') + '<br>(' + porcentajes.applymap(lambda x: f'{x:.2f}%') + ')'

fig = go.Figure(data=go.Heatmap(
    z=tabla_heatmap.values,
    x=tabla_heatmap.columns,
    y=tabla_heatmap.index,
    colorscale='Blues',
    text=texto_combinado.values,
    texttemplate='%{text}',
    hovertemplate='Servicio: %{y}<br>Tipo de Cámara: %{x}<br>Comparendos: %{z:,.0f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Distribución de Comparendos Electrónicos por Servicio del Vehículo y Tipo de Cámara',
        x=0.5,
        xanchor='center',
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(
        text='Tipo de Cámara',
        font=dict(weight='bold')
    ),
    yaxis_title=dict(
        text='Servicio del Vehículo',
        font=dict(weight='bold')
    ),
    template='plotly_white',
    width=1055, 
    height=500
)

fig.show()

**Vehículos particulares (525,986 infracciones)**:
   - **Cámaras fijas**: 338,641 infracciones (64.38% de las infracciones de particulares).
   - **Cámaras móviles**: 186,359 infracciones (35.43% de las infracciones de particulares).
   - **Carril Bus**: 986 infracciones (0.19% de las infracciones de particulares).
   - Los vehículos particulares son detectados mayoritariamente por cámaras fijas, aunque las móviles también tienen una participación significativa.

**Vehículos de servicio público (87,553 infracciones)**:
   - **Cámaras fijas**: 68,604 infracciones (78.36% de las infracciones de públicos).
   - **Cámaras móviles**: 18,225 infracciones (20.82% de las infracciones de públicos).
   - **Carril Bus**: 724 infracciones (0.83% de las infracciones de públicos).
   - Al igual que los particulares, predominan las detecciones por cámaras fijas, pero con una proporción aún más alta.

**Otros servicios (oficial y diplomático) (821 infracciones)**:
   - **Cámaras fijas**: 412 infracciones (50.18%).
   - **Cámaras móviles**: 389 infracciones (47.38%).
   - **Carril Bus**: 20 infracciones (2.44%).
   - La distribución entre fijo y móvil es casi equitativa.

---

**Hallazgos:**

- **Predominio de cámaras fijas en todos los servicios**: Independientemente del servicio del vehículo, las cámaras fijas son el principal medio de detección.

- **Comportamiento similar entre particulares y públicos**: Ambos grupos muestran un patrón comparable en la distribución por tipo de cámara, con una clara dominancia de las cámaras fijas seguidas por las móviles.

- **Las categorías "Otros" (oficial y diplomático)** tienen un volumen tan bajo que no afectan significativamente los patrones generales, pero muestran una distribución más equilibrada entre fijo y móvil.

## Análisis de la Clase del Vehículo Infractor Según el Tipo de Cámara

Se construye una tabla pivote que cruza la clase del vehículo infractor con los tipos de cámara, agrupando las categorías de vehículos pesados (camión, bus, microbús, buseta, volqueta, tractocamión) en una sola denominación "VEHÍCULOS PESADOS", y las categorías minoritarias (motocarro, cuatrimoto, no reportado, sin clase) en "OTROS", con el fin de facilitar la visualización y el análisis comparativo. Se utiliza la suma ponderada de `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones en cada combinación. Los resultados se presentan mediante un mapa de calor interactivo, donde las categorías de clase se ordenan de mayor a menor según su total de comparendos, permitiendo identificar qué tipos de cámara son más utilizados para detectar infracciones en cada clase de vehículo, así como revelar posibles patrones de especialización tecnológica según el tipo de vehículo infractor.

In [12]:
df_comparendos_electronicos_copy['CLASE_AGRUPADA'] = df_comparendos_electronicos_copy['CLASE_VEHICULO_INFRACTOR'].replace({
    'CAMION': 'VEHÍCULOS PESADOS',
    'BUS': 'VEHÍCULOS PESADOS',
    'MICROBUS': 'VEHÍCULOS PESADOS',
    'BUSETA': 'VEHÍCULOS PESADOS',
    'VOLQUETA': 'VEHÍCULOS PESADOS',
    'TRACTOCAMION': 'VEHÍCULOS PESADOS',
    'MOTOCARRO': 'OTROS',
    'CUATRIMOTO': 'OTROS',
    'NO REPORTADO': 'OTROS',
    'SIN CLASE': 'OTROS'
})

tabla_heatmap = df_comparendos_electronicos_copy.pivot_table(
    values='CANTIDAD_INFRACCIONES',
    index='CLASE_AGRUPADA',
    columns='Tipo Camara',
    aggfunc='sum',
    fill_value=0
)

orden = tabla_heatmap.sum(axis=1).sort_values(ascending=True).index
tabla_heatmap = tabla_heatmap.loc[orden]

total_por_fila = tabla_heatmap.sum(axis=1)
porcentajes = tabla_heatmap.div(total_por_fila, axis=0) * 100
porcentajes = porcentajes.round(2)
texto_combinado = tabla_heatmap.applymap(lambda x: f'{int(x):,}') + '<br>(' + porcentajes.applymap(lambda x: f'{x:.2f}%') + ')'

fig = go.Figure(data=go.Heatmap(
    z=tabla_heatmap.values,
    x=tabla_heatmap.columns,
    y=tabla_heatmap.index,
    colorscale='Blues',
    text=texto_combinado.values,
    texttemplate='%{text}',
    hovertemplate='Clase: %{y}<br>Tipo de Cámara: %{x}<br>Comparendos: %{z:,.0f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Distribución de Comparendos Electrónicos por Clase del Vehículo y Tipo de Cámara',
        x=0.5,
        xanchor='center',
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(
        text='Tipo de Cámara',
        font=dict(weight='bold')
    ),
    yaxis_title=dict(
        text='Clase del Vehículo',
        font=dict(weight='bold')
    ),
    template='plotly_white',
    width=1055, 
    height=500
)

fig.show()

**AUTOMÓVIL (321,953 infracciones)**:
- **Cámaras fijas**: 187,485 (58.23% de sus infracciones).
- **Cámaras móviles**: 133,167 (41.36% de sus infracciones).
- **Carril Bus**: 1,301 (0.40% de sus infracciones).
- Los automóviles son detectados principalmente por cámaras fijas, aunque las móviles tienen una participación muy significativa (casi 42%).

**CAMIONETA (129,827 infracciones)**:
- **Cámaras fijas**: 76,524 (58.94% de sus infracciones).
- **Cámaras móviles**: 52,974 (40.80% de sus infracciones).
- **Carril Bus**: 329 (0.25% de sus infracciones)
- Presenta un patrón similar al de los automóviles, con dominancia de cámaras fijas.

**MOTOCICLETA (102,708 infracciones)**:
- **Cámaras fijas**: 102,708 (100% de sus infracciones)
- **Cámaras móviles**: 0
- **Carril Bus**: 0
   - **Las motocicletas solo son detectadas por cámaras fijas**. Este hallazgo es relevante, ya que sugiere que las cámaras móviles no registran infracciones cometidas por motocicletas, o que estas no circulan por las zonas donde operan las patrullas móviles.

**CAMPERO (39,955 infracciones)**:
- **Cámaras fijas**: 24,289 (60.79% de sus infracciones)
- **Cámaras móviles**: 15,581 (39.00% de sus infracciones)
- **Carril Bus**: 85 (0.21% de sus infracciones)
- Patrón similar a automóviles y camionetas.

**VEHÍCULOS PESADOS (19,736 infracciones)**:
- **Cámaras fijas**: 16,479 (83.50% de sus infracciones)
- **Cámaras móviles**: 3,243 (16.43% de sus infracciones)
- **Carril Bus**: 14 (0.07% de sus infracciones)

**OTROS (181 infracciones)**:
- **Cámaras fijas**: 172 (95.03%)
- **Cámaras móviles**: 8 (4.42%)
- **Carril Bus**: 1 (0.55%)
- Este grupo tiene la mayor proporción de detecciones por cámaras fijas (95.03%).

---

**Hallazgos:**

- **Las motocicletas son detectadas exclusivamente por cámaras fijas**: Este es el hallazgo más relevante del análisis. Las 102,708 infracciones de motocicletas fueron registradas únicamente por cámaras fijas.

- **Automóviles y camionetas tienen una distribución similar**: Ambos presentan una relación aproximada de 60% fijo y 40% móvil, lo que indica que son los principales objetivos tanto de las cámaras fijas como de las patrullas móviles.

- **Especialización tecnológica clara**: Las cámaras fijas son el único medio para detectar infracciones de motocicletas.

- **Los vehículos pesados son controlados mayoritariamente por cámaras fijas**: Su alta proporción (87%) sugiere que circulan principalmente por vías principales equipadas con esta tecnología.

## Análisis de la Clasificación de Cámara Según el Tipo de Cámara

Se clasifican las cámaras en dos categorías según su nombre: "Mal Parqueo" para aquellas cuya denominación contiene la palabra "mal" (refiriéndose a las cámaras especializadas en detección de estacionamiento prohibido), y "Con Dirección" para el resto de las cámaras identificadas por su ubicación y dirección específica. Se construye una tabla pivote que cruza esta clasificación con el tipo de cámara (Fijo, Móvil, Carril Bus), utilizando la suma ponderada de `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones en cada combinación. Para cada clasificación, se calcula el porcentaje que representa cada tipo de cámara sobre el total de sus comparendos. Los resultados se presentan mediante un mapa de calor interactivo donde cada celda muestra tanto el valor absoluto como el porcentaje correspondiente, permitiendo identificar qué tipos de cámara predominan en cada categoría y revelar patrones de especialización tecnológica entre las cámaras dedicadas al mal parqueo y aquellas con dirección específica.

In [13]:
df_comparendos_electronicos_copy['Clasificacion_Camara'] = df_comparendos_electronicos_copy['Camara_y_direccion'].apply(
    lambda x: 'Mal Parqueo' if 'mal' in str(x).lower() else 'Con Dirección'
)

tabla_heatmap = df_comparendos_electronicos_copy.pivot_table(
    values='CANTIDAD_INFRACCIONES',
    index='Clasificacion_Camara',
    columns='Tipo Camara',
    aggfunc='sum',
    fill_value=0
)

orden = tabla_heatmap.sum(axis=1).sort_values(ascending=True).index
tabla_heatmap = tabla_heatmap.loc[orden]

total_por_fila = tabla_heatmap.sum(axis=1)
porcentajes = tabla_heatmap.div(total_por_fila, axis=0) * 100
porcentajes = porcentajes.round(2)
texto_combinado = tabla_heatmap.applymap(lambda x: f'{int(x):,}') + '<br>(' + porcentajes.applymap(lambda x: f'{x:.2f}%') + ')'

fig = go.Figure(data=go.Heatmap(
    z=tabla_heatmap.values,
    x=tabla_heatmap.columns,
    y=tabla_heatmap.index,
    colorscale='Blues',
    text=texto_combinado.values,
    texttemplate='%{text}',
    hovertemplate='Clasificación: %{y}<br>Tipo de Cámara: %{x}<br>Comparendos: %{z:,.0f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Distribución de Comparendos Electrónicos por Clasificación de Cámara y Tipo de Cámara',
        x=0.5,
        xanchor='center',
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(
        text='Tipo de Cámara',
        font=dict(weight='bold')
    ),
    yaxis_title=dict(
        text='Clasificación de Cámara',
        font=dict(weight='bold')
    ),
    template='plotly_white',
    width=1055, 
    height=500
)

fig.show()

**Cámaras "Con Dirección" (409,387 infracciones - 66.63% del total)**:
- **Cámaras fijas**: 407,657 infracciones (99.58% de esta clasificación).
- **Cámaras Carril Bus**: 1,730 infracciones (0.42% de esta clasificación).
- **Cámaras móviles**: 0 infracciones.
- Las cámaras con dirección (identificadas por su ubicación específica como "CALLE X CON CARRERA Y") son abrumadoramente de tipo fijo, con una presencia mínima de cámaras Carril Bus.

**Cámaras "Mal Parqueo" (204,973 infracciones - 33.37% del total)**:
- **Cámaras móviles**: 204,973 infracciones (100% de esta clasificación)
- **Cámaras fijas**: 0 infracciones
- **Cámaras Carril Bus**: 0 infracciones
- Todas las cámaras denominadas "Carro mal parqueo" son exclusivamente de tipo móvil.

---

**Hallazgos:**

-  **Especialización perfecta**: Existe una separación total entre ambas clasificaciones. Las cámaras de mal parqueo son 100% móviles, mientras que las cámaras con dirección son casi exclusivamente fijas (99.58%).

- **Confirmación de hallazgos previos**: Este resultado valida y complementa los análisis anteriores donde se observó que:
   - La infracción C02 (estacionamiento prohibido) era detectada exclusivamente por cámaras móviles.
   - Las cámaras con dirección (fijas) detectan principalmente exceso de velocidad (C29), paso de semáforos en rojo o señal de «pare» (D04) y otras infracciones.

- **Rol de las cámaras Carril Bus**: Aunque clasificadas como "Con Dirección" (porque sus nombres contienen ubicaciones específicas como "CALLE X CON CARRERA Y"), representan solo el 0.42% de esta categoría, lo cual es consistente con su implementación reciente.

-  **Consistencia del dato**: La alta consistencia observada (99.58% y 100%) indica que la base de datos es confiable en cuanto a la asignación de tipo de cámara según su función, lo que valida los análisis previos que dependen de esta relación.

**Nota:**

- El mantenimiento y operación de las cámaras móviles es crítico para el control del estacionamiento indebido.
- Las cámaras fijas son insustituibles para el control de velocidad, semáforos y otras infracciones de tránsito.
- No hay redundancia entre ambos sistemas, por lo que la falla de uno no puede ser compensada por el otro.

## Distribución Geográfica de Comparendos Electrónicos por Tipo de Cámara

Se filtran y examinan por separado las cámaras de tipo fijo, móvil y Carril Bus. Para cada tipo de cámara, se calculan los porcentajes de participación de cada ubicación dentro de su categoría y se incorporan las fechas de primer y último registro, lo que permite conocer la antigüedad y continuidad operativa de cada punto de control. Los resultados se presentan mediante tablas de frecuencias que muestran, para cada tipo de cámara, las ubicaciones con sus respectivos conteos, porcentajes y períodos de operación, permitiendo identificar los puntos críticos de detección dentro de cada categoría tecnológica, comparar los patrones de distribución entre cámaras fijas, móviles y de carril bus.

In [14]:
total_fijo = df_comparendos_electronicos[df_comparendos_electronicos['Tipo Camara'] == 'Fijo']['CANTIDAD_INFRACCIONES'].sum()
total_movil = df_comparendos_electronicos[df_comparendos_electronicos['Tipo Camara'] == 'Movil']['CANTIDAD_INFRACCIONES'].sum()
total_carril = df_comparendos_electronicos[df_comparendos_electronicos['Tipo Camara'] == 'Carril Bus']['CANTIDAD_INFRACCIONES'].sum()

df_fijo = df_comparendos_electronicos[df_comparendos_electronicos['Tipo Camara'] == 'Fijo']
df_fijo_ubicaciones = df_fijo.groupby('Camara_y_direccion')['CANTIDAD_INFRACCIONES'].sum().sort_values(ascending=False).reset_index()
df_fijo_ubicaciones.columns = ['Cámara y Dirección', 'Comparendos']
df_fijo_ubicaciones['Porcentaje'] = (df_fijo_ubicaciones['Comparendos'] / total_fijo * 100).round(2)

fechas_fijo = df_fijo.groupby('Camara_y_direccion')['fecha_comparendo'].agg(['min', 'max']).reset_index()
fechas_fijo.columns = ['Cámara y Dirección', 'Primer Registro', 'Último Registro']
fechas_fijo['Primer Registro'] = pd.to_datetime(fechas_fijo['Primer Registro']).dt.date
fechas_fijo['Último Registro'] = pd.to_datetime(fechas_fijo['Último Registro']).dt.date

df_fijo_ubicaciones = df_fijo_ubicaciones.merge(fechas_fijo, on='Cámara y Dirección')

print("Distribución de Comparendos Electrónicos por Cámara Fija")
pd.set_option('display.max_columns', None)  
pd.set_option('display.width', None)        
pd.set_option('display.max_colwidth', None) 
display(df_fijo_ubicaciones.T)
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

display(HTML("<hr style='border: 2px solid #696969; margin: 20px 0;'>"))

df_movil = df_comparendos_electronicos[df_comparendos_electronicos['Tipo Camara'] == 'Movil']
df_movil_ubicaciones = df_movil.groupby('Camara_y_direccion')['CANTIDAD_INFRACCIONES'].sum().sort_values(ascending=False).reset_index()
df_movil_ubicaciones.columns = ['Cámara y Dirección', 'Comparendos']
df_movil_ubicaciones['Porcentaje'] = (df_movil_ubicaciones['Comparendos'] / total_movil * 100).round(2)

fechas_movil = df_movil.groupby('Camara_y_direccion')['fecha_comparendo'].agg(['min', 'max']).reset_index()
fechas_movil.columns = ['Cámara y Dirección', 'Primer Registro', 'Último Registro']
fechas_movil['Primer Registro'] = pd.to_datetime(fechas_movil['Primer Registro']).dt.date
fechas_movil['Último Registro'] = pd.to_datetime(fechas_movil['Último Registro']).dt.date

df_movil_ubicaciones = df_movil_ubicaciones.merge(fechas_movil, on='Cámara y Dirección')

print("Distribución de Comparendos Electrónicos por Cámara Móvil")
display(df_movil_ubicaciones.T)

display(HTML("<hr style='border: 2px solid #696969; margin: 20px 0;'>"))

df_carril = df_comparendos_electronicos[df_comparendos_electronicos['Tipo Camara'] == 'Carril Bus']
df_carril_ubicaciones = df_carril.groupby('Camara_y_direccion')['CANTIDAD_INFRACCIONES'].sum().sort_values(ascending=False).reset_index()
df_carril_ubicaciones.columns = ['Cámara y Dirección', 'Comparendos']
df_carril_ubicaciones['Porcentaje'] = (df_carril_ubicaciones['Comparendos'] / total_carril * 100).round(2)

fechas_carril = df_carril.groupby('Camara_y_direccion')['fecha_comparendo'].agg(['min', 'max']).reset_index()
fechas_carril.columns = ['Cámara y Dirección', 'Primer Registro', 'Último Registro']
fechas_carril['Primer Registro'] = pd.to_datetime(fechas_carril['Primer Registro']).dt.date
fechas_carril['Último Registro'] = pd.to_datetime(fechas_carril['Último Registro']).dt.date

df_carril_ubicaciones = df_carril_ubicaciones.merge(fechas_carril, on='Cámara y Dirección')

print("Distribución de Comparendos Electrónicos por Cámara Carril Bus")
pd.set_option('display.max_columns', None)  
pd.set_option('display.width', None)        
pd.set_option('display.max_colwidth', None) 
display(df_carril_ubicaciones.T)
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

Distribución de Comparendos Electrónicos por Cámara Fija


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36
Cámara y Dirección,VIA 11 CON CARRERA 8,CARRERA 51B CON CALLE 79,CALLE 30 CON CARRERA 6B,CARRERA 53 ENTRE CALLE 104 Y 106,CALLE 45 CON CARRERA 1,CALLE 30 CON CARRERA 8,CALLE 82 CON CARRERA 51B,CARRERA 51B CON CALLE 103,CARRERA 6 CON CALLE 72,AVENIDA CIRCUNVALAR CON CARRERA 9G,VIA 40 CON CALLE 73,CALLE 45 CON CARRERA 20,CALLE 82 CON CARRERA 56,CALLE 45B CON CARRERA 14,CALLE 45 CON CARRERA 21,CARRERA 15 CON CALLE 21,CALLE 72 CON CARRERA 44,CALLE 84 ENTRE CARRERA 59 Y 59B,CALLE 94 CON CARRERA 58,CARRERA 46 CON CALLE 100,CALLE 56 CON CARRERA 14,CALLE 19 CON CARRERA 4C,CALLE 53 CON CARRERA 45,CALLE 87 CON CARRERA 21,CALLE 76 CON CARRERA 38C-100,CALLE 19 CON CARRERA 3D,CARRERA 53 CON CALLE 86,CALLE 70 CON CARRERA 46,CALLE 34 CON CARRERA 45,AVENIDA CIRCUNVALAR CON CARRERA 31,CALLE 98 CON CARRERA 56,CALLE 45 CON CARRERA 21 SENTIDO SUR-NORTE,CALLE 61 CON CARRERA 35,CALLE 45 CON CARRERA 21 SENTIDO NORTE-SUR,CARRERA 27 CON CALLE 82C,AVENIDA CIRCUNVALAR (CALLE 110) ENTRE CARRERA 9G Y 12 SENTIDO SUR A NORTE,AVENIDA CIRCUNVALAR (CALLE 110) ENTRE CARRERA 9G Y 12 SENTIDO NORTE A SUR
Comparendos,52029,33293,24261,22979,22800,21832,19626,17116,16292,16186,12983,12740,12032,10568,10036,9910,8849,8479,7863,7705,7385,6980,6781,6319,6236,6215,4394,4241,3133,2225,2128,1182,1176,1125,390,102,66
Porcentaje,12.76,8.17,5.95,5.64,5.59,5.36,4.81,4.2,4.0,3.97,3.18,3.13,2.95,2.59,2.46,2.43,2.17,2.08,1.93,1.89,1.81,1.71,1.66,1.55,1.53,1.52,1.08,1.04,0.77,0.55,0.52,0.29,0.29,0.28,0.1,0.03,0.02
Primer Registro,2018-01-01,2018-01-01,2018-01-01,2018-10-16,2018-01-01,2018-01-01,2018-01-10,2018-01-02,2018-01-01,2018-01-01,2018-01-02,2018-01-01,2018-01-09,2018-01-02,2018-01-01,2018-01-01,2018-01-10,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-04,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-09,2018-01-03,2018-01-01,2018-10-13,2024-11-24,2018-01-30,2024-11-22,2018-01-02,2024-11-29,2024-12-06
Último Registro,2025-12-30,2025-12-28,2025-12-30,2025-12-31,2025-12-30,2025-12-31,2025-12-30,2025-12-12,2025-12-29,2024-11-28,2025-12-20,2024-11-19,2025-12-30,2025-12-31,2025-10-05,2025-12-19,2025-11-07,2025-12-30,2025-08-31,2025-12-31,2021-05-03,2025-12-30,2025-12-30,2021-05-24,2025-12-31,2025-12-28,2022-07-19,2025-12-29,2025-12-31,2019-07-23,2024-08-05,2025-12-28,2025-07-03,2025-12-31,2020-03-02,2025-10-08,2025-12-29


Distribución de Comparendos Electrónicos por Cámara Móvil


,0,1,2,3,4
Cámara y Dirección,Carro mal parqueo Sur,Carro mal parqueo Norte,Carro mal parqueo Sur - Norte,Carro mal parqueo alto norte,Carro mal parqueo alto sur
Comparendos,73544,65306,37993,17090,11040
Porcentaje,35.88,31.86,18.54,8.34,5.39
Primer Registro,2018-01-02,2018-01-02,2018-09-29,2023-06-28,2023-08-15
Último Registro,2025-12-31,2025-12-31,2025-12-30,2025-12-31,2025-12-31


Distribución de Comparendos Electrónicos por Cámara Carril Bus


,0,1,2,3,4,5,6,7,8,9
Cámara y Dirección,CARRERA 46 CON CALLE 34 SENTIDO SUR-NORTE,CALLE 70 CON CARRERA 46 SENTIDO SUR-NORTE,CALLE 70 CON CARRERA 46 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 8 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 33 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 43 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 43 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 1E SENTIDO SUR-NORTE,CARRERA 46 CON CALLE 34 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 1E SENTIDO NORTE-SUR
Comparendos,725,562,300,69,32,18,9,8,4,3
Porcentaje,41.91,32.49,17.34,3.99,1.85,1.04,0.52,0.46,0.23,0.17
Primer Registro,2025-10-23,2025-10-21,2025-10-21,2025-10-23,2025-10-23,2025-10-25,2025-10-24,2025-10-21,2025-10-21,2025-10-21
Último Registro,2025-11-17,2025-12-31,2025-12-11,2025-12-29,2025-12-29,2025-11-05,2025-11-02,2025-10-24,2025-10-25,2025-10-23


**Análisis de concentración en cámaras fijas:**
- Las primeras 5 ubicaciones concentran el **38.11%** de las infracciones de este grupo.
- Las primeras 10 ubicaciones concentran el **60.45%** de las infracciones.
- La ubicación "VIA 11 CON CARRERA 8" es la más activa, con 52,029 infracciones, equivalente a 12.76% del grupo.
- Se observan ubicaciones con sentidos diferenciados en la parte baja de la tabla, como por ejemplo (CALLE 45 CON CARRERA 21 SENTIDO SUR-NORTE y NORTE-SUR), lo que indica que en algunos puntos la dirección de circulación influye en el volumen de detecciones.

**Análisis de concentración en cámaras Carril Bus:**
- Las primeras 2 ubicaciones concentran el **74.4%** de las infracciones de este grupo.
- La ubicación "CARRERA 46 CON CALLE 34 SENTIDO SUR-NORTE" es la más activa, con 725 infracciones (41.91% del grupo).
- Se observa una marcada asimetría por sentido de circulación: en la CARRERA 46 CON CALLE 34, el sentido SUR-NORTE registra 725 infracciones frente a solo 4 en sentido NORTE-SUR. En la CALLE 70 CON CARRERA 46, el sentido SUR-NORTE (562) casi duplica al sentido NORTE-SUR (300).

---

**Hallazgos:**

- **Las cámaras fijas tienen una distribución más dispersa**: Con 37 ubicaciones, muestran una concentración moderada en las primeras posiciones, pero con una cola más larga de ubicaciones con menor actividad.

- **Las cámaras Carril Bus tienen alta concentración**: Solo 10 ubicaciones, pero con las dos primeras concentrando el 74.4% del grupo. La ubicación "CARRERA 46 CON CALLE 34 SENTIDO SUR-NORTE" es, con diferencia, la más crítica para este tipo de infracción (C14).

-  **Asimetría por sentido en Carril Bus**: En las ubicaciones con doble sentido registrado, existe una marcada diferencia en el volumen de infracciones, lo que sugiere que la dirección de circulación influye significativamente en la probabilidad de cometer la infracción de transitar por carril restringido.

- **Corredores críticos identificados**:
   - **Para cámaras fijas**: El corredor de la VIA 11, la CARRERA 51B y la CALLE 30 son los puntos con mayor detección de infracciones.
   - **Para cámaras Carril Bus**: El corredor de la CARRERA 46 CON CALLE 34 y la CALLE 70 CON CARRERA 46 son los puntos críticos para la detección de infracciones por invasión de carril exclusivo.

**Nota:** Los recursos de mantenimiento y operación deberían priorizar las ubicaciones con mayor concentración en cada tipo de cámara.

## Evolución Mensual de Comparendos Electrónicos por Tipo de Cámara

Se examina la evolución mensual de los comparendos electrónicos por tipo de cámara (Fijo, Móvil y Carril Bus), con el objetivo de identificar comportamientos diferenciados en la detección de infracciones según la tecnología utilizada. Se presentan tablas de frecuencias que muestran, para cada tipo de cámara, los meses con mayor y menor volumen de infracciones, así como su porcentaje de participación dentro del total de cada categoría. Complementariamente, se genera un gráfico de líneas interactivo que permite visualizar las tendencias temporales de cada tipo de cámara de manera simultánea, facilitando la comparación de sus patrones estacionales, la identificación de períodos de inactividad o picos específicos, y la evaluación del impacto diferencial de eventos externos como la pandemia de COVID-19 en cada tecnología de vigilancia.

In [15]:
df_comparendos_electronicos_copy['año_mes'] = df_comparendos_electronicos_copy['fecha_comparendo'].dt.to_period('M').astype(str)
infracciones_por_mes_camara = df_comparendos_electronicos_copy.groupby(['año_mes', 'Tipo Camara'])['CANTIDAD_INFRACCIONES'].sum().reset_index()

total_fijo = infracciones_por_mes_camara[infracciones_por_mes_camara['Tipo Camara'] == 'Fijo']['CANTIDAD_INFRACCIONES'].sum()
total_movil = infracciones_por_mes_camara[infracciones_por_mes_camara['Tipo Camara'] == 'Movil']['CANTIDAD_INFRACCIONES'].sum()
total_carril = infracciones_por_mes_camara[infracciones_por_mes_camara['Tipo Camara'] == 'Carril Bus']['CANTIDAD_INFRACCIONES'].sum()

print("  - Cámara Fija")
df_fijo_mensual = infracciones_por_mes_camara[infracciones_por_mes_camara['Tipo Camara'] == 'Fijo'].copy()
df_fijo_mensual = df_fijo_mensual[['año_mes', 'CANTIDAD_INFRACCIONES']].sort_values('año_mes').reset_index(drop=True)
df_fijo_mensual.columns = ['Año-Mes', 'Comparendos']
df_fijo_mensual['Porcentaje'] = (df_fijo_mensual['Comparendos'] / total_fijo * 100).round(2)
df_fijo_mensual = df_fijo_mensual.sort_values('Comparendos', ascending=False).reset_index(drop=True)

pd.set_option('display.max_columns', None)  
pd.set_option('display.width', None)        
pd.set_option('display.max_colwidth', None) 
display(df_fijo_mensual.T)
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

display(HTML("<hr style='border: 2px solid #696969; margin: 20px 0;'>"))

print("Distribución de Comparendos Electrónicos por Mes - Cámara Móvil")
df_movil_mensual = infracciones_por_mes_camara[infracciones_por_mes_camara['Tipo Camara'] == 'Movil'].copy()
df_movil_mensual = df_movil_mensual[['año_mes', 'CANTIDAD_INFRACCIONES']].sort_values('año_mes').reset_index(drop=True)
df_movil_mensual.columns = ['Año-Mes', 'Comparendos']
df_movil_mensual['Porcentaje'] = (df_movil_mensual['Comparendos'] / total_movil * 100).round(2)
df_movil_mensual = df_movil_mensual.sort_values('Comparendos', ascending=False).reset_index(drop=True)


pd.set_option('display.max_columns', None)  
pd.set_option('display.width', None)        
pd.set_option('display.max_colwidth', None) 
display(df_movil_mensual.T)
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

display(HTML("<hr style='border: 2px solid #696969; margin: 20px 0;'>"))

print("Distribución de Comparendos Electrónicos por Mes - Cámara Carril Bus")
df_carril_mensual = infracciones_por_mes_camara[infracciones_por_mes_camara['Tipo Camara'] == 'Carril Bus'].copy()
df_carril_mensual = df_carril_mensual[['año_mes', 'CANTIDAD_INFRACCIONES']].sort_values('año_mes').reset_index(drop=True)
df_carril_mensual.columns = ['Año-Mes', 'Comparendos']
df_carril_mensual['Porcentaje'] = (df_carril_mensual['Comparendos'] / total_carril * 100).round(2)
df_carril_mensual = df_carril_mensual.sort_values('Comparendos', ascending=False).reset_index(drop=True)

pd.set_option('display.max_columns', None)  
pd.set_option('display.width', None)        
pd.set_option('display.max_colwidth', None) 
display(df_carril_mensual.T)
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

display(HTML("<hr style='border: 2px solid #696969; margin: 20px 0;'>"))

meses = sorted(infracciones_por_mes_camara['año_mes'].unique())

fig = go.Figure()

colores = {
    'Fijo': 'cornflowerblue',
    'Movil': 'mediumpurple',
    'Carril Bus': 'mediumorchid'
}

for tipo in ['Fijo', 'Movil', 'Carril Bus']:
    df_tipo = infracciones_por_mes_camara[infracciones_por_mes_camara['Tipo Camara'] == tipo]
    
    max_valor_tipo = df_tipo['CANTIDAD_INFRACCIONES'].max()
    max_fila_tipo = df_tipo[df_tipo['CANTIDAD_INFRACCIONES'] == max_valor_tipo].iloc[0]
    max_mes_tipo = max_fila_tipo['año_mes']
    
    min_valor_tipo = df_tipo['CANTIDAD_INFRACCIONES'].min()
    min_fila_tipo = df_tipo[df_tipo['CANTIDAD_INFRACCIONES'] == min_valor_tipo].iloc[0]
    min_mes_tipo = min_fila_tipo['año_mes']
    
    fig.add_trace(go.Scatter(
        x=df_tipo['año_mes'],
        y=df_tipo['CANTIDAD_INFRACCIONES'],
        mode='lines+markers',
        name=tipo,
        line=dict(color=colores[tipo], width=2),
        marker=dict(size=4),
        hovertemplate=f'{tipo}<br>Mes: %{{x}}<br>Comparendos: %{{y:,.0f}}<extra></extra>'
    ))
    

    fig.add_trace(go.Scatter(
        x=[max_mes_tipo],
        y=[max_valor_tipo],
        mode='markers',
        name=f'Máximo {tipo}',
        marker=dict(size=8, color='red', symbol='circle'),
        showlegend=False,
        hoverinfo='none'
    ))
    
    fig.add_trace(go.Scatter(
        x=[min_mes_tipo],
        y=[min_valor_tipo],
        mode='markers',
        name=f'Mínimo {tipo}',
        marker=dict(size=8, color='green', symbol='circle'),
        showlegend=False,
        hoverinfo='none'
    ))

fig.add_vrect(
    x0="2020-03", x1="2020-12",
    fillcolor="red", opacity=0.1,
    line_width=0,
    annotation_text="COVID-19", 
    annotation_position="top left",
    annotation=dict(font_size=12, font_color="red")
)

fig.update_layout(
    title=dict(
        text='Evolución Mensual de Comparendos Electrónicos por Tipo de Cámara',
        x=0.5,
        xanchor='center',
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(
        text='Año-Mes',
        font=dict(weight='bold')
    ),
    yaxis_title=dict(
        text='Cantidad de Comparendos',
        font=dict(weight='bold')
    ),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=1, 
        y=1, 
        xanchor='center',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.8)',
        font=dict(size=12),
        title='Tipo de Cámara'
    ),
    width=1055, 
    height=500
)

tick_positions = meses[::6]

fig.update_xaxes(
    tickangle=-45,
    tickvals=tick_positions,
    ticktext=tick_positions
)

fig.show()

  - Cámara Fija


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95
Año-Mes,2019-03,2022-12,2019-04,2022-06,2022-05,2024-05,2019-07,2019-02,2022-11,2019-08,2019-06,2022-07,2019-05,2022-04,2023-12,2024-06,2024-04,2024-03,2019-01,2023-07,2023-04,2023-06,2022-09,2023-03,2024-09,2023-05,2022-10,2024-07,2018-11,2020-01,2023-09,2023-02,2023-08,2019-10,2018-01,2019-12,2023-10,2020-02,2018-12,2024-10,2019-09,2020-03,2019-11,2018-10,2024-01,2024-02,2022-08,2021-01,2020-04,2020-08,2024-08,2020-09,2023-11,2018-03,2020-12,2018-05,2022-03,2020-10,2018-04,2018-09,2018-02,2021-12,2018-07,2022-01,2018-06,2021-04,2018-08,2020-11,2025-10,2024-11,2021-07,2021-02,2021-03,2021-08,2025-04,2021-05,2024-12,2020-06,2021-09,2025-01,2020-07,2025-03,2021-11,2021-10,2021-06,2025-05,2022-02,2025-07,2025-02,2025-09,2025-06,2023-01,2025-12,2025-08,2025-11,2020-05
Comparendos,7222,7102,6310,6272,6169,6098,6054,5903,5903,5893,5873,5830,5759,5718,5692,5659,5578,5565,5528,5491,5482,5449,5408,5339,5311,5303,5224,5089,5009,4903,4862,4815,4798,4761,4759,4737,4703,4681,4662,4656,4590,4566,4563,4510,4473,4468,4446,4444,4400,4289,4202,4159,4125,4086,4080,4072,3996,3859,3832,3776,3687,3590,3577,3574,3567,3563,3461,3436,3401,3377,3368,3359,3318,3122,3103,3074,3019,2976,2924,2885,2806,2681,2671,2646,2561,2423,2389,2379,2324,2298,2194,2106,2038,1986,1882,1386
Porcentaje,1.77,1.74,1.55,1.54,1.51,1.5,1.49,1.45,1.45,1.45,1.44,1.43,1.41,1.4,1.4,1.39,1.37,1.37,1.36,1.35,1.34,1.34,1.33,1.31,1.3,1.3,1.28,1.25,1.23,1.2,1.19,1.18,1.18,1.17,1.17,1.16,1.15,1.15,1.14,1.14,1.13,1.12,1.12,1.11,1.1,1.1,1.09,1.09,1.08,1.05,1.03,1.02,1.01,1.0,1.0,1.0,0.98,0.95,0.94,0.93,0.9,0.88,0.88,0.88,0.88,0.87,0.85,0.84,0.83,0.83,0.83,0.82,0.81,0.77,0.76,0.75,0.74,0.73,0.72,0.71,0.69,0.66,0.66,0.65,0.63,0.59,0.59,0.58,0.57,0.56,0.54,0.52,0.5,0.49,0.46,0.34


Distribución de Comparendos Electrónicos por Mes - Cámara Móvil


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89
Año-Mes,2019-05,2023-09,2019-06,2023-08,2023-10,2018-01,2021-02,2023-11,2019-07,2023-07,2018-03,2018-05,2019-04,2018-04,2020-10,2024-04,2018-10,2018-11,2018-06,2023-12,2021-03,2024-01,2025-10,2018-07,2018-02,2024-07,2024-05,2025-09,2018-08,2019-08,2024-10,2023-06,2024-03,2018-09,2024-02,2025-03,2020-01,2024-06,2025-05,2025-07,2018-12,2025-04,2025-02,2025-01,2023-05,2025-11,2024-08,2019-10,2019-09,2020-02,2019-03,2024-12,2024-09,2023-03,2019-12,2020-11,2024-11,2022-06,2025-06,2020-12,2019-02,2019-11,2025-12,2022-07,2022-10,2022-08,2022-11,2023-04,2022-05,2021-10,2021-07,2025-08,2021-01,2021-11,2022-09,2022-02,2022-12,2022-03,2021-09,2021-04,2022-01,2020-03,2022-04,2021-08,2021-12,2019-01,2021-05,2023-02,2020-09,2021-06
Comparendos,4084,3822,3647,3581,3575,3375,3329,3310,3267,3263,3247,3101,3065,3039,3031,3027,2958,2943,2942,2879,2848,2810,2766,2761,2751,2731,2729,2713,2707,2703,2653,2594,2555,2499,2496,2476,2444,2442,2439,2422,2420,2418,2413,2411,2410,2376,2370,2362,2349,2308,2283,2275,2264,2258,2185,2179,2146,2145,2119,2074,2025,2008,1898,1829,1707,1682,1667,1656,1567,1520,1471,1423,1392,1385,1362,1353,1340,1332,1317,1309,1282,1265,1118,1095,1094,1000,900,888,684,615
Porcentaje,1.99,1.86,1.78,1.75,1.74,1.65,1.62,1.61,1.59,1.59,1.58,1.51,1.5,1.48,1.48,1.48,1.44,1.44,1.44,1.4,1.39,1.37,1.35,1.35,1.34,1.33,1.33,1.32,1.32,1.32,1.29,1.27,1.25,1.22,1.22,1.21,1.19,1.19,1.19,1.18,1.18,1.18,1.18,1.18,1.18,1.16,1.16,1.15,1.15,1.13,1.11,1.11,1.1,1.1,1.07,1.06,1.05,1.05,1.03,1.01,0.99,0.98,0.93,0.89,0.83,0.82,0.81,0.81,0.76,0.74,0.72,0.69,0.68,0.68,0.66,0.66,0.65,0.65,0.64,0.64,0.63,0.62,0.55,0.53,0.53,0.49,0.44,0.43,0.33,0.3


Distribución de Comparendos Electrónicos por Mes - Cámara Carril Bus


,0,1,2
Año-Mes,2025-11,2025-12,2025-10
Comparendos,1073,353,304
Porcentaje,62.02,20.4,17.57


**Ausencia de registros de cámaras móviles durante abril-agosto de 2020**: 

Durante los meses de abril, mayo, junio, julio y agosto de 2020, las cámaras móviles no registraron infracciones, lo que representa una interrupción total de su operación durante este período. Esta ausencia de detecciones se explica por las estrictas medidas de confinamiento y restricción de movilidad implementadas en Barranquilla durante el pico de la pandemia de COVID-19. Las cámaras móviles, que dependen de patrullas y personal en desplazamiento para detectar la infracción de estacionamiento prohibido, no pudieron operar debido a las limitaciones de circulación establecidas por las autoridades. A diferencia de las cámaras fijas que permanecieron activas detectando infracciones en puntos estáticos (aunque con volúmenes reducidos), las cámaras móviles requieren desplazamiento y personal operativo, lo que las hizo inviables durante el período de aislamiento estricto. 

**Fuente**: https://barranquilla.gov.co/coronavirus/60-dias-de-crisis-covid19-barranquilla

---

**Ausencia de registros de cámaras móviles durante enero de 2023**: 

Al analizar la serie temporal de los comparendos electrónicos, se observó que enero de 2023 corresponde al segundo período con el valor más bajo en la evolución mensual de infracciones. A través de este análisis bivariado, es posible entender que este valor mínimo se debe a que durante el mes de enero de 2023 las cámaras móviles no estuvieron en funcionamiento. La ausencia de este tipo de cámara explica la caída significativa en el volumen de comparendos durante ese mes. Esta interrupción operativa podría deberse a diversos factores como el período de vacaciones de fin de año, mantenimiento programado de las unidades móviles, o reasignación de personal durante la temporada de fin de año, lo que habría limitado la capacidad de detección del sistema durante ese período.

--- 

**Posible explicación para el período 2019-05 (máximo) de cámaras móviles**: 

Durante mayo de 2019, las cámaras móviles alcanzaron su pico máximo de detección de infracciones, coincidiendo con el mes de mayor volumen de comparendos electrónicos en toda la serie histórica. Este comportamiento atípicamente alto puede explicarse por los múltiples cierres viales que se presentaron en Barranquilla durante ese mes, los cuales superaron los 5 eventos. Los cierres viales generaron alteraciones significativas en los patrones normales de circulación, desviando el tránsito hacia vías alternas y creando condiciones de congestión que facilitaron la comisión de infracciones como el estacionamiento en sitios prohibidos (C02), que es precisamente el tipo de infracción que las cámaras móviles están diseñadas para detectar.

**Fuentes**:
- https://barranquilla.gov.co/transito/por-trabajos-cierres-de-vias-a-partir-del-viernes-3-de-mayo
- https://barranquilla.gov.co/transito/por-ampliacion-de-la-calle-30-cierres-de-vias-a-partir-del-martes-14-de-mayo-de-2019
- https://barranquilla.gov.co/transito/por-proyecto-mall-plaza-cierres-de-vias-a-partir-del-martes-21-de-mayo-de-2019
- https://barranquilla.gov.co/transito/por-reparacion-de-redes-cierres-de-vias-a-partir-del-martes-28-de-mayo

--- 

**Posible explicación para el período 2021-06 (mínimo) de cámaras móviles**: 

Durante junio de 2021, las cámaras móviles registraron su volúmen más bajo de detección de infracciones. Este comportamiento puede explicarse por el contexto de reactivación gradual de la ciudad después de los picos más críticos de la pandemia, combinado con el aumento sostenido del teletrabajo durante ese período. Si bien las medidas de confinamiento estricto de 2020 se habían flexibilizado, muchas empresas y entidades mantuvieron esquemas de trabajo remoto o híbrido, lo que redujo significativamente el número de vehículos en circulación. Sin embargo, a diferencia de lo que podría pensarse, esta reactivación no pudo implicar un aumento del tránsito vehicular, sino que las personas aprovecharon la nueva libertad para salir a caminar, pasear en bicicleta y disfrutar de actividades al aire libre que antes no podían realizar debido a las restricciones sanitarias. Este cambio en los hábitos de movilidad, con un mayor protagonismo de peatones y ciclistas, explica por qué las infracciones detectadas por cámaras móviles (enfocadas principalmente en estacionamiento prohibido) alcanzaron sus niveles más bajos durante este período, ya que simplemente había menos vehículos circulando y estacionándose en las calles.

**Fuentes:**
- https://barranquilla.gov.co/funcionarios/teletrabajo-una-tendencia-en-crecimiento
- https://barranquilla.gov.co/descubre/barranquilla-atlantico-anuncian-que-estan-abiertos-para-el-mundo
- https://barranquilla.gov.co/transito/vuelve-biciquilla-este-jueves-24-de-junio
- https://barranquilla.gov.co/cultura/concierto-musica-tradicional-baile-y-talleres-arte-reactivan-cultura-barranquilla
- https://barranquilla.gov.co/desarrolloeconomico/barranquilla-reactiva-inicia-nueva-etapa-de-reapertura-plena-integral
- https://barranquilla.gov.co/cultura/talleres-artisticos-baila-ton-y-conciertos-en-los-parques-reactivan-el-sector-cultural
- https://barranquilla.gov.co/mi-barranquilla/ven-vive-barranquilla-atlantico-alianza-impulsa-reactivacion
- https://barranquilla.gov.co/desarrolloeconomico/barranquilla-se-alista-para-comenzar-su-reactivacion-plena

---

**Posible explicación para el período 2019-03 (máximo) de cámaras fijas**: 

Durante marzo de 2019, las cámaras fijas registraron su volumen más alto de detección de infracciones, alcanzando niveles máximos en la serie histórica. Este comportamiento puede explicarse por los múltiples cierres viales que se presentaron en Barranquilla durante ese mes, los cuales incluyeron cierres derivados de las celebraciones del Carnaval (cuyas festividades se extienden hasta principios de marzo) y otras circunstancias propias de la operación vial de la ciudad, como eventos culturales, deportivos o trabajos de infraestructura. Los cierres generaron alteraciones significativas en los patrones normales de circulación, desviando el tránsito hacia vías alternas y creando condiciones de congestión que incrementaron la probabilidad de comisión de infracciones, especialmente en las vías donde operan las cámaras fijas.

**Fuentes**:

- https://barranquilla.gov.co/transito/por-eventos-de-pre-carnaval-cierres-de-vias-para-viernes-1-de-marzo
- https://barranquilla.gov.co/transito/por-carnavales-2019-cierre-de-vias
- https://barranquilla.gov.co/transito/por-izaje-de-equipos-cierres-de-vias-para-el-jueves-14-de-marzo-de-2019
- https://barranquilla.gov.co/transito/por-trabajos-de-ampliacion-de-la-calle-30-a-partir-del-martes-12-de-marzo-cierres-de-vias
- https://barranquilla.gov.co/transito/cierres-de-vias-para-el-domingo-17-de-marzo-de-2019
- https://barranquilla.gov.co/transito/por-reparacion-de-pavimento-cierre-total-de-la-carrera-50-entre-calles-85-y-86
- https://barranquilla.gov.co/transito/cierre-de-vias-para-el-sabado-23-de-marzo-de-2019

---

**Posible explicación para el período 2020-05 (mínimo) de cámaras fijas**: 

Durante mayo de 2020, las cámaras fijas registraron su volumen más bajo de detección de infracciones en toda la serie histórica. Este comportamiento se explica por las estrictas medidas de restricción de movilidad implementadas en Barranquilla durante el pico de la pandemia de COVID-19. El confinamiento obligatorio, la suspensión del servicio de Transmetro, la restricción máxima de circulación mediante el calendario de 'pico y cédula', la ley seca y la limitación de salidas exclusivamente para trabajar o por emergencias médicas redujeron drásticamente el flujo vehicular en las vías de la ciudad.

**Fuente**: https://barranquilla.gov.co/coronavirus/60-dias-de-crisis-covid19-barranquilla

---

**Explicación para el período 2025-10 (mínimo de cámaras de carril bus):** 

El mínimo registrado en las cámaras de carril bus durante octubre de 2025 se explica por el hecho de que estos equipos comenzaron a funcionar a partir del 17 de octubre de 2025, por lo que no tuvieron un mes completo de operación para realizar detecciones. Al haber iniciado sus labores a mediados de octubre, el período de octubre representa menos quince días de funcionamiento parcial, sin que se haya alcanzado una operación continua a lo largo de un ciclo mensual completo.

### Distribución Mensual y Anual de Comparendos Electrónicos por Tipo de Cámara

Se construyen tablas pivote que cruzan los meses y los años para cada tipo de cámara (Fijo y Móvil), utilizando la suma ponderada de `CANTIDAD_INFRACCIONES` para visualizar el volumen de infracciones en cada combinación temporal. Los resultados se presentan mediante mapas de calor independientes para cada tipo de cámara, con los meses ordenados de diciembre a enero para facilitar la identificación de patrones estacionales. Esta visualización permite comparar el comportamiento de las cámaras fijas y móviles a lo largo del tiempo, identificar períodos de alta o baja actividad específicos de cada tecnología, y detectar posibles estacionalidades o cambios operativos en la detección de infracciones según el tipo de cámara.

In [16]:
tipos_camara = ['Fijo', 'Movil']

for tipo in tipos_camara:
    df_tipo = df_comparendos_electronicos_copy[df_comparendos_electronicos_copy['Tipo Camara'] == tipo]
    
    df_tipo['año'] = df_tipo['fecha_comparendo'].dt.year
    df_tipo['mes_numero'] = df_tipo['fecha_comparendo'].dt.month
    
    tabla_heatmap = df_tipo.pivot_table(
        values='CANTIDAD_INFRACCIONES',
        index='mes_numero',
        columns='año',
        aggfunc='sum',
        fill_value=0
    )
    

    nombres_meses = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 
                     'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
    
    tabla_heatmap.index = nombres_meses
    tabla_heatmap = tabla_heatmap.iloc[::-1]
    
    fig = go.Figure(data=go.Heatmap(
        z=tabla_heatmap.values,
        x=tabla_heatmap.columns,
        y=tabla_heatmap.index,
        colorscale='blues',
        text=tabla_heatmap.values,
        texttemplate='%{text:,.0f}',
        hovertemplate=f'Cámara: {tipo}<br>Mes: %{{y}}<br>Año: %{{x}}<br>Comparendos: %{{z:,.0f}}<extra></extra>'
    ))
    
    fig.update_layout(
        title=dict(
            text=f'Distribución de Comparendos Electrónicos por Año y Mes - Cámara {tipo}',
            x=0.5,
            xanchor='center',
            font=dict(size=16, weight='bold')
        ),
        xaxis_title=dict(
            text='Año',
            font=dict(weight='bold')
        ),
        yaxis_title=dict(
            text='Mes',
            font=dict(weight='bold')
        ),
        template='plotly_white',
        width=1055, 
        height=500
    )
    
    fig.show()

### Distribución Mensual de Comparendos Electrónicos por Tipo de Cámara

Se examina la variabilidad mensual de los comparendos electrónicos desagregada por tipo de cámara (Fijo y Móvil), agrupando los registros por mes y año y sumando la columna `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones por cada combinación. Para cada tipo de cámara, se presentan estadísticos descriptivos (media, desviación estándar, mínimo, máximo y cuartiles) para cada mes, con el fin de caracterizar su comportamiento típico y su dispersión interanual. Complementariamente, se generan diagramas de caja (boxplot) que visualizan la distribución de las infracciones para cada mes, incluyendo la media como referencia. Para validar si las diferencias observadas entre meses son estadísticamente significativas para cada tipo de cámara, se aplica un análisis de varianza que incluye la prueba de normalidad de Shapiro-Wilk para evaluar la distribución de los datos por mes, la prueba de Levene para verificar la homogeneidad de varianzas, y dependiendo del cumplimiento de los supuestos, se realiza un ANOVA paramétrico (con prueba post-hoc de Tukey HSD) o un Kruskal-Wallis no paramétrico (con prueba post-hoc de Dunn y corrección de Bonferroni). Este análisis permite identificar, de manera diferenciada por tipo de tecnología, qué meses presentan mayor variabilidad en la detección de infracciones, detectar meses con comportamientos atípicos, comparar la estabilidad estacional entre las cámaras fijas y móviles, y determinar si existe estacionalidad estadísticamente significativa en el volumen de comparendos detectados por cada tipo de cámara.

In [17]:
df_comparendos_electronicos_copy['año'] = df_comparendos_electronicos_copy['fecha_comparendo'].dt.year
df_comparendos_electronicos_copy['mes_nombre'] = df_comparendos_electronicos_copy['fecha_comparendo'].dt.month

for tipo in tipos_camara:
    df_tipo = df_comparendos_electronicos_copy[df_comparendos_electronicos_copy['Tipo Camara'] == tipo]
    
    df_boxplot_tipo = df_tipo.groupby(['mes_nombre', 'año'])['CANTIDAD_INFRACCIONES'].sum().reset_index()
    df_boxplot_tipo.columns = ['mes', 'año', 'infracciones']
    df_boxplot_tipo['mes_nombre'] = df_boxplot_tipo['mes'].apply(lambda x: nombres_meses[x-1])
    
    print(f"Estadísticas Descriptivas - Cámara {tipo}")
    estadisticas = df_boxplot_tipo.groupby('mes_nombre')['infracciones'].describe().round(0).fillna(0).astype(int).reindex(nombres_meses)
    display(estadisticas.T)
    
    display(HTML("<hr style='border: 1px solid #696969; margin: 20px 0;'>"))
    
    fig = go.Figure()
    
    fig.add_trace(go.Box(
        x=df_boxplot_tipo['mes_nombre'],
        y=df_boxplot_tipo['infracciones'],
        marker_color='mediumpurple',
        name='',
        boxmean=True,
        hovertemplate='Mes: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
    ))
    
    fig.update_layout(
        title=dict(
            text=f'Distribución de Comparendos Electrónicos por Mes - Cámara {tipo}',
            x=0.5,
            xanchor='center',
            font=dict(size=16, weight='bold')
        ),
        xaxis_title=dict(
            text='Mes',
            font=dict(weight='bold')
        ),
        yaxis_title=dict(
            text='Cantidad de Comparendos',
            font=dict(weight='bold')
        ),
        template='plotly_white',
        width=1055,
        height=500
    )
    
    fig.show()
    
    
    df_anova = df_boxplot_tipo.copy()
    
    meses_grupos = [
        df_anova[df_anova['mes_nombre'] == mes]['infracciones'].values 
        for mes in nombres_meses
    ]
    
    print("\nPrueba de normalidad (Shapiro-Wilk) por mes:")
    
    normalidad = []
    for mes, grupo in zip(nombres_meses, meses_grupos):
        if len(grupo) >= 3:  
            stat, p = stats.shapiro(grupo)
            normalidad.append(p > 0.05)
            print(f"{mes}: p-valor = {p:.6f} -> {'Normal' if p > 0.05 else 'No normal'}")
        else:
            print(f"{mes}: No hay suficientes datos para evaluar normalidad")
    
    cumple_normalidad = all(normalidad)
    print(f"¿Se cumple normalidad en todos los meses? {'Sí' if cumple_normalidad else 'No'}")
    
    stat_levene, p_levene = stats.levene(*meses_grupos)
    
    print(f"\nLevene p-valor: {p_levene:.6f}")
    print(f"¿Varianzas iguales? {'Sí' if p_levene > 0.05 else 'No'}")
    
    if cumple_normalidad and p_levene > 0.05:
        print("\nSe cumplen los supuestos → Se aplica ANOVA")
        
        f_stat, p_valor = stats.f_oneway(*meses_grupos)
        
        print(f"Estadístico F: {f_stat:.4f}")
        print(f"P-valor: {p_valor:.6f}")
        
        print(f"¿Hay diferencias entre meses? {'Sí' if p_valor < 0.05 else 'No'}")
        
        if p_valor < 0.05:
            print("\nPrueba post-hoc Tukey HSD:")
            
            tukey = pairwise_tukeyhsd(
                endog=df_anova['infracciones'],
                groups=df_anova['mes_nombre'],
                alpha=0.05
            )
            
            print(tukey)
    
    else:
        print("\nNo se cumplen los supuestos → Se aplica Kruskal-Wallis")
        
        h_stat, p_valor = stats.kruskal(*meses_grupos)
        
        print(f"Estadístico H: {h_stat:.4f}")
        print(f"P-valor: {p_valor:.6f}")
        
        print(f"¿Hay diferencias entre meses? {'Sí' if p_valor < 0.05 else 'No'}")
        
        if p_valor < 0.05:
            print("\nPrueba post-hoc Dunn (con corrección Bonferroni):")
            
            df_dunn = df_anova[['mes_nombre', 'infracciones']].copy()
            
            dunn = sp.posthoc_dunn(
                df_dunn,
                val_col='infracciones',
                group_col='mes_nombre',
                p_adjust='bonferroni'
            )
            
            dunn = dunn.loc[nombres_meses, nombres_meses]
            display(dunn)
    
    display(HTML("<hr style='border: 2px solid #696969; margin: 20px 0;'>"))

Estadísticas Descriptivas - Cámara Fijo


mes_nombre,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
count,8,8,8,8,8,8,8,8,8,8,8,8
mean,4084,3953,4597,4748,4286,4319,4324,4025,4166,4220,3871,4365
std,1140,1246,1430,1176,1829,1659,1452,1175,1115,852,1298,1576
min,2106,2324,2681,3103,1386,2194,2379,1986,2298,2646,1882,2038
25%,3402,3116,3826,3765,2911,2872,3228,3376,3563,3744,3200,3447
50%,4458,4078,4326,4941,4688,4508,4333,4246,4374,4583,3780,4371
75%,4795,4714,5396,5613,5844,5712,5576,4534,4974,4718,4674,4976
max,5528,5903,7222,6310,6169,6272,6054,5893,5408,5224,5903,7102



Prueba de normalidad (Shapiro-Wilk) por mes:
Enero: p-valor = 0.566351 -> Normal
Febrero: p-valor = 0.686548 -> Normal
Marzo: p-valor = 0.877398 -> Normal
Abril: p-valor = 0.454767 -> Normal
Mayo: p-valor = 0.298971 -> Normal
Junio: p-valor = 0.144952 -> Normal
Julio: p-valor = 0.265104 -> Normal
Agosto: p-valor = 0.946519 -> Normal
Septiembre: p-valor = 0.555494 -> Normal
Octubre: p-valor = 0.322844 -> Normal
Noviembre: p-valor = 0.996399 -> Normal
Diciembre: p-valor = 0.989764 -> Normal
¿Se cumple normalidad en todos los meses? Sí

Levene p-valor: 0.319416
¿Varianzas iguales? Sí

Se cumplen los supuestos → Se aplica ANOVA
Estadístico F: 0.2820
P-valor: 0.987697
¿Hay diferencias entre meses? No


Estadísticas Descriptivas - Cámara Movil


mes_nombre,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
count,7,8,8,7,7,7,7,7,8,8,8,8
mean,2102,2195,2283,2233,2461,2358,2535,2223,2126,2572,2252,2021
std,887,774,687,860,1029,931,683,869,982,688,630,577
min,1000,888,1265,1118,900,615,1471,1095,684,1520,1385,1094
25%,1337,1857,2026,1482,1988,2132,2126,1552,1351,2198,1923,1758
50%,2411,2360,2380,2418,2439,2442,2731,2370,2306,2710,2162,2130
75%,2627,2560,2628,3033,2915,2768,3012,2705,2552,2976,2518,2311
max,3375,3329,3247,3065,4084,3647,3267,3581,3822,3575,3310,2879



Prueba de normalidad (Shapiro-Wilk) por mes:
Enero: p-valor = 0.514465 -> Normal
Febrero: p-valor = 0.823290 -> Normal
Marzo: p-valor = 0.420945 -> Normal
Abril: p-valor = 0.100205 -> Normal
Mayo: p-valor = 0.952927 -> Normal
Junio: p-valor = 0.563038 -> Normal
Julio: p-valor = 0.442168 -> Normal
Agosto: p-valor = 0.783135 -> Normal
Septiembre: p-valor = 0.804548 -> Normal
Octubre: p-valor = 0.710706 -> Normal
Noviembre: p-valor = 0.813963 -> Normal
Diciembre: p-valor = 0.798258 -> Normal
¿Se cumple normalidad en todos los meses? Sí

Levene p-valor: 0.961975
¿Varianzas iguales? Sí

Se cumplen los supuestos → Se aplica ANOVA
Estadístico F: 0.3436
P-valor: 0.972743
¿Hay diferencias entre meses? No


**Patrones clave identificados:**

- **Las cámaras fijas tienen mayor volumen y variabilidad**: Con un promedio mensual mayor (casi el doble que las móviles), las cámaras fijas son el principal instrumento de control. Su mayor variabilidad indica que son más sensibles a factores externos como cierres viales o eventos masivos.

- **Las cámaras móviles son más estables**: Con un rango promedio de solo 551 infracciones entre el mes de mayor y menor promedio, las móviles muestran un comportamiento más uniforme a lo largo del año, excepto por los picos atípicos en mayo.

- **Mayo es el mes más impredecible para ambos tipos**: Tanto en cámaras fijas como móviles, mayo presenta la mayor variabilidad, influenciada por eventos extremos como los cierres viales de 2019 (picos máximos) y la pandemia de 2020 (valles mínimos).

- **Estacionalidad diferenciada**:
   - **Cámaras fijas**: Mayor actividad en abril y diciembre; menor en noviembre y febrero.
   - **Cámaras móviles**: Mayor actividad en octubre y julio; menor en diciembre y enero.

- **Impacto de la pandemia en los registros**: La ausencia de datos en varios meses para las cámaras móviles (count inferior a 8) confirma la interrupción de su operación durante el confinamiento de 2020, mientras que las cámaras fijas mantuvieron registro continuo durante todo el período.

---

**Hallazgos:**

- **Las cámaras fijas son el pilar del sistema de control**: Con volúmenes casi duplicados a los de las móviles y operación continua durante toda la serie histórica, son la tecnología más confiable y de mayor cobertura.

- **Las cámaras móviles complementan el control con un rol específico**: Su menor volumen y estabilidad relativa reflejan su especialización en la detección de estacionamiento prohibido (C02), con patrones estacionales que probablemente responden a la actividad comercial y turística de la ciudad.

- **Mayo como mes crítico**: La alta variabilidad en ambos tipos de cámara durante mayo sugiere que es un mes especialmente sensible a intervenciones externas (cierres viales, eventos masivos, restricciones sanitarias), lo que debería considerarse en la planificación de operativos de control.

**Nota**: Los meses de abril y diciembre (picos en fijas) y octubre y julio (picos en móviles) requieren especial atención en términos de mantenimiento y operación de las cámaras, así como campañas preventivas de educación vial.


---

- **Verificación de supuestos para cámaras fijas**: Las pruebas de normalidad (Shapiro-Wilk) aplicadas a cada mes para las cámaras fijas arrojaron p-valores superiores a 0.05 en todos los casos (el más bajo fue junio con 0.1449, aún por encima del umbral), lo que indica que los datos de cada mes siguen una distribución normal. La prueba de Levene con un p-valor de 0.3194 confirma que las varianzas entre los meses son homogéneas. Dado que ambos supuestos se cumplen, es apropiado aplicar un ANOVA paramétrico para comparar las medias mensuales de las cámaras fijas.

- **Resultado del ANOVA para cámaras fijas**: El estadístico F es de 0.2820 con un p-valor de 0.9877, un valor extremadamente alto y cercano a 1. Esto indica que **no existen diferencias estadísticamente significativas entre las medias de los meses** para las cámaras fijas. En otras palabras, el volumen promedio de comparendos detectados por cámaras fijas se mantiene constante a lo largo del año.

---

- **Verificación de supuestos para cámaras móviles**: Las pruebas de normalidad (Shapiro-Wilk) aplicadas a cada mes para las cámaras móviles también arrojaron p-valores superiores a 0.05 en todos los casos (el más bajo fue abril con 0.1002, aún por encima del umbral), lo que indica que los datos de cada mes siguen una distribución normal. La prueba de Levene con un p-valor de 0.9620 confirma que las varianzas entre los meses son homogéneas. Dado que ambos supuestos se cumplen, es apropiado aplicar un ANOVA paramétrico para comparar las medias mensuales de las cámaras móviles.

- **Resultado del ANOVA para cámaras móviles**: El estadístico F es de 0.3436 con un p-valor de 0.9727, nuevamente un valor extremadamente alto. Esto indica que **tampoco existen diferencias estadísticamente significativas entre las medias de los meses** para las cámaras móviles.

---

**Hallazgos**:

- **Ausencia de estacionalidad en ambos tipos de cámara**: Ni las cámaras fijas ni las móviles presentan diferencias estadísticamente significativas en el volumen de comparendos a lo largo de los meses del año. Esto significa que, a nivel mensual, ambos tipos de tecnología detectan volúmenes similares de infracciones independientemente de la época del año.

- **Comportamiento consistente entre tecnologías**: A pesar de que las cámaras fijas y móviles tienen funciones diferentes (las fijas detectan exceso de velocidad, semáforos, etc.; las móviles detectan estacionamiento prohibido), ambas comparten la misma característica de no presentar estacionalidad mensual.

- **Contraste con la descomposición STL**: Este resultado complementa el análisis de descomposición STL, donde se observó que las cámaras fijas tenían una amplitud estacional de 1,952 comparendos y las móviles de 1,101 comparendos. Sin embargo, al someter estas diferencias a una prueba estadística (ANOVA), se confirma que las oscilaciones observadas entre meses no son lo suficientemente grandes ni consistentes a lo largo de los años como para ser consideradas estadísticamente significativas.

## Análisis del Servicio del Vehículo Infractor Según el Código de Infracción

Se construye una tabla pivote que cruza el servicio del vehículo infractor con los códigos de infracción, agrupando las categorías con baja representación (OFICIAL y DIPLOMÁTICO) en una categoría denominada "OTROS" para facilitar la visualización. Se utiliza la suma ponderada de `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones en cada combinación. Para cada categoría de servicio, se calcula el porcentaje que representa cada código de infracción sobre el total de sus comparendos, permitiendo identificar la contribución relativa de cada tipo de infracción. Los resultados se presentan mediante un mapa de calor interactivo, donde las categorías de servicio se ordenan de mayor a menor según su total de comparendos, y cada celda muestra tanto el valor absoluto como el porcentaje correspondiente. Esta visualización permite identificar qué tipos de infracción son más frecuentes en cada categoría de servicio vehicular, revelando patrones de comportamiento diferenciados entre vehículos particulares, públicos y otros servicios.

In [18]:
tabla_heatmap = df_comparendos_electronicos_copy.pivot_table(
    values='CANTIDAD_INFRACCIONES',
    index='SERVICIO_AGRUPADO',
    columns='COD_INFRACCION',
    aggfunc='sum',
    fill_value=0
)

orden = tabla_heatmap.sum(axis=1).sort_values(ascending=True).index
tabla_heatmap = tabla_heatmap.loc[orden]

total_por_fila = tabla_heatmap.sum(axis=1)
porcentajes = tabla_heatmap.div(total_por_fila, axis=0) * 100
porcentajes = porcentajes.round(2)
texto_combinado = tabla_heatmap.applymap(lambda x: f'{int(x):,}') + '<br>(' + porcentajes.applymap(lambda x: f'{x:.2f}%') + ')'

fig = go.Figure(data=go.Heatmap(
    z=tabla_heatmap.values,
    x=tabla_heatmap.columns,
    y=tabla_heatmap.index,
    colorscale='Blues',
    text=texto_combinado.values,
    texttemplate='%{text}',
    hovertemplate='Servicio: %{y}<br>Código: %{x}<br>Comparendos: %{z:,.0f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Distribución de Comparendos por Servicio del Vehículo y Código de Infracción',
        x=0.5,
        xanchor='center',
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(
        text='Código de Infracción',
        font=dict(weight='bold')
    ),
    yaxis_title=dict(
        text='Servicio del Vehículo',
        font=dict(weight='bold')
    ),
    template='plotly_white',
    width=1055, 
    height=500
)

fig.show()

- Los vehículos particulares concentran sus infracciones principalmente en dos tipos: exceso de velocidad (45.85%) y estacionamiento prohibido (35.43%), que en conjunto representan el **81.28%** de sus comparendos.

- Al igual que los particulares, el exceso de velocidad (C29) es la infracción más frecuente en vehículos públicos, representando casi la mitad (49.41%). Sin embargo, presentan una distribución más equilibrada entre las demás infracciones, con el estacionamiento prohibido (20.82%) y el bloqueo de calzada (18.01%), esta última teniendo una participación mayor que en los particulares.

- A diferencia de los particulares y públicos, en los vehículos oficiales y diplomáticos la infracción más frecuente es el **estacionamiento prohibido (C02)** con 47.38%, superando al exceso de velocidad (38.25%). Aunque el volumen es bajo, este patrón sugiere un comportamiento diferenciado de este grupo.


---

**Hallazgos:**

- **El exceso de velocidad (C29) es la infracción predominante en todos los servicios**: Representa entre el 38% y el 49% de los comparendos en cada categoría, siendo ligeramente más alta en vehículos públicos (49.41%) que en particulares (45.85%).

- **El estacionamiento prohibido (C02) muestra el patrón más diferenciado**: Es la infracción más frecuente en vehículos OTROS (47.38%), la segunda en particulares (35.43%), y la tercera en públicos (20.82%). Esto sugiere que los vehículos oficiales y diplomáticos tienen una mayor propensión a estacionarse en sitios prohibidos.

- **Los vehículos públicos tienen una mayor proporción de infracciones de bloqueo de calzada (C03)**: Con un 18.01%, duplican la proporción de los particulares (9.39%). Esto podría estar relacionado con las maniobras que realizan los vehículos de transporte público al recoger o dejar pasajeros.

- **La infracción de carril restringido (C14) tiene muy baja participación**: Aunque es ligeramente más frecuente en OTROS (2.44%) y públicos (0.83%), su volumen es marginal debido a la implementación reciente de las cámaras Carril Bus.

**Nota:** La alta incidencia de bloqueo de calzada en vehículos públicos sugiere la necesidad de revisar las zonas de parada autorizadas para transporte público.

## Análisis de la Clase del Vehículo Infractor Según el Código de Infracción

Se construye una tabla pivote que cruza la clase del vehículo infractor con los códigos de infracción, agrupando las categorías de vehículos pesados (camión, bus, microbús, buseta, volqueta, tractocamión) en "VEHÍCULOS PESADOS" y las categorías minoritarias (motocarro, cuatrimoto, no reportado, sin clase) en "OTROS" para facilitar la visualización. Se utiliza la suma ponderada de `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones en cada combinación. Para cada clase de vehículo, se calcula el porcentaje que representa cada código de infracción sobre el total de sus comparendos, permitiendo identificar la contribución relativa de cada tipo de infracción. Los resultados se presentan mediante un mapa de calor interactivo, donde las clases de vehículo se ordenan de mayor a menor según su total de comparendos, y cada celda muestra tanto el valor absoluto como el porcentaje correspondiente. Esta visualización permite identificar qué tipos de infracción son más frecuentes en cada clase de vehículo, revelando patrones de comportamiento diferenciados entre automóviles, camionetas, motocicletas, camperos, vehículos pesados y otras clases minoritarias.

In [19]:
tabla_heatmap = df_comparendos_electronicos_copy.pivot_table(
    values='CANTIDAD_INFRACCIONES',
    index='CLASE_AGRUPADA',
    columns='COD_INFRACCION',
    aggfunc='sum',
    fill_value=0
)

orden = tabla_heatmap.sum(axis=1).sort_values(ascending=True).index
tabla_heatmap = tabla_heatmap.loc[orden]

total_por_fila = tabla_heatmap.sum(axis=1)
porcentajes = tabla_heatmap.div(total_por_fila, axis=0) * 100
porcentajes = porcentajes.round(2)
texto_combinado = tabla_heatmap.applymap(lambda x: f'{int(x):,}') + '<br>(' + porcentajes.applymap(lambda x: f'{x:.2f}%') + ')'

fig = go.Figure(data=go.Heatmap(
    z=tabla_heatmap.values,
    x=tabla_heatmap.columns,
    y=tabla_heatmap.index,
    colorscale='Blues',
    text=texto_combinado.values,
    texttemplate='%{text}',
    hovertemplate='Clase: %{y}<br>Código: %{x}<br>Comparendos: %{z:,.0f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Distribución de Comparendos por Clase del Vehículo y Código de Infracción',
        x=0.5,
        xanchor='center',
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(
        text='Código de Infracción',
        font=dict(weight='bold')
    ),
    yaxis_title=dict(
        text='Clase del Vehículo',
        font=dict(weight='bold')
    ),
    template='plotly_white',
    width=1055, 
    height=500
)

fig.show()

- Los automóviles concentran sus infracciones principalmente en estacionamiento prohibido (C02) con 41.36% y exceso de velocidad (C29) con 38.53%, que en conjunto representan el **79.89%** de sus comparendos.

- Al igual que los automóviles, las camionetas presentan una distribución casi equilibrada entre exceso de velocidad (41.10%) y estacionamiento prohibido (40.80%), que en conjunto representan el **81.90%** de sus comparendos.

- Las motocicletas tienen un patrón completamente diferente: el **76.97%** de sus infracciones corresponden a exceso de velocidad (C29), seguidas por paso de semáforo en rojo (D04) con 22.23%. Es notable que no registran infracciones de estacionamiento prohibido (C02) y transitar por sitios restringidos (C14).

- Similar a automóviles y camionetas, con predominancia de exceso de velocidad (44.50%) y estacionamiento prohibido (39.00%), que suman el **83.50%** de sus comparendos.

- Los vehículos pesados tienen una distribución más heterogénea. El exceso de velocidad (C29) representa el **52.63%**, seguido por bloqueo de calzada o intersección (C03) con 21.70%, que en este grupo tiene una participación mucho mayor que en otras clases. El estacionamiento prohibido (C02) solo representa el 16.43%, muy por debajo de automóviles y camionetas.

- Este grupo (que incluye motocarro, cuatrimoto, no reportado y sin clase) presenta un patrón similar a las motocicletas: el **74.03%** de sus infracciones corresponden a C29 (exceso de velocidad). Es notable que no registran infracciones de no respetar los pasos de peatones (C32).

---

**Hallazgos:**

- Para reducir infracciones en motocicletas, se debe priorizar el control de velocidad y el respeto a semáforos.
- Para vehículos pesados, además del control de velocidad, se debe prestar especial atención al bloqueo de calzada o intersección.
- Las campañas de estacionamiento prohibido deben enfocarse en automóviles, camionetas y camperos.

## Distribución Geográfica de Comparendos Electrónicos por Código de Infracción

Se examina, para cada código de infracción, la distribución de los comparendos electrónicos según la ubicación específica de la cámara (variable `Camara_y_direccion`), utilizando la suma ponderada de `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones por cada punto de control. Para cada código, se presentan tablas de frecuencias que muestran las ubicaciones con sus respectivos conteos, porcentajes dentro del código, y las fechas de primer y último registro, permitiendo identificar los puntos críticos por tipo de infracción, evaluar la antigüedad y continuidad operativa de cada cámara, y comprender la especialización geográfica del sistema de vigilancia para cada tipo de falta cometida.

In [20]:
codigos = df_comparendos_electronicos['COD_INFRACCION'].unique()

for codigo in codigos:
    df_codigo = df_comparendos_electronicos[df_comparendos_electronicos['COD_INFRACCION'] == codigo]
    
    total_codigo = df_codigo['CANTIDAD_INFRACCIONES'].sum()
    
    df_ubicaciones = df_codigo.groupby('Camara_y_direccion')['CANTIDAD_INFRACCIONES'].sum().sort_values(ascending=False).reset_index()
    df_ubicaciones.columns = ['Cámara y Dirección', 'Comparendos']
    
    df_ubicaciones['Porcentaje'] = (df_ubicaciones['Comparendos'] / total_codigo * 100).round(2)
    
    fechas = df_codigo.groupby('Camara_y_direccion')['fecha_comparendo'].agg(['min', 'max']).reset_index()
    fechas.columns = ['Cámara y Dirección', 'Primer Registro', 'Último Registro']
    fechas['Primer Registro'] = pd.to_datetime(fechas['Primer Registro']).dt.date
    fechas['Último Registro'] = pd.to_datetime(fechas['Último Registro']).dt.date
    
    df_ubicaciones = df_ubicaciones.merge(fechas, on='Cámara y Dirección')
    
    print(f"Distribución de Comparendos Electrónicos por Código {codigo}")
    pd.set_option('display.max_columns', None)  
    pd.set_option('display.width', None)        
    pd.set_option('display.max_colwidth', None) 
    display(df_ubicaciones.T)
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')
    pd.reset_option('display.max_colwidth')
    
    display(HTML("<hr style='border: 2px solid #696969; margin: 20px 0;'>"))

Distribución de Comparendos Electrónicos por Código C29


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29
Cámara y Dirección,VIA 11 CON CARRERA 8,CARRERA 51B CON CALLE 79,CALLE 30 CON CARRERA 6B,CARRERA 51B CON CALLE 103,CARRERA 6 CON CALLE 72,AVENIDA CIRCUNVALAR CON CARRERA 9G,CALLE 30 CON CARRERA 8,VIA 40 CON CALLE 73,CARRERA 15 CON CALLE 21,CALLE 45 CON CARRERA 20,CALLE 45 CON CARRERA 21,CALLE 84 ENTRE CARRERA 59 Y 59B,CALLE 94 CON CARRERA 58,CALLE 19 CON CARRERA 4C,CALLE 87 CON CARRERA 21,CALLE 45 CON CARRERA 1,CALLE 76 CON CARRERA 38C-100,CALLE 19 CON CARRERA 3D,CARRERA 53 CON CALLE 86,CALLE 56 CON CARRERA 14,CARRERA 46 CON CALLE 100,AVENIDA CIRCUNVALAR CON CARRERA 31,CALLE 98 CON CARRERA 56,CARRERA 53 ENTRE CALLE 104 Y 106,CALLE 61 CON CARRERA 35,CALLE 45 CON CARRERA 21 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 21 SENTIDO SUR-NORTE,CARRERA 27 CON CALLE 82C,AVENIDA CIRCUNVALAR (CALLE 110) ENTRE CARRERA 9G Y 12 SENTIDO SUR A NORTE,AVENIDA CIRCUNVALAR (CALLE 110) ENTRE CARRERA 9G Y 12 SENTIDO NORTE A SUR
Comparendos,52029,33293,24261,17116,16292,16186,15766,12983,9910,9273,8566,8479,7863,6980,6319,6266,6236,6215,4394,4263,2716,2225,2128,1465,1176,942,855,390,102,66
Porcentaje,18.27,11.69,8.52,6.01,5.72,5.68,5.54,4.56,3.48,3.26,3.01,2.98,2.76,2.45,2.22,2.2,2.19,2.18,1.54,1.5,0.95,0.78,0.75,0.51,0.41,0.33,0.3,0.14,0.04,0.02
Primer Registro,2018-01-01,2018-01-01,2018-01-01,2018-01-02,2018-01-01,2018-01-01,2018-01-01,2018-01-02,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-10-13,2018-10-16,2018-01-30,2024-11-22,2024-11-24,2018-01-02,2024-11-29,2024-12-06
Último Registro,2025-12-30,2025-12-28,2025-12-30,2025-12-12,2025-12-29,2024-11-28,2025-12-31,2025-12-20,2025-12-19,2024-11-18,2025-10-05,2025-12-30,2025-08-31,2025-12-30,2021-05-24,2025-12-30,2025-12-31,2025-12-28,2022-07-19,2021-05-03,2025-12-31,2019-07-23,2024-08-05,2025-12-31,2025-07-03,2025-12-31,2025-12-28,2020-03-02,2025-10-08,2025-12-29


Distribución de Comparendos Electrónicos por Código C32


,0,1,2,3,4,5,6,7,8
Cámara y Dirección,CARRERA 53 ENTRE CALLE 104 Y 106,CARRERA 46 CON CALLE 100,CALLE 45 CON CARRERA 20,CALLE 45 CON CARRERA 1,CALLE 30 CON CARRERA 8,CALLE 56 CON CARRERA 14,CALLE 45 CON CARRERA 21,CALLE 45 CON CARRERA 21 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 21 SENTIDO NORTE-SUR
Comparendos,1927,475,408,241,119,118,49,23,9
Porcentaje,57.2,14.1,12.11,7.15,3.53,3.5,1.45,0.68,0.27
Primer Registro,2018-11-28,2018-01-04,2018-01-01,2018-01-28,2018-01-05,2018-01-06,2018-01-12,2024-11-26,2025-02-24
Último Registro,2025-12-30,2025-11-30,2024-11-19,2025-10-26,2024-11-07,2021-04-29,2025-09-24,2025-09-02,2025-12-10


Distribución de Comparendos Electrónicos por Código D04


...

,0,1,2,3,4
Cámara y Dirección,Carro mal parqueo Sur,Carro mal parqueo Norte,Carro mal parqueo Sur - Norte,Carro mal parqueo alto norte,Carro mal parqueo alto sur
Comparendos,73544,65306,37993,17090,11040
Porcentaje,35.88,31.86,18.54,8.34,5.39
Primer Registro,2018-01-02,2018-01-02,2018-09-29,2023-06-28,2023-08-15
Último Registro,2025-12-31,2025-12-31,2025-12-30,2025-12-31,2025-12-31


Distribución de Comparendos Electrónicos por Código C03


,0,1,2,3,4,5,6
Cámara y Dirección,CALLE 82 CON CARRERA 51B,CALLE 82 CON CARRERA 56,CALLE 45B CON CARRERA 14,CALLE 72 CON CARRERA 44,CALLE 53 CON CARRERA 45,CALLE 70 CON CARRERA 46,CALLE 34 CON CARRERA 45
Comparendos,19626,12032,10568,8849,6781,4241,3133
Porcentaje,30.09,18.45,16.2,13.57,10.4,6.5,4.8
Primer Registro,2018-01-10,2018-01-09,2018-01-02,2018-01-10,2018-01-04,2018-01-09,2018-01-03
Último Registro,2025-12-30,2025-12-30,2025-12-31,2025-11-07,2025-12-30,2025-12-29,2025-12-31


Distribución de Comparendos Electrónicos por Código C14


,0,1,2,3,4,5,6,7,8,9
Cámara y Dirección,CARRERA 46 CON CALLE 34 SENTIDO SUR-NORTE,CALLE 70 CON CARRERA 46 SENTIDO SUR-NORTE,CALLE 70 CON CARRERA 46 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 8 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 33 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 43 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 43 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 1E SENTIDO SUR-NORTE,CARRERA 46 CON CALLE 34 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 1E SENTIDO NORTE-SUR
Comparendos,725,562,300,69,32,18,9,8,4,3
Porcentaje,41.91,32.49,17.34,3.99,1.85,1.04,0.52,0.46,0.23,0.17
Primer Registro,2025-10-23,2025-10-21,2025-10-21,2025-10-23,2025-10-23,2025-10-25,2025-10-24,2025-10-21,2025-10-21,2025-10-21
Último Registro,2025-11-17,2025-12-31,2025-12-11,2025-12-29,2025-12-29,2025-11-05,2025-11-02,2025-10-24,2025-10-25,2025-10-23


**Código C29 - Exceso de velocidad (284,755 infracciones)**:

- Las primeras 5 ubicaciones concentran el **50.21%** de todas las infracciones por exceso de velocidad.
- La "VIA 11 CON CARRERA 8" es el punto más crítico, con más de 52 mil detecciones, equivalente al 18.27% del total de C29.
- Se observan ubicaciones con sentidos diferenciados en la parte baja de la tabla (ej: CALLE 45 CON CARRERA 21 SENTIDO NORTE-SUR y SUR-NORTE), lo que indica que en algunos puntos la dirección de circulación influye en la detección.
- La mayoría de las ubicaciones operan desde 2018 hasta 2025, lo que indica una cobertura continua a lo largo de todo el período.

**Código C32 - No respetar paso de peatones (3,369 infracciones)**

- La infracción C32 tiene un volumen significativamente menor que C29 (3,369 vs 284,755).
- La "CARRERA 53 ENTRE CALLE 104 Y 106" concentra más de la mitad (57.2%) de todas las detecciones de esta infracción.
- Las primeras 3 ubicaciones concentran el **83.41%** del total de C32.
- La mayoría de los registros se concentran entre 2018 y 2024, con pocos registros en 2025.

**Código D04 - No detenerse ante semáforo en rojo o una señal de «pare» (54,303 infracciones)**

- Las primeras 2 ubicaciones concentran el **66.07%** de todas las infracciones por paso de semáforo en rojo o señal de «pare».
- La "CARRERA 53 ENTRE CALLE 104 Y 106" y la "CALLE 45 CON CARRERA 1" son los puntos más críticos para esta infracción.
- Operación continua desde 2018 hasta 2025 en la mayoría de ubicaciones.

**Código C02 - Estacionamiento prohibido (204,973 infracciones)**

- Las dos primeras ubicaciones ("Sur" y "Norte") concentran el **67.74%** de todas las infracciones de estacionamiento prohibido.
- Operación continua desde 2018 hasta 2025, con algunas cámaras más recientes (2023).

**Código C03 - Bloqueo de calzada o intersección (65,230 infracciones)**

- Las primeras 5 ubicaciones concentran el **88.71%** de todas las infracciones por bloqueo de calzada.
- La "CALLE 82 CON CARRERA 51B" es el punto más crítico, con el 30.09% del total de C03.
- La mayoría de las ubicaciones operan desde 2018, con algunas más recientes (2022).

**Código C14 - Transitar por sitios restringidos (1,730 infracciones)**

- La "CARRERA 46 CON CALLE 34 SENTIDO SUR-NORTE" concentra el **41.91%** de todas las infracciones por invasión de carril restringido.
- Las primeras 3 ubicaciones concentran el **91.74%** del total de C14.
- **Asimetría por sentido**: En la CARRERA 46 CON CALLE 34, el sentido SUR-NORTE (725) tiene 181 veces más infracciones que el sentido NORTE-SUR (4). En la CALLE 70 CON CARRERA 46, el sentido SUR-NORTE (562) casi duplica al NORTE-SUR (300).
- Todas las ubicaciones comenzaron a operar en octubre de 2025, lo que confirma la implementación reciente de las cámaras Carril Bus a partir del 17 de octubre de 2025.

---

**Hallazgos:**

- Cada código de infracción tiene sus propios puntos críticos, lo que refleja una estrategia de ubicación de cámaras según el tipo de infracción a controlar.

- La "CARRERA 53 ENTRE CALLE 104 Y 106" y la "CALLE 45 CON CARRERA 1" son puntos críticos tanto para no respeto al paso de peatones como para paso de semáforo en rojo, lo que sugiere que estas intersecciones concentran múltiples riesgos viales.

- Con solo 3 ubicaciones que concentran el 91.74% de las detecciones, la infracción de carril restringido está altamente localizada, lo que es coherente con su implementación reciente.

---

**Notas:**
- Para reducir el exceso de velocidad, se debe priorizar la "VIA 11 CON CARRERA 8" y el corredor de la CARRERA 51B.
- Para el paso de semáforo en rojo, la "CARRERA 53 ENTRE CALLE 104 Y 106" y la "CALLE 45 CON CARRERA 1" requieren especial atención.
- Para el estacionamiento prohibido, las cámaras "Sur" y "Norte" son las más estratégicas.
- Para la invasión de carril restringido, la "CARRERA 46 CON CALLE 34 SENTIDO SUR-NORTE" es el punto crítico absoluto.

## Evolución Mensual de Comparendos Electrónicos por Código de Infracción

Se examina la evolución mensual de los comparendos electrónicos por cada código de infracción (C02, C03, C14, C29, C32, D04), agrupando los registros por mes y sumando la columna `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones por cada período. Para cada código, se presentan tablas de frecuencias que muestran los meses con mayor y menor volumen, junto con sus porcentajes de participación dentro del total de ese código. Complementariamente, se generan gráficos de líneas interactivos que ilustran la tendencia temporal de cada infracción, incluyendo estadísticos como el promedio mensual, la desviación estándar, y los meses con valores máximo y mínimo, con un área sombreada que destaca el período crítico de la pandemia de COVID-19 para todos los códigos excepto C14 (dada su implementación reciente). Este análisis permite identificar patrones estacionales diferenciados por tipo de infracción, evaluar el impacto de eventos externos en cada código, y comparar el comportamiento temporal de las distintas faltas de tránsito.

In [21]:
infracciones_por_mes_codigo = df_comparendos_electronicos_copy.groupby(['año_mes', 'COD_INFRACCION'])['CANTIDAD_INFRACCIONES'].sum().reset_index()

total_infracciones = df_comparendos_electronicos['CANTIDAD_INFRACCIONES'].sum()

for codigo in codigos:
    df_codigo_mensual = infracciones_por_mes_codigo[infracciones_por_mes_codigo['COD_INFRACCION'] == codigo].copy()
    df_codigo_mensual = df_codigo_mensual[['año_mes', 'CANTIDAD_INFRACCIONES']].sort_values('año_mes').reset_index(drop=True)
    df_codigo_mensual.columns = ['Mes', 'Comparendos']
    total_codigo = df_codigo_mensual['Comparendos'].sum()
    
    df_codigo_mensual['Porcentaje'] = (df_codigo_mensual['Comparendos'] / total_codigo * 100).round(2)
    df_codigo_mensual = df_codigo_mensual.sort_values('Comparendos', ascending=False).reset_index(drop=True)
    
    print(f"Distribución de Comparendos Electrónicos por Mes - Código {codigo}")
    pd.set_option('display.max_columns', None)  
    pd.set_option('display.width', None)        
    pd.set_option('display.max_colwidth', None) 
    display(df_codigo_mensual.T)
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')
    pd.reset_option('display.max_colwidth')
    
    display(HTML("<hr style='border: 2px solid #696969; margin: 20px 0;'>"))

for codigo in codigos:
    df_codigo = infracciones_por_mes_codigo[infracciones_por_mes_codigo['COD_INFRACCION'] == codigo]
    
    total_codigo = df_codigo['CANTIDAD_INFRACCIONES'].sum()
    desviacion_codigo = df_codigo['CANTIDAD_INFRACCIONES'].std()
    promedio_codigo = df_codigo['CANTIDAD_INFRACCIONES'].mean()
    
    meses_codigo = sorted(df_codigo['año_mes'].unique())
    
    if len(meses_codigo) > 12:
        tick_positions = meses_codigo[::6]
    elif len(meses_codigo) > 6:
        tick_positions = meses_codigo[::3]
    else:
        tick_positions = meses_codigo
    
    max_valor = df_codigo['CANTIDAD_INFRACCIONES'].max()
    max_fila = df_codigo[df_codigo['CANTIDAD_INFRACCIONES'] == max_valor].iloc[0]
    max_mes = max_fila['año_mes']
    
    min_valor = df_codigo['CANTIDAD_INFRACCIONES'].min()
    min_fila = df_codigo[df_codigo['CANTIDAD_INFRACCIONES'] == min_valor].iloc[0]
    min_mes = min_fila['año_mes']
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=df_codigo['año_mes'],
        y=df_codigo['CANTIDAD_INFRACCIONES'],
        mode='lines+markers',
        name=codigo,
        line=dict(color='cornflowerblue', width=2),
        marker=dict(size=4, color='cornflowerblue'),
        hovertemplate=f'Código: {codigo}<br>Mes: %{{x}}<br>Comparendos: %{{y:,.0f}}<extra></extra>',
        showlegend=False
    ))
    
    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='none',
        name=f'Total: {int(total_codigo):,}',
        showlegend=True
    ))
    
    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='none',
        name=f'Std: {int(desviacion_codigo):,}',
        showlegend=True
    ))
    
    x_min = df_codigo['año_mes'].iloc[0]
    x_max = df_codigo['año_mes'].iloc[-1]
    fig.add_trace(go.Scatter(
        x=[x_min, x_max],
        y=[promedio_codigo, promedio_codigo],
        mode='lines',
        name=f'Promedio: {int(promedio_codigo):,}',
        line=dict(color='red', width=1.4, dash='dot'),
        showlegend=True,
        hoverinfo='none'
    ))
    
    fig.add_trace(go.Scatter(
        x=[max_mes],
        y=[max_valor],
        mode='markers',
        name=f'Máximo: {max_mes} ({int(max_valor):,})',
        marker=dict(size=8, color='red', symbol='circle'),
        hoverinfo='none'
    ))
    
    fig.add_trace(go.Scatter(
        x=[min_mes],
        y=[min_valor],
        mode='markers',
        name=f'Mínimo: {min_mes} ({int(min_valor):,})',
        marker=dict(size=8, color='green', symbol='circle'),
        hoverinfo='none'
    ))
    
    if codigo != 'C14':
        fig.add_vrect(
            x0="2020-03", x1="2020-12",
            fillcolor="red", opacity=0.1,
            line_width=0,
            annotation_text="COVID-19", 
            annotation_position="top left",
            annotation=dict(font_size=12, font_color="red")
        )
    
    fig.update_layout(
        title=dict(
            text=f'Evolución Mensual de Comparendos - Código {codigo}',
            x=0.5,
            xanchor='center',
            font=dict(size=16, weight='bold')
        ),
        xaxis_title=dict(
            text='Año-Mes',
            font=dict(weight='bold')
        ),
        yaxis_title=dict(
            text='Cantidad de Comparendos',
            font=dict(weight='bold')
        ),
        template='plotly_white',
        hovermode='x unified',
        legend=dict(
            x=1, 
            y=1, 
            xanchor='center',
            yanchor='top',
            bgcolor='rgba(255,255,255,0.5)',
            font=dict(size=12)
        ),
        width=1055,
        height=500
    )
    
    fig.update_xaxes(
        tickangle=-45,
        tickvals=tick_positions,
        ticktext=tick_positions
    )
    
    fig.show()

Distribución de Comparendos Electrónicos por Mes - Código C29


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95
Mes,2019-03,2019-04,2019-08,2019-07,2019-01,2022-12,2019-02,2019-06,2018-01,2021-01,2019-05,2024-03,2022-04,2019-10,2024-06,2020-01,2019-09,2018-11,2020-03,2024-05,2019-11,2023-04,2019-12,2020-02,2020-12,2018-10,2023-07,2020-08,2020-04,2024-07,2022-05,2018-03,2018-05,2023-03,2023-12,2018-04,2024-04,2018-12,2024-01,2023-05,2023-09,2020-10,2023-06,2018-09,2020-09,2024-02,2023-10,2022-11,2022-03,2024-08,2023-02,2022-07,2023-08,2022-06,2018-02,2021-12,2018-07,2021-07,2022-01,2021-04,2021-02,2018-08,2023-11,2018-06,2020-11,2021-03,2021-08,2021-05,2024-09,2022-10,2022-09,2021-11,2021-09,2020-06,2020-07,2021-06,2024-10,2021-10,2022-02,2022-08,2025-03,2025-01,2024-11,2025-04,2024-12,2025-05,2025-06,2025-07,2023-01,2025-02,2025-09,2025-08,2025-10,2020-05,2025-12,2025-11
Comparendos,5402,4959,4745,4692,4450,4383,4320,4003,3977,3976,3967,3883,3864,3840,3834,3813,3717,3705,3670,3662,3638,3626,3609,3594,3547,3500,3487,3467,3442,3415,3380,3362,3360,3358,3323,3291,3272,3267,3234,3211,3207,3204,3181,3135,3126,3116,3113,3108,3103,3101,3085,3069,3054,3045,3025,2959,2958,2946,2941,2922,2919,2905,2905,2902,2830,2769,2713,2688,2610,2411,2406,2362,2346,2335,2312,2299,2264,2163,1974,1949,1846,1749,1733,1701,1512,1507,1506,1367,1350,1308,1194,1173,1153,1125,1062,764
Porcentaje,1.9,1.74,1.67,1.65,1.56,1.54,1.52,1.41,1.4,1.4,1.39,1.36,1.36,1.35,1.35,1.34,1.31,1.3,1.29,1.29,1.28,1.27,1.27,1.26,1.25,1.23,1.22,1.22,1.21,1.2,1.19,1.18,1.18,1.18,1.17,1.16,1.15,1.15,1.14,1.13,1.13,1.13,1.12,1.1,1.1,1.09,1.09,1.09,1.09,1.09,1.08,1.08,1.07,1.07,1.06,1.04,1.04,1.03,1.03,1.03,1.03,1.02,1.02,1.02,0.99,0.97,0.95,0.94,0.92,0.85,0.84,0.83,0.82,0.82,0.81,0.81,0.8,0.76,0.69,0.68,0.65,0.61,0.61,0.6,0.53,0.53,0.53,0.48,0.47,0.46,0.42,0.41,0.4,0.4,0.37,0.27


Distribución de Comparendos Electrónicos por Mes - Código C32


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95
Mes,2018-12,2019-01,2019-03,2019-02,2019-04,2024-06,2019-07,2019-12,2024-04,2022-11,2024-05,2024-09,2020-02,2019-05,2020-11,2022-12,2024-07,2023-06,2024-01,2019-11,2019-08,2020-01,2019-09,2023-04,2020-08,2022-09,2020-10,2023-05,2024-11,2023-02,2019-06,2020-03,2022-05,2024-03,2023-09,2020-09,2020-12,2024-02,2024-10,2018-11,2019-10,2018-01,2023-03,2024-12,2022-06,2022-10,2021-02,2018-10,2021-04,2025-01,2021-01,2023-08,2023-12,2025-07,2021-07,2021-03,2022-01,2021-09,2023-11,2020-04,2020-06,2018-02,2021-12,2022-07,2023-07,2018-03,2025-08,2022-04,2025-05,2025-04,2025-06,2020-07,2025-03,2021-05,2021-08,2022-02,2022-03,2025-02,2021-10,2025-09,2024-08,2023-10,2021-06,2023-01,2022-08,2018-04,2021-11,2018-06,2018-05,2018-07,2018-08,2018-09,2020-05,2025-10,2025-12,2025-11
Comparendos,140,124,107,96,79,72,68,62,62,61,55,54,54,51,51,50,50,48,48,47,47,46,45,45,45,44,44,43,40,40,40,39,39,39,39,39,37,36,35,35,35,34,32,32,32,32,30,29,29,29,29,28,28,28,28,28,27,27,27,27,25,25,24,23,23,22,22,22,21,21,21,21,21,20,20,19,18,18,17,16,15,15,15,15,14,13,11,11,9,9,7,7,7,6,6,3
Porcentaje,4.16,3.68,3.18,2.85,2.34,2.14,2.02,1.84,1.84,1.81,1.63,1.6,1.6,1.51,1.51,1.48,1.48,1.42,1.42,1.4,1.4,1.37,1.34,1.34,1.34,1.31,1.31,1.28,1.19,1.19,1.19,1.16,1.16,1.16,1.16,1.16,1.1,1.07,1.04,1.04,1.04,1.01,0.95,0.95,0.95,0.95,0.89,0.86,0.86,0.86,0.86,0.83,0.83,0.83,0.83,0.83,0.8,0.8,0.8,0.8,0.74,0.74,0.71,0.68,0.68,0.65,0.65,0.65,0.62,0.62,0.62,0.62,0.62,0.59,0.59,0.56,0.53,0.53,0.5,0.47,0.45,0.45,0.45,0.45,0.42,0.39,0.33,0.33,0.27,0.27,0.21,0.21,0.21,0.18,0.18,0.09


Distribución de Comparendos Electrónicos por Mes - Código D04


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89
Mes,2019-05,2023-09,2019-06,2023-08,2023-10,2018-01,2021-02,2023-11,2019-07,2023-07,2018-03,2018-05,2019-04,2018-04,2020-10,2024-04,2018-10,2018-11,2018-06,2023-12,2021-03,2024-01,2025-10,2018-07,2018-02,2024-07,2024-05,2025-09,2018-08,2019-08,2024-10,2023-06,2024-03,2018-09,2024-02,2025-03,2020-01,2024-06,2025-05,2025-07,2018-12,2025-04,2025-02,2025-01,2023-05,2025-11,2024-08,2019-10,2019-09,2020-02,2019-03,2024-12,2024-09,2023-03,2019-12,2020-11,2024-11,2022-06,2025-06,2020-12,2019-02,2019-11,2025-12,2022-07,2022-10,2022-08,2022-11,2023-04,2022-05,2021-10,2021-07,2025-08,2021-01,2021-11,2022-09,2022-02,2022-12,2022-03,2021-09,2021-04,2022-01,2020-03,2022-04,2021-08,2021-12,2019-01,2021-05,2023-02,2020-09,2021-06
Comparendos,4084,3822,3647,3581,3575,3375,3329,3310,3267,3263,3247,3101,3065,3039,3031,3027,2958,2943,2942,2879,2848,2810,2766,2761,2751,2731,2729,2713,2707,2703,2653,2594,2555,2499,2496,2476,2444,2442,2439,2422,2420,2418,2413,2411,2410,2376,2370,2362,2349,2308,2283,2275,2264,2258,2185,2179,2146,2145,2119,2074,2025,2008,1898,1829,1707,1682,1667,1656,1567,1520,1471,1423,1392,1385,1362,1353,1340,1332,1317,1309,1282,1265,1118,1095,1094,1000,900,888,684,615
Porcentaje,1.99,1.86,1.78,1.75,1.74,1.65,1.62,1.61,1.59,1.59,1.58,1.51,1.5,1.48,1.48,1.48,1.44,1.44,1.44,1.4,1.39,1.37,1.35,1.35,1.34,1.33,1.33,1.32,1.32,1.32,1.29,1.27,1.25,1.22,1.22,1.21,1.19,1.19,1.19,1.18,1.18,1.18,1.18,1.18,1.18,1.16,1.16,1.15,1.15,1.13,1.11,1.11,1.1,1.1,1.07,1.06,1.05,1.05,1.03,1.01,0.99,0.98,0.93,0.89,0.83,0.82,0.81,0.81,0.76,0.74,0.72,0.69,0.68,0.68,0.66,0.66,0.65,0.65,0.64,0.64,0.63,0.62,0.55,0.53,0.53,0.49,0.44,0.43,0.33,0.3


Distribución de Comparendos Electrónicos por Mes - Código C03


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91
Mes,2022-06,2022-09,2022-07,2022-10,2022-08,2025-10,2024-09,2022-12,2022-11,2023-12,2022-05,2023-07,2024-10,2023-06,2024-04,2024-05,2023-03,2023-08,2023-10,2023-09,2023-05,2025-04,2024-11,2025-11,2024-12,2019-06,2023-04,2024-02,2023-02,2024-03,2024-07,2023-11,2019-03,2025-09,2024-06,2025-01,2022-04,2025-12,2025-02,2024-08,2019-02,2019-05,2025-07,2024-01,2025-05,2020-09,2025-03,2019-04,2019-07,2020-02,2022-03,2023-01,2020-01,2019-08,2018-12,2025-06,2018-06,2019-12,2018-11,2018-07,2018-09,2025-08,2018-05,2018-10,2018-08,2019-10,2018-01,2020-08,2018-04,2018-03,2018-02,2019-11,2019-09,2020-03,2020-10,2019-01,2020-11,2021-12,2021-01,2021-07,2021-06,2021-03,2021-10,2021-11,2021-04,2020-12,2021-09,2022-02,2021-02,2021-05,2021-08,2022-01
Comparendos,2380,2146,2145,1992,1983,1934,1859,1822,1778,1747,1739,1647,1549,1495,1446,1439,1278,1256,1236,1153,1149,1123,1113,1034,1006,977,971,924,921,920,919,909,866,773,769,765,752,693,690,682,675,657,648,578,553,539,511,495,447,430,419,407,402,367,349,348,331,331,329,325,321,304,290,256,256,217,201,201,195,189,188,180,161,146,146,108,89,73,72,64,60,58,55,51,45,41,35,33,31,25,15,3
Porcentaje,3.65,3.29,3.29,3.05,3.04,2.96,2.85,2.79,2.73,2.68,2.67,2.52,2.37,2.29,2.22,2.21,1.96,1.93,1.89,1.77,1.76,1.72,1.71,1.59,1.54,1.5,1.49,1.42,1.41,1.41,1.41,1.39,1.33,1.19,1.18,1.17,1.15,1.06,1.06,1.05,1.03,1.01,0.99,0.89,0.85,0.83,0.78,0.76,0.69,0.66,0.64,0.62,0.62,0.56,0.54,0.53,0.51,0.51,0.5,0.5,0.49,0.47,0.44,0.39,0.39,0.33,0.31,0.31,0.3,0.29,0.29,0.28,0.25,0.22,0.22,0.17,0.14,0.11,0.11,0.1,0.09,0.09,0.08,0.08,0.07,0.06,0.05,0.05,0.05,0.04,0.02,0.0


Distribución de Comparendos Electrónicos por Mes - Código C14


,0,1,2
Mes,2025-11,2025-12,2025-10
Comparendos,1073,353,304
Porcentaje,62.02,20.4,17.57


**Posible explicación para los puntos máximos de los codigos (C29, D04, C02) - períodos 2019-03 y 2019-05:** 

Los máximos históricos registrados en los códigos C29 (exceso de velocidad), D04 (paso de semáforo en rojo o señal de «pare») y C02 (estacionamiento prohibido) durante ciertos meses pueden explicarse por los múltiples cierres viales que se presentaron en Barranquilla en esos períodos. Los cierres viales generan alteraciones significativas en los patrones normales de circulación, desviando el tránsito hacia vías alternas y creando condiciones de congestión que incrementan la probabilidad de comisión de infracciones. En el caso del exceso de velocidad (C29), los conductores tienden a acelerar en vías alternas menos controladas o al intentar recuperar tiempo perdido por los desvíos. Para el paso de semáforo en rojo o señal de «pare» (D04), la congestión y las demoras generan impaciencia entre los conductores, aumentando las infracciones en intersecciones. En cuanto al estacionamiento prohibido (C02), los cierres reducen la disponibilidad de estacionamiento en zonas comerciales y laborales, llevando a los conductores a estacionarse en sitios no permitidos. De esta manera, los picos en estos códigos de infracción son un reflejo directo de las condiciones anormales de movilidad generadas por los cierres viales en la ciudad.

**Fuentes**:
- https://barranquilla.gov.co/transito/por-eventos-de-pre-carnaval-cierres-de-vias-para-viernes-1-de-marzo
- https://barranquilla.gov.co/transito/por-carnavales-2019-cierre-de-vias
- https://barranquilla.gov.co/transito/por-izaje-de-equipos-cierres-de-vias-para-el-jueves-14-de-marzo-de-2019
- https://barranquilla.gov.co/transito/por-trabajos-de-ampliacion-de-la-calle-30-a-partir-del-martes-12-de-marzo-cierres-de-vias
- https://barranquilla.gov.co/transito/cierres-de-vias-para-el-domingo-17-de-marzo-de-2019
- https://barranquilla.gov.co/transito/por-reparacion-de-pavimento-cierre-total-de-la-carrera-50-entre-calles-85-y-86
- https://barranquilla.gov.co/transito/cierre-de-vias-para-el-sabado-23-de-marzo-de-2019
- https://barranquilla.gov.co/transito/cierres-de-vias-para-el-sabado-1-y-domingo-2-de-junio-de-20
- https://barranquilla.gov.co/transito/por-reparacion-de-pavimento-cierres-de-vias-a-partir-del-miercoles-5-de-junio-de-2019
- https://barranquilla.gov.co/transito/por-reparacion-de-pavimento-cierres-de-vias-para-el-martes-18-de-junio-de-2019
- https://barranquilla.gov.co/transito/cierres-de-vias-a-partir-del-viernes-21-de-junio-de-2019

---

**Posible explicación para el punto máximo del código (C32) - período 2018-12:**

El máximo histórico registrado en el código C32 (no respetar el paso de peatones) puede explicarse por los múltiples cierres viales y los partidos del Junior de Barranquilla, tanto nacionales como internacionales de importancia, que se presentaron durante ese período. Los cierres viales generan alteraciones en los patrones normales de circulación, creando condiciones de congestión que incrementan la impaciencia de los conductores y reducen su atención hacia los cruces peatonales. Por otro lado, los partidos de alta convocatoria del Junior atraen a miles de aficionados a las inmediaciones del estadio Metropolitano y otros puntos de encuentro, lo que genera un flujo masivo de peatones en las vías aledañas. En este contexto de alta afluencia peatonal y condiciones de tráfico atípicas, los conductores, con frecuencia, no ceden el paso en los cruces designados. 

**Fuentes:**
- https://barranquilla.gov.co/transito/medidas-para-el-partido-junior-vs-atletico-paranaense-este-miercoles-5-de-diciembre
- https://barranquilla.gov.co/transito/medidas-para-el-partido-junior-vs-independiente-medellin-es-sabado-8-de-diciembre-de-2018
- https://barranquilla.gov.co/transito/cierres-de-vias-para-el-miercoles-12-de-diciembre-de-2018
- https://barranquilla.gov.co/transito/cierres-de-vias-para-el-viernes-7sabado-8-y-lunes-10-de-diciembre-de-2018
- https://barranquilla.gov.co/transito/por-ciclopaseo-cierres-de-vias-para-el-domingo-16-de-diciembre-de-2018
- https://barranquilla.gov.co/transito/por-avance-en-la-obra-de-canalizacion-del-arroyo-la-felicidad-cierres-de-vias-para-el-jueves-13-de-diciembre-de-2018
- https://barranquilla.gov.co/transito/por-trabajos-de-ampliacion-de-la-calle-30-cierres-de-vias-a-partir-de-manana-19-de-diciembre
- https://barranquilla.gov.co/transito/por-obras-del-arroyo-de-la-calle-76-cierre-de-vias-a-partir-del-sabado-1-de-diiembre

---

**Posible explicación para el punto máximo del código (C03) - período 2022-06:** 

El máximo histórico registrado en el código C03 (bloquear calzada o intersección) puede explicarse por los múltiples cierres viales y la temporada de fuertes lluvias que se presentaron durante ese período. Los cierres viales generan alteraciones en los patrones normales de circulación, desviando el tránsito hacia vías alternas y creando condiciones de congestión que incrementan la probabilidad de que los conductores bloqueen intersecciones al intentar avanzar en medio del tráfico. Por otro lado, la temporada de fuertes lluvias agrava esta situación, ya que las precipitaciones intensas reducen la visibilidad, generan encharcamientos y afectan la fluidez vehicular, lo que lleva a los conductores a realizar maniobras desesperadas como cruzar intersecciones aunque no tengan espacio suficiente, quedando detenidos en medio de la calzada y bloqueando el flujo en direcciones transversales. La combinación de cierres viales que ya de por sí congestionan la ciudad, sumada a las difíciles condiciones de movilidad por las lluvias, crea un escenario propicio para que esta infracción alcance sus niveles máximos. 

**Fuentes:**
- https://barranquilla.gov.co/transito/por-instalacion-de-tuberias-e-instalacion-de-acometidas-cierres-de-vias-a-partir-del-viernes-17-de-junio
- https://barranquilla.gov.co/adi/atencion-a-los-cierres-en-la-circunvalar
- https://barranquilla.gov.co/transito/a-partir-de-manana-1-de-junio-de-2022-se-implementan-cambios-de-sentido-vial-en-el-sector-de-alameda-del-rio
- https://barranquilla.gov.co/transito/por-evento-vuelta-a-colombia-cierre-de-vias-para-el-viernes-3-de-junio-de-2022
- https://barranquilla.gov.co/salud/ante-la-temporada-de-lluvias-distrito-invita-a-no-bajar-la-guardia-frente-al-dengue
- https://barranquilla.gov.co/transito/cierres-de-vias-para-el-martes-7-de-junio
- https://barranquilla.gov.co/transito/cierres-de-vias-a-partir-del-sabado-4-de-junio-de-2022
- https://barranquilla.gov.co/salud/en-temporada-de-lluvias-autocuidado-y-responsabilidad-ante-enfermedades-respiratorias

---

**Posible explicación para los puntos mínimos de los codigos (C29, C32, D04) - período 2025-11:** 

Los puntos mínimos registrados en los códigos C29 (exceso de velocidad), C32 (no respetar paso de peatones) y D04 (paso de semáforo en rojo o señal de «pare») pueden explicarse por los múltiples cierres viales en zonas con cámaras y la inauguración del megapuente norte-sur del intercambiador vial de la Circunvalar durante ese período. Los cierres viales en zonas estratégicas donde operan las cámaras de detección redujeron temporalmente el flujo vehicular en esos puntos específicos, ya que el tránsito fue desviado hacia vías alternas, disminuyendo las oportunidades de cometer infracciones en los lugares habituales de monitoreo. Por otro lado, la inauguración del megapuente norte-sur del intercambiador vial de la Circunvalar representó una mejora significativa en la infraestructura vial de la ciudad, descongestionando puntos críticos, optimizando los tiempos de desplazamiento y reduciendo las condiciones de estrés y apresuramiento que suelen llevar a los conductores a cometer estas infracciones.

**Fuentes:**
- https://barranquilla.gov.co/obraspublicas/alcalde-char-habilita-puente-norte-sur-del-intercambiador-vial-de-la-avenida-circunvalar
- https://barranquilla.gov.co/mi-barranquilla/cierres-temporales-por-competencia-de-karts-y-evento-de-la-oficina-de-la-mujer
- https://barranquilla.gov.co/transito/medidas-que-regulan-circulacion-de-motociclistas-continuan-vigentes-en-barranquilla-hasta-octubre-de-2026
- https://barranquilla.gov.co/mi-barranquilla/cierre-temporal-por-obra-de-acueducto-en-la-avenida-circunvalar-con-puente-de-la-carrera-46

--- 

**Posible explicación para el punto mínimo del código (C02) - período 2021-06:** 

Durante junio de 2021, las cámaras móviles registraron su volúmen más bajo de detección de infracciones. Este comportamiento puede explicarse por el contexto de reactivación gradual de la ciudad después de los picos más críticos de la pandemia, combinado con el aumento sostenido del teletrabajo durante ese período. Si bien las medidas de confinamiento estricto de 2020 se habían flexibilizado, muchas empresas y entidades mantuvieron esquemas de trabajo remoto o híbrido, lo que redujo significativamente el número de vehículos en circulación. Sin embargo, a diferencia de lo que podría pensarse, esta reactivación no pudo implicar un aumento del tránsito vehicular, sino que las personas aprovecharon la nueva libertad para salir a caminar, pasear en bicicleta y disfrutar de actividades al aire libre que antes no podían realizar debido a las restricciones sanitarias. Este cambio en los hábitos de movilidad, con un mayor protagonismo de peatones y ciclistas, explica por qué las infracciones detectadas por cámaras móviles (enfocadas principalmente en estacionamiento prohibido) alcanzaron sus niveles más bajos durante este período, ya que simplemente había menos vehículos circulando y estacionándose en las calles.

**Fuentes:**
- https://barranquilla.gov.co/funcionarios/teletrabajo-una-tendencia-en-crecimiento
- https://barranquilla.gov.co/descubre/barranquilla-atlantico-anuncian-que-estan-abiertos-para-el-mundo
- https://barranquilla.gov.co/transito/vuelve-biciquilla-este-jueves-24-de-junio
- https://barranquilla.gov.co/cultura/concierto-musica-tradicional-baile-y-talleres-arte-reactivan-cultura-barranquilla
- https://barranquilla.gov.co/desarrolloeconomico/barranquilla-reactiva-inicia-nueva-etapa-de-reapertura-plena-integral
- https://barranquilla.gov.co/cultura/talleres-artisticos-baila-ton-y-conciertos-en-los-parques-reactivan-el-sector-cultural
- https://barranquilla.gov.co/mi-barranquilla/ven-vive-barranquilla-atlantico-alianza-impulsa-reactivacion
- https://barranquilla.gov.co/desarrolloeconomico/barranquilla-se-alista-para-comenzar-su-reactivacion-plena

---

**Posible explicación para el punto mínimo del código (C03) - período 2022-01:** 

El punto mínimo registrado en el código C03 (bloquear calzada o intersección) puede explicarse por los múltiples cierres viales en zonas con cámaras y la disminución en las cifras de accidentabilidad durante ese período. Los cierres viales en zonas estratégicas donde operan las cámaras de detección redujeron temporalmente el flujo vehicular en esos puntos específicos, ya que el tránsito fue desviado hacia vías alternas, disminuyendo las oportunidades de cometer infracciones de bloqueo de calzada en los lugares habituales de monitoreo. Por otro lado, la disminución en las cifras de accidentabilidad refleja una conducción más cuidadosa por parte de los conductores, posiblemente influenciada por campañas de educación vial, mayor presencia de autoridades de tránsito o condiciones de movilidad más fluidas. Una menor siniestralidad suele estar asociada con una conducción más responsable, lo que incluye evitar maniobras peligrosas como bloquear intersecciones al intentar avanzar en medio del tráfico. 

**Fuentes:**
- https://barranquilla.gov.co/transito/cierres-de-vias-a-partir-del-sabado-15-hasta-el-lunes-17-de-enero-de-2022
- https://barranquilla.gov.co/transito/por-construccion-de-canales-boxculvert-cierres-viales-desde-el-19-de-enero-hasta-el-30-de-junio-de-2022
- https://barranquilla.gov.co/transito/por-reconduccion-de-lineas-cierre-de-vias-para-el-viernes-21-de-enero-de-2022
- https://barranquilla.gov.co/transito/cierres-viales-por-instalacion-de-puente-provisional-sobre-la-avenida-del-rio
- https://barranquilla.gov.co/transito/por-carrera-atletica-san-silvestre-cierres-de-vias-para-el-viernes-31-de-diciembre
- https://barranquilla.gov.co/transito/durante-la-celebracion-del-ano-nuevo-se-disminuyo-la-accidentalidad-en-barranquilla

---

**Explicación para el punto mínimo del código (C14):** 

El mínimo registrado en las cámaras de carril bus durante octubre de 2025 se explica por el hecho de que estos equipos comenzaron a funcionar a partir del 17 de octubre de 2025, por lo que no tuvieron un mes completo de operación para realizar detecciones. Al haber iniciado sus labores a mediados de octubre, el período de octubre representa menos quince días de funcionamiento parcial, sin que se haya alcanzado una operación continua a lo largo de un ciclo mensual completo.

### Distribución Mensual y Anual de Comparendos Electrónicos por Código de Infracción

Se construyen tablas pivote que cruzan los meses y los años para cada código de infracción (C29, C02, C03, D04 y C32), utilizando la suma ponderada de `CANTIDAD_INFRACCIONES` para visualizar el volumen de infracciones en cada combinación temporal. Los resultados se presentan mediante mapas de calor independientes para cada código, con los meses ordenados de diciembre a enero para facilitar la identificación de patrones estacionales específicos de cada tipo de infracción. Esta visualización permite comparar el comportamiento temporal de los diferentes códigos.

In [22]:
codigos_heatmap = ['C29', 'C02', 'C03', 'D04', 'C32']

for codigo in codigos_heatmap:
    df_codigo = df_comparendos_electronicos_copy[df_comparendos_electronicos_copy['COD_INFRACCION'] == codigo]
    
    df_codigo['año'] = df_codigo['fecha_comparendo'].dt.year
    df_codigo['mes_numero'] = df_codigo['fecha_comparendo'].dt.month
    
    tabla_heatmap = df_codigo.pivot_table(
        values='CANTIDAD_INFRACCIONES',
        index='mes_numero',
        columns='año',
        aggfunc='sum',
        fill_value=0
    )
    
    tabla_heatmap.index = nombres_meses
    tabla_heatmap = tabla_heatmap.iloc[::-1]
    
    fig = go.Figure(data=go.Heatmap(
        z=tabla_heatmap.values,
        x=tabla_heatmap.columns,
        y=tabla_heatmap.index,
        colorscale='blues',
        text=tabla_heatmap.values,
        texttemplate='%{text:,.0f}',
        hovertemplate=f'Código: {codigo}<br>Mes: %{{y}}<br>Año: %{{x}}<br>Comparendos: %{{z:,.0f}}<extra></extra>'
    ))
    
    fig.update_layout(
        title=dict(
            text=f'Distribución de Comparendos Electrónicos por Año y Mes - Código {codigo}',
            x=0.5,
            xanchor='center',
            font=dict(size=16, weight='bold')
        ),
        xaxis_title=dict(
            text='Año',
            font=dict(weight='bold')
        ),
        yaxis_title=dict(
            text='Mes',
            font=dict(weight='bold')
        ),
        template='plotly_white',
        width=1055, 
        height=500
    )
    
    fig.show()

### Distribución Mensual de Comparendos por Código de Infracción

Se examina la variabilidad mensual de los comparendos electrónicos desagregada por cada código de infracción (C29, C02, C03, D04 y C32), agrupando los registros por mes y año y sumando la columna `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones por cada combinación. Para cada código, se presentan estadísticos descriptivos (media, desviación estándar, mínimo, máximo y cuartiles) para cada mes, con el fin de caracterizar su comportamiento típico y su dispersión interanual de manera específica por tipo de infracción. Complementariamente, se generan diagramas de caja (boxplot) que visualizan la distribución de las infracciones para cada mes, incluyendo la media como referencia. Para validar si las diferencias observadas entre meses son estadísticamente significativas para cada código, se aplica un análisis de varianza que incluye la prueba de normalidad de Shapiro-Wilk para evaluar la distribución de los datos por mes, la prueba de Levene para verificar la homogeneidad de varianzas, y dependiendo del cumplimiento de los supuestos, se realiza un ANOVA paramétrico (con prueba post-hoc de Tukey HSD) o un Kruskal-Wallis no paramétrico (con prueba post-hoc de Dunn y corrección de Bonferroni). Este análisis permite identificar, de manera diferenciada por código, qué meses presentan mayor variabilidad en la detección de cada infracción, detectar meses con comportamientos atípicos específicos de cada falta de tránsito, comparar la estabilidad estacional entre los distintos tipos de infracción, y determinar si existe estacionalidad estadísticamente significativa en el volumen de comparendos para cada código.

In [23]:
for codigo in codigos_heatmap:
    df_codigo = df_comparendos_electronicos_copy[df_comparendos_electronicos_copy['COD_INFRACCION'] == codigo]
    df_boxplot_codigo = df_codigo.groupby(['mes_nombre', 'año'])['CANTIDAD_INFRACCIONES'].sum().reset_index()
    df_boxplot_codigo.columns = ['mes', 'año', 'infracciones']
    df_boxplot_codigo['mes_nombre'] = df_boxplot_codigo['mes'].apply(lambda x: nombres_meses[x-1])

    print(f"Estadísticas Descriptivas - Código {codigo}")
    estadisticas = df_boxplot_codigo.groupby('mes_nombre')['infracciones'].describe().round(0).fillna(0).astype(int).reindex(nombres_meses)
    display(estadisticas.T)
    
    display(HTML("<hr style='border: 1px solid #696969; margin: 20px 0;'>"))
    
    fig = go.Figure()
    
    fig.add_trace(go.Box(
        x=df_boxplot_codigo['mes_nombre'],
        y=df_boxplot_codigo['infracciones'],
        marker_color='mediumpurple',
        name='',
        boxmean=True,
        hovertemplate='Mes: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
    ))
    
    fig.update_layout(
        title=dict(
            text=f'Distribución de Comparendos Electrónicos por Mes - Código {codigo}',
            x=0.5,
            xanchor='center',
            font=dict(size=16, weight='bold')
        ),
        xaxis_title=dict(
            text='Mes',
            font=dict(weight='bold')
        ),
        yaxis_title=dict(
            text='Cantidad de Comparendos',
            font=dict(weight='bold')
        ),
        template='plotly_white',
        width=1055,
        height=500
    )
    
    fig.show()
    
    df_anova = df_boxplot_codigo.copy()
    
    meses_grupos = [
        df_anova[df_anova['mes_nombre'] == mes]['infracciones'].values 
        for mes in nombres_meses
    ]
    

    print("Prueba de normalidad (Shapiro-Wilk) por mes:")
    
    normalidad = []
    for mes, grupo in zip(nombres_meses, meses_grupos):
        if len(grupo) >= 3:  
            stat, p = stats.shapiro(grupo)
            normalidad.append(p > 0.05)
            print(f"{mes}: p-valor = {p:.6f} -> {'Normal' if p > 0.05 else 'No normal'}")
        else:
            print(f"{mes}: No hay suficientes datos para evaluar normalidad")
            normalidad.append(False)
    
    cumple_normalidad = all(normalidad) if normalidad else False
    print(f"¿Se cumple normalidad en todos los meses? {'Sí' if cumple_normalidad else 'No'}")
    
    grupos_validos = [g for g in meses_grupos if len(g) > 0]
    
    if len(grupos_validos) > 1:
        stat_levene, p_levene = stats.levene(*grupos_validos)
        print(f"\nLevene p-valor: {p_levene:.6f}")
        print(f"¿Varianzas iguales? {'Sí' if p_levene > 0.05 else 'No'}")
        
        if cumple_normalidad and p_levene > 0.05:
            print("\nSe cumplen los supuestos → Se aplica ANOVA")
            
            f_stat, p_valor = stats.f_oneway(*meses_grupos)
            
            print(f"Estadístico F: {f_stat:.4f}")
            print(f"P-valor: {p_valor:.6f}")
            
            print(f"¿Hay diferencias entre meses? {'Sí' if p_valor < 0.05 else 'No'}")
            
            if p_valor < 0.05:
                print("\nPrueba post-hoc Tukey HSD:")
                
                tukey = pairwise_tukeyhsd(
                    endog=df_anova['infracciones'],
                    groups=df_anova['mes_nombre'],
                    alpha=0.05
                )
                
                print(tukey)
        
        else:
            print("\nNo se cumplen los supuestos → Se aplica Kruskal-Wallis")
            
            h_stat, p_valor = stats.kruskal(*grupos_validos)
            
            print(f"Estadístico H: {h_stat:.4f}")
            print(f"P-valor: {p_valor:.6f}")
            
            print(f"¿Hay diferencias entre meses? {'Sí' if p_valor < 0.05 else 'No'}")
            
            if p_valor < 0.05:
                print("\nPrueba post-hoc Dunn (con corrección Bonferroni):")
                
                import scikit_posthocs as sp
                
                df_dunn = df_anova[['mes_nombre', 'infracciones']].copy()
                
                dunn = sp.posthoc_dunn(
                    df_dunn,
                    val_col='infracciones',
                    group_col='mes_nombre',
                    p_adjust='bonferroni'
                )
                
                dunn = dunn.loc[nombres_meses, nombres_meses]
                display(dunn)
    
    display(HTML("<hr style='border: 2px solid #696969; margin: 20px 0;'>"))

Estadísticas Descriptivas - Código C29


mes_nombre,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
count,8,8,8,8,8,8,8,8,8,8,8,8
mean,3186,2918,3424,3385,2862,2888,3031,2888,2718,2706,2631,2958
std,1117,925,1015,913,1027,829,957,1050,770,872,991,1116
min,1350,1308,1846,1701,1125,1506,1367,1173,1194,1153,764,1062
25%,2643,2683,3020,3184,2393,2326,2788,2522,2391,2239,2205,2597
50%,3524,3055,3360,3366,3286,2974,3014,2980,2868,2762,2868,3295
75%,3976,3236,3723,3686,3450,3344,3433,3192,3153,3278,3240,3562
max,4450,4320,5402,4959,3967,4003,4692,4745,3717,3840,3705,4383


Prueba de normalidad (Shapiro-Wilk) por mes:
Enero: p-valor = 0.252333 -> Normal
Febrero: p-valor = 0.618653 -> Normal
Marzo: p-valor = 0.606818 -> Normal
Abril: p-valor = 0.585704 -> Normal
Mayo: p-valor = 0.142705 -> Normal
Junio: p-valor = 0.838420 -> Normal
Julio: p-valor = 0.737475 -> Normal
Agosto: p-valor = 0.790487 -> Normal
Septiembre: p-valor = 0.448000 -> Normal
Octubre: p-valor = 0.753975 -> Normal
Noviembre: p-valor = 0.442478 -> Normal
Diciembre: p-valor = 0.214524 -> Normal
¿Se cumple normalidad en todos los meses? Sí

Levene p-valor: 0.999755
¿Varianzas iguales? Sí

Se cumplen los supuestos → Se aplica ANOVA
Estadístico F: 0.5440
P-valor: 0.867679
¿Hay diferencias entre meses? No


Estadísticas Descriptivas - Código C02


mes_nombre,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
count,7,8,8,7,7,7,7,7,8,8,8,8
mean,2102,2195,2283,2233,2461,2358,2535,2223,2126,2572,2252,2021
std,887,774,687,860,1029,931,683,869,982,688,630,577
min,1000,888,1265,1118,900,615,1471,1095,684,1520,1385,1094
25%,1337,1857,2026,1482,1988,2132,2126,1552,1351,2198,1923,1758
50%,2411,2360,2380,2418,2439,2442,2731,2370,2306,2710,2162,2130
75%,2627,2560,2628,3033,2915,2768,3012,2705,2552,2976,2518,2311
max,3375,3329,3247,3065,4084,3647,3267,3581,3822,3575,3310,2879


...

mes_nombre,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
count,8,8,8,8,8,8,8,8,8,8,8,8
mean,537,509,586,698,661,603,488,478,541,564,520,602
std,179,194,180,288,352,293,207,134,195,206,322,214
min,334,308,303,258,254,187,285,293,313,308,81,277
25%,361,375,461,508,342,322,333,396,420,393,275,466
50%,575,420,592,788,656,670,404,474,490,567,478,564
75%,620,644,714,863,959,824,621,519,697,741,758,763
max,846,812,847,1080,1084,984,847,734,812,808,956,906


Prueba de normalidad (Shapiro-Wilk) por mes:
Enero: p-valor = 0.315076 -> Normal
Febrero: p-valor = 0.112285 -> Normal
Marzo: p-valor = 0.768033 -> Normal
Abril: p-valor = 0.531865 -> Normal
Mayo: p-valor = 0.052820 -> Normal
Junio: p-valor = 0.442570 -> Normal
Julio: p-valor = 0.158862 -> Normal
Agosto: p-valor = 0.836801 -> Normal
Septiembre: p-valor = 0.311623 -> Normal
Octubre: p-valor = 0.196932 -> Normal
Noviembre: p-valor = 0.546964 -> Normal
Diciembre: p-valor = 0.826780 -> Normal
¿Se cumple normalidad en todos los meses? Sí

Levene p-valor: 0.066028
¿Varianzas iguales? Sí

Se cumplen los supuestos → Se aplica ANOVA
Estadístico F: 0.6363
P-valor: 0.792909
¿Hay diferencias entre meses? No


Estadísticas Descriptivas - Código C32


mes_nombre,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
count,8,8,8,8,8,8,8,8,8,8,8,8
mean,44,40,38,37,31,33,31,25,34,27,34,47
std,34,26,29,23,19,20,19,14,16,13,20,41
min,15,18,18,13,7,11,9,7,7,6,3,6
25%,28,24,22,22,17,20,22,15,24,16,23,27
50%,32,33,30,28,30,28,26,21,39,30,38,34
75%,46,44,39,49,45,42,34,32,44,35,48,53
max,124,96,107,79,55,72,68,47,54,44,61,140


Prueba de normalidad (Shapiro-Wilk) por mes:
Enero: p-valor = 0.002947 -> No normal
Febrero: p-valor = 0.041829 -> No normal
Marzo: p-valor = 0.001134 -> No normal
Abril: p-valor = 0.213659 -> Normal
Mayo: p-valor = 0.359725 -> Normal
Junio: p-valor = 0.493389 -> Normal
Julio: p-valor = 0.124091 -> Normal
Agosto: p-valor = 0.290620 -> Normal
Septiembre: p-valor = 0.508772 -> Normal
Octubre: p-valor = 0.639795 -> Normal
Noviembre: p-valor = 0.823669 -> Normal
Diciembre: p-valor = 0.030597 -> No normal
¿Se cumple normalidad en todos los meses? No

Levene p-valor: 0.962166
¿Varianzas iguales? Sí

No se cumplen los supuestos → Se aplica Kruskal-Wallis
Estadístico H: 5.0994
P-valor: 0.926260
¿Hay diferencias entre meses? No


**Código C29 - Exceso de velocidad (284,755 infracciones)**

- El exceso de velocidad tiene sus picos en el primer trimestre del año (marzo y abril), mientras que los meses de septiembre a noviembre presentan los volúmenes más bajos. La variabilidad es alta en enero y diciembre, posiblemente por efectos de temporada de vacaciones y fin de año.

- Todos los meses cumplen con normalidad (p-valores > 0.05, el más bajo fue mayo con 0.1427) y las varianzas son homogéneas (p=0.9998). El ANOVA arroja un estadístico F de 0.5440 con un p-valor de 0.8677, muy por encima del umbral de 0.05. **No existen diferencias estadísticamente significativas entre las medias de los meses** para el exceso de velocidad. Las fluctuaciones observadas en los promedios mensuales (marzo con mayor promedio, noviembre con menor) no son consistentes a lo largo de los años.

---

**Código C02 - Estacionamiento prohibido (204,973 infracciones)**

- Las cámaras móviles (responsables del C02) presentan menos años de registro (count=7) en varios meses, confirmando la interrupción de operación durante el confinamiento de 2020.

- El estacionamiento prohibido alcanza sus máximos en octubre, julio y mayo, mientras que diciembre y enero son los meses de menor actividad. La variabilidad es especialmente alta en mayo (influenciada por el pico de 2019 y el valle de 2020).

- Todos los meses cumplen con normalidad (p-valores > 0.05, el más bajo fue abril con 0.1002) y las varianzas son homogéneas (p=0.9620). El ANOVA arroja un estadístico F de 0.3436 con un p-valor de 0.9727. **No existen diferencias estadísticamente significativas entre las medias de los meses** para el estacionamiento prohibido.

---

**Código C03 - Bloqueo de calzada (65,230 infracciones)**

- El bloqueo de calzada tiene una marcada estacionalidad, con promedios mucho más altos entre mayo y octubre (segundo semestre), mientras que enero es el mes de menor actividad. La variabilidad también es mucho mayor en los meses de alta actividad.

- Todos los meses cumplen con normalidad (p-valores > 0.05, el más bajo fue agosto con 0.0563, aún por encima del umbral) y las varianzas son homogéneas (p=0.3381). El ANOVA arroja un estadístico F de 0.6916 con un p-valor de 0.7427. **No existen diferencias estadísticamente significativas entre las medias de los meses** para el bloqueo de calzada.

---

**Código D04 - Paso de semáforo en rojo o señal de pare (54,303 infracciones)**

- El paso de semáforo en rojo tiene sus picos en abril y mayo, mientras que julio y agosto presentan los promedios más bajos. La variabilidad es notablemente alta en noviembre, influenciada por valores atípicos.

- Todos los meses cumplen con normalidad (p-valores > 0.05, el más bajo fue mayo con 0.0528, apenas por encima del umbral) y las varianzas son homogéneas (p=0.0660, ligeramente por encima de 0.05). El ANOVA arroja un estadístico F de 0.636

---

**Código C32 - No respetar paso de peatones (3,369 infracciones)**

- El no respeto al paso de peatones es la infracción de menor volumen. Presenta una clara estacionalidad con picos en diciembre y enero (temporada de fin de año y vacaciones), mientras que agosto y octubre son los meses de menor actividad.

- Este código no cumple con el supuesto de normalidad. Enero (p=0.0029), febrero (p=0.0418), marzo (p=0.0011) y diciembre (p=0.0306) presentan p-valores inferiores a 0.05, indicando que estos meses no siguen una distribución normal. Sin embargo, las varianzas son homogéneas (p=0.9622). Dado que no se cumple la normalidad, se aplica la prueba no paramétrica de Kruskal-Wallis, que arroja un estadístico H de 5.0994 con un p-valor de 0.9263. **Tampoco existen diferencias estadísticamente significativas entre los meses** para el paso de peatones.

---

**Hallazgos:**

- **Mayo como mes crítico para múltiples infracciones**: Es el mes con mayor variabilidad para C02 y D04, influenciado por los picos de 2019 (cierres viales) y los valles de 2020 (pandemia).

- **Diciembre concentra la mayor variabilidad en C29 y C32**: La temporada de fin de año genera comportamientos extremos tanto en exceso de velocidad como en respeto a peatones.

- Ninguno de los cinco códigos analizados (C29, C02, C03, D04, C32) presenta diferencias estadísticamente significativas en el volumen de comparendos a lo largo de los meses del año. Esto indica que, independientemente del tipo de infracción, los patrones mensuales se mantienen constantes sin que ningún mes destaque consistentemente por encima de los demás.

- Estos resultados complementan el análisis de descomposición STL, donde se observó que algunos códigos presentaban amplitudes estacionales considerables (C29: 1,322; C03: 1,508; D04: 598). Sin embargo, al someter las diferencias mensuales a una prueba estadística (ANOVA o Kruskal-Wallis), se confirma que las oscilaciones observadas entre meses no son lo suficientemente grandes ni consistentes a lo largo de los años como para ser consideradas estadísticamente significativas.

- Es el único código donde algunos meses (enero, febrero, marzo y diciembre) no siguen una distribución normal. Esto puede deberse al bajo volumen de infracciones de esta categoría (solo 3,369 comparendos en total), lo que genera distribuciones más erráticas y menos estables que en otros códigos con mayor volumen.

---

**Notas:**

- Las campañas de control de velocidad deben intensificarse en marzo y abril.
- El control de estacionamiento prohibido debe priorizarse en octubre, julio y mayo.
- El bloqueo de calzada requiere especial atención en el segundo semestre del año.
- El respeto al paso de peatones debe reforzarse en diciembre y enero, temporada de mayor afluencia peatonal.

## Análisis de la Clase del Vehículo Infractor Según el Servicio del Vehículo Infractor

Se construye una tabla pivote que cruza la clase del vehículo infractor (con agrupación de vehículos pesados y categorías minoritarias en "OTROS") con el servicio del vehículo (PARTICULAR, PÚBLICO y OTROS, que incluye OFICIAL y DIPLOMÁTICO), utilizando la suma ponderada de `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones en cada combinación. Para cada clase de vehículo, se calcula el porcentaje que representa cada servicio sobre el total de sus comparendos, permitiendo identificar la composición del parque automotor infractor. Los resultados se presentan mediante un mapa de calor interactivo, donde las clases de vehículo se ordenan de mayor a menor según su total de comparendos, y cada celda muestra tanto el valor absoluto como el porcentaje correspondiente. Esta visualización permite identificar qué servicios predominan dentro de cada clase de vehículo, revelando patrones de distribución del parque automotor que comete infracciones detectadas por el sistema de cámaras.

In [24]:
tabla_heatmap = df_comparendos_electronicos_copy.pivot_table(
    values='CANTIDAD_INFRACCIONES',
    index='CLASE_AGRUPADA',
    columns='SERVICIO_AGRUPADO',
    aggfunc='sum',
    fill_value=0
)

orden = tabla_heatmap.sum(axis=1).sort_values(ascending=True).index
tabla_heatmap = tabla_heatmap.loc[orden]

total_por_fila = tabla_heatmap.sum(axis=1)
porcentajes = tabla_heatmap.div(total_por_fila, axis=0) * 100
porcentajes = porcentajes.round(2)
texto_combinado = tabla_heatmap.applymap(lambda x: f'{int(x):,}') + '<br>(' + porcentajes.applymap(lambda x: f'{x:.2f}%') + ')'

fig = go.Figure(data=go.Heatmap(
    z=tabla_heatmap.values,
    x=tabla_heatmap.columns,
    y=tabla_heatmap.index,
    colorscale='Blues',
    text=texto_combinado.values,
    texttemplate='%{text}',
    hovertemplate='Clase: %{y}<br>Servicio: %{x}<br>Comparendos: %{z:,.0f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Distribución de Comparendos por Clase del Vehículo y Servicio del Vehículo',
        x=0.5,
        xanchor='center',
        font=dict(size=16, weight='bold')
    ),
    xaxis_title=dict(
        text='Servicio del Vehículo',
        font=dict(weight='bold')
    ),
    yaxis_title=dict(
        text='Clase del Vehículo',
        font=dict(weight='bold')
    ),
    template='plotly_white',
    width=1055, 
    height=500
)

fig.show()

- **AUTOMÓVIL (321,953 infracciones)**: La abrumadora mayoría de los automóviles infractores son de servicio particular (82.80%), mientras que los de servicio público representan el 17.14%.

- **CAMIONETA (129,827 infracciones)**: Similar a los automóviles, las camionetas infractoras son predominantemente particulares (89.27%), con una presencia menor de servicio público (10.42%).

- **MOTOCICLETA (102,708 infracciones)**: Las motocicletas tienen una concentración casi absoluta en el servicio particular (99.93%), con una presencia mínima de servicio OTROS. No se registran infracciones de motocicletas públicas.

- **CAMPERO (39,955 infracciones)**: Al igual que las clases anteriores, los camperos infractores son mayoritariamente particulares (98.82%), con muy baja representación de servicio público y OTROS.

- **VEHÍCULOS PESADOS (19,736 infracciones)** Los vehículos pesados (camión, bus, microbús, buseta, volqueta, tractocamión) son mayoritariamente de servicio público (93.75%), a diferencia de las clases anteriores donde predominaba el servicio particular. Los particulares representan solo el 6.14%. Lo más probable es que esto se deba a las categorias microbús, bús, buseta que son mayoritariamente para el servicio al público.

- **OTROS (181 infracciones)**: Esta categoría (que incluye motocarro, cuatrimoto, no reportado, sin clase) tiene una mayor presencia de servicio particular (95.58%), seguido de público (4.42%). No se registran vehículos de servicio diplomático u oficial.

---

**Hallazgos:**

- Segmentación clara por servicio**: Los vehículos particulares dominan en automóviles, camionetas, motocicletas y camperos (entre el 82% y el 99%). Los vehículos de servicio público dominan en la categoría de vehículos pesados (93.75%).

- **Las motocicletas son casi exclusivamente particulares**: Con un 99.93%, las motocicletas infractoras son abrumadoramente de servicio particular, lo que refleja que el uso de motocicletas como servicio OTROS es marginal en el contexto de los comparendos electrónicos.

- **Los vehículos pesados son predominantemente públicos**: Este hallazgo es consistente con la naturaleza de estos vehículos (buses, busetas, camiones, etc.), que en su mayoría prestan servicios de transporte público o de carga.


---

**Notas**
- Las campañas de control y educación vial para automóviles, camionetas, motocicletas y camperos deben enfocarse principalmente en conductores particulares.
- Para vehículos pesados, las estrategias deben dirigirse a empresas de transporte público y de carga.
- La alta concentración de motocicletas en el servicio particular sugiere que las políticas de control de velocidad y semáforos para este tipo de vehículo deben enfocarse en conductores particulares.

## Distribución Geográfica de Comparendos Electrónicos por Servicio del Vehículo Infractor

Se examina, para cada categoría de servicio del vehículo (PARTICULAR, PÚBLICO y OTROS). Para cada servicio, se presentan tablas de frecuencias que muestran las ubicaciones con sus respectivos conteos, porcentajes dentro del servicio, y las fechas de primer y último registro, permitiendo identificar los puntos críticos de detección para cada tipo de servicio vehicular, evaluar la antigüedad y continuidad operativa de las cámaras para cada grupo, y comprender si existen patrones geográficos diferenciados en la comisión de infracciones según el servicio del vehículo infractor.

In [25]:
servicios = df_comparendos_electronicos_copy['SERVICIO_AGRUPADO'].unique()

for servicio in servicios:
    df_servicio = df_comparendos_electronicos_copy[df_comparendos_electronicos_copy['SERVICIO_AGRUPADO'] == servicio]
    
    total_servicio = df_servicio['CANTIDAD_INFRACCIONES'].sum()
    
    df_ubicaciones = df_servicio.groupby('Camara_y_direccion')['CANTIDAD_INFRACCIONES'].sum().sort_values(ascending=False).reset_index()
    df_ubicaciones.columns = ['Cámara y Dirección', 'Comparendos']
    
    df_ubicaciones['Porcentaje'] = (df_ubicaciones['Comparendos'] / total_servicio * 100).round(2)
    
    fechas = df_servicio.groupby('Camara_y_direccion')['fecha_comparendo'].agg(['min', 'max']).reset_index()
    fechas.columns = ['Cámara y Dirección', 'Primer Registro', 'Último Registro']
    fechas['Primer Registro'] = pd.to_datetime(fechas['Primer Registro']).dt.date
    fechas['Último Registro'] = pd.to_datetime(fechas['Último Registro']).dt.date
    
    df_ubicaciones = df_ubicaciones.merge(fechas, on='Cámara y Dirección')
    
    print(f"Distribución de Comparendos Electrónicos - Servicio: {servicio}")
    pd.set_option('display.max_columns', None)  
    pd.set_option('display.width', None)        
    pd.set_option('display.max_colwidth', None) 
    display(df_ubicaciones.T)
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')
    pd.reset_option('display.max_colwidth')
    
    display(HTML("<hr style='border: 2px solid #696969; margin: 20px 0;'>"))

Distribución de Comparendos Electrónicos - Servicio: PARTICULAR


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51
Cámara y Dirección,Carro mal parqueo Sur,Carro mal parqueo Norte,VIA 11 CON CARRERA 8,Carro mal parqueo Sur - Norte,CARRERA 51B CON CALLE 79,CARRERA 53 ENTRE CALLE 104 Y 106,CALLE 30 CON CARRERA 6B,CALLE 45 CON CARRERA 1,CALLE 30 CON CARRERA 8,CALLE 82 CON CARRERA 51B,CARRERA 51B CON CALLE 103,Carro mal parqueo alto norte,AVENIDA CIRCUNVALAR CON CARRERA 9G,VIA 40 CON CALLE 73,CALLE 82 CON CARRERA 56,CARRERA 6 CON CALLE 72,Carro mal parqueo alto sur,CALLE 45 CON CARRERA 20,CARRERA 15 CON CALLE 21,CALLE 84 ENTRE CARRERA 59 Y 59B,CALLE 45 CON CARRERA 21,CALLE 94 CON CARRERA 58,CARRERA 46 CON CALLE 100,CALLE 45B CON CARRERA 14,CALLE 72 CON CARRERA 44,CALLE 56 CON CARRERA 14,CALLE 19 CON CARRERA 4C,CALLE 76 CON CARRERA 38C-100,CALLE 87 CON CARRERA 21,CALLE 19 CON CARRERA 3D,CALLE 53 CON CARRERA 45,CARRERA 53 CON CALLE 86,CALLE 70 CON CARRERA 46,CALLE 98 CON CARRERA 56,AVENIDA CIRCUNVALAR CON CARRERA 31,CALLE 61 CON CARRERA 35,CALLE 45 CON CARRERA 21 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 21 SENTIDO NORTE-SUR,CALLE 34 CON CARRERA 45,CARRERA 46 CON CALLE 34 SENTIDO SUR-NORTE,CALLE 70 CON CARRERA 46 SENTIDO SUR-NORTE,CARRERA 27 CON CALLE 82C,CALLE 70 CON CARRERA 46 SENTIDO NORTE-SUR,AVENIDA CIRCUNVALAR (CALLE 110) ENTRE CARRERA 9G Y 12 SENTIDO SUR A NORTE,AVENIDA CIRCUNVALAR (CALLE 110) ENTRE CARRERA 9G Y 12 SENTIDO NORTE A SUR,CALLE 45 CON CARRERA 8 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 33 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 43 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 43 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 1E SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 1E SENTIDO NORTE-SUR,CARRERA 46 CON CALLE 34 SENTIDO NORTE-SUR
Comparendos,66280,60719,44596,34468,30878,20576,19604,18744,17352,16453,16225,14946,13217,11411,10730,10535,9946,9150,8276,7613,7538,7250,7029,6590,6479,6147,5670,5554,5320,5215,5007,4167,3536,1956,1822,1008,947,924,616,355,350,347,202,102,57,40,19,9,5,3,2,1
Porcentaje,12.6,11.54,8.48,6.55,5.87,3.91,3.73,3.56,3.3,3.13,3.08,2.84,2.51,2.17,2.04,2.0,1.89,1.74,1.57,1.45,1.43,1.38,1.34,1.25,1.23,1.17,1.08,1.06,1.01,0.99,0.95,0.79,0.67,0.37,0.35,0.19,0.18,0.18,0.12,0.07,0.07,0.07,0.04,0.02,0.01,0.01,0.0,0.0,0.0,0.0,0.0,0.0
Primer Registro,2018-01-02,2018-01-02,2018-01-01,2018-09-29,2018-01-01,2018-10-16,2018-01-01,2018-01-01,2018-01-01,2018-01-10,2018-01-02,2023-06-28,2018-01-01,2018-01-02,2018-01-09,2018-01-01,2023-08-15,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-02,2018-01-10,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2019-02-21,2018-01-01,2018-01-09,2018-10-13,2018-01-01,2018-01-30,2024-11-24,2024-11-22,2018-01-03,2025-10-23,2025-10-21,2018-01-26,2025-10-21,2024-11-29,2024-12-06,2025-10-26,2025-10-23,2025-10-25,2025-10-24,2025-10-22,2025-10-21,2025-10-21
Último Registro,2025-12-31,2025-12-31,2025-12-30,2025-12-30,2025-12-28,2025-12-31,2025-12-30,2025-12-30,2025-12-31,2025-12-30,2025-12-12,2025-12-31,2024-11-28,2025-12-20,2025-12-30,2025-12-29,2025-12-31,2024-11-19,2025-12-19,2025-12-30,2025-10-05,2025-08-31,2025-12-31,2025-12-31,2025-11-07,2021-05-03,2025-12-30,2025-12-31,2021-05-24,2025-12-28,2025-12-30,2022-07-19,2025-12-29,2024-08-05,2019-07-23,2025-07-03,2025-12-28,2025-12-31,2025-12-31,2025-11-17,2025-12-31,2020-03-02,2025-12-11,2025-10-08,2025-12-29,2025-12-29,2025-12-28,2025-11-05,2025-11-02,2025-10-24,2025-10-23,2025-10-21


Distribución de Comparendos Electrónicos - Servicio: PUBLICO


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50
Cámara y Dirección,VIA 11 CON CARRERA 8,Carro mal parqueo Sur,CARRERA 6 CON CALLE 72,CALLE 30 CON CARRERA 6B,Carro mal parqueo Norte,CALLE 30 CON CARRERA 8,CALLE 45 CON CARRERA 1,CALLE 45B CON CARRERA 14,CALLE 45 CON CARRERA 20,Carro mal parqueo Sur - Norte,CALLE 82 CON CARRERA 51B,AVENIDA CIRCUNVALAR CON CARRERA 9G,CALLE 34 CON CARRERA 45,CALLE 45 CON CARRERA 21,CARRERA 51B CON CALLE 79,CARRERA 53 ENTRE CALLE 104 Y 106,CALLE 72 CON CARRERA 44,Carro mal parqueo alto norte,CALLE 53 CON CARRERA 45,CARRERA 15 CON CALLE 21,VIA 40 CON CALLE 73,CALLE 19 CON CARRERA 4C,CALLE 82 CON CARRERA 56,CALLE 56 CON CARRERA 14,Carro mal parqueo alto sur,CALLE 19 CON CARRERA 3D,CALLE 87 CON CARRERA 21,CARRERA 51B CON CALLE 103,CALLE 84 ENTRE CARRERA 59 Y 59B,CALLE 70 CON CARRERA 46,CALLE 76 CON CARRERA 38C-100,CARRERA 46 CON CALLE 100,CALLE 94 CON CARRERA 58,AVENIDA CIRCUNVALAR CON CARRERA 31,CARRERA 46 CON CALLE 34 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 21 SENTIDO SUR-NORTE,CARRERA 53 CON CALLE 86,CALLE 70 CON CARRERA 46 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 21 SENTIDO NORTE-SUR,CALLE 98 CON CARRERA 56,CALLE 61 CON CARRERA 35,CALLE 70 CON CARRERA 46 SENTIDO NORTE-SUR,CARRERA 27 CON CALLE 82C,CALLE 45 CON CARRERA 8 SENTIDO SUR-NORTE,AVENIDA CIRCUNVALAR (CALLE 110) ENTRE CARRERA 9G Y 12 SENTIDO NORTE A SUR,CALLE 45 CON CARRERA 33 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 43 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 1E SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 43 SENTIDO NORTE-SUR,CARRERA 46 CON CALLE 34 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 1E SENTIDO NORTE-SUR
Comparendos,7386,7073,5745,4622,4483,4449,4044,3968,3572,3477,3162,2948,2516,2483,2390,2386,2363,2123,1763,1625,1548,1302,1294,1233,1069,994,993,879,854,705,678,666,603,396,367,227,223,208,198,170,167,96,43,22,9,9,9,5,4,3,1
Porcentaje,8.44,8.08,6.56,5.28,5.12,5.08,4.62,4.53,4.08,3.97,3.61,3.37,2.87,2.84,2.73,2.73,2.7,2.42,2.01,1.86,1.77,1.49,1.48,1.41,1.22,1.14,1.13,1.0,0.98,0.81,0.77,0.76,0.69,0.45,0.42,0.26,0.25,0.24,0.23,0.19,0.19,0.11,0.05,0.03,0.01,0.01,0.01,0.01,0.0,0.0,0.0
Primer Registro,2018-01-01,2018-01-02,2018-01-01,2018-01-01,2018-01-02,2018-01-01,2018-01-01,2018-01-18,2018-01-01,2018-09-29,2018-01-11,2018-01-01,2018-01-03,2018-01-01,2018-01-01,2018-10-24,2018-01-18,2023-06-28,2018-01-04,2018-01-02,2018-01-04,2018-01-01,2018-01-16,2018-01-01,2023-08-15,2018-01-01,2018-01-01,2018-01-06,2018-01-17,2018-01-12,2018-01-03,2018-01-08,2018-01-01,2018-01-02,2025-10-23,2024-11-24,2018-01-25,2025-10-21,2024-11-24,2018-10-21,2018-02-01,2025-10-24,2018-01-02,2025-10-23,2024-12-21,2025-11-01,2025-10-25,2025-10-21,2025-10-29,2025-10-21,2025-10-21
Último Registro,2025-12-26,2025-12-31,2025-12-16,2025-12-29,2025-12-29,2025-12-30,2025-12-30,2025-12-23,2024-11-17,2025-12-30,2025-12-30,2024-11-01,2025-12-30,2025-10-05,2025-12-25,2025-12-30,2025-11-07,2025-12-31,2025-12-29,2025-12-19,2025-12-08,2025-12-22,2025-12-22,2021-05-01,2025-12-30,2025-12-25,2021-05-23,2025-08-28,2025-12-25,2025-12-06,2025-12-04,2025-12-21,2025-08-24,2019-07-17,2025-11-17,2025-12-23,2022-06-19,2025-12-31,2025-12-31,2024-07-31,2025-06-25,2025-12-08,2020-02-20,2025-12-28,2025-10-19,2025-12-29,2025-11-01,2025-10-22,2025-11-01,2025-10-25,2025-10-21


Distribución de Comparendos Electrónicos - Servicio: OTROS


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42
Cámara y Dirección,Carro mal parqueo Sur,Carro mal parqueo Norte,Carro mal parqueo Sur - Norte,VIA 11 CON CARRERA 8,CALLE 30 CON CARRERA 6B,CALLE 30 CON CARRERA 8,Carro mal parqueo alto sur,CARRERA 51B CON CALLE 79,VIA 40 CON CALLE 73,Carro mal parqueo alto norte,AVENIDA CIRCUNVALAR CON CARRERA 9G,CALLE 45 CON CARRERA 20,CARRERA 53 ENTRE CALLE 104 Y 106,CALLE 45 CON CARRERA 21,CALLE 84 ENTRE CARRERA 59 Y 59B,CARRERA 51B CON CALLE 103,CALLE 45 CON CARRERA 1,CARRERA 6 CON CALLE 72,CALLE 82 CON CARRERA 51B,CALLE 53 CON CARRERA 45,CARRERA 46 CON CALLE 100,CALLE 94 CON CARRERA 58,CALLE 45B CON CARRERA 14,CARRERA 15 CON CALLE 21,CALLE 19 CON CARRERA 4C,CALLE 82 CON CARRERA 56,CALLE 45 CON CARRERA 21 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 8 SENTIDO SUR-NORTE,CALLE 72 CON CARRERA 44,AVENIDA CIRCUNVALAR CON CARRERA 31,CALLE 19 CON CARRERA 3D,CALLE 87 CON CARRERA 21,CALLE 56 CON CARRERA 14,CALLE 45 CON CARRERA 33 SENTIDO NORTE-SUR,CARRERA 53 CON CALLE 86,CALLE 70 CON CARRERA 46 SENTIDO SUR-NORTE,CALLE 76 CON CARRERA 38C-100,CARRERA 46 CON CALLE 34 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 21 SENTIDO NORTE-SUR,CALLE 98 CON CARRERA 56,CALLE 70 CON CARRERA 46 SENTIDO NORTE-SUR,CALLE 34 CON CARRERA 45,CALLE 61 CON CARRERA 35
Comparendos,191,104,48,47,35,31,25,25,24,21,21,18,17,15,12,12,12,12,11,11,10,10,10,9,8,8,8,7,7,7,6,6,5,4,4,4,4,3,3,2,2,1,1
Porcentaje,23.26,12.67,5.85,5.72,4.26,3.78,3.05,3.05,2.92,2.56,2.56,2.19,2.07,1.83,1.46,1.46,1.46,1.46,1.34,1.34,1.22,1.22,1.22,1.1,0.97,0.97,0.97,0.85,0.85,0.85,0.73,0.73,0.61,0.49,0.49,0.49,0.49,0.37,0.37,0.24,0.24,0.12,0.12
Primer Registro,2018-01-12,2018-01-15,2018-10-03,2018-01-01,2018-01-22,2018-10-12,2023-08-17,2018-02-08,2018-01-06,2023-07-15,2018-05-18,2018-01-11,2019-02-17,2018-03-11,2018-05-15,2018-04-08,2018-01-01,2018-02-26,2020-01-08,2019-06-27,2018-01-06,2018-02-18,2019-03-14,2019-04-10,2018-01-18,2019-07-06,2025-01-03,2025-12-06,2023-07-24,2018-02-13,2018-03-21,2018-05-06,2018-03-13,2025-11-10,2018-04-19,2025-11-14,2018-05-25,2025-11-09,2025-03-19,2019-01-28,2025-11-27,2024-06-19,2024-12-30
Último Registro,2025-10-24,2025-10-28,2025-09-22,2025-08-12,2025-11-19,2025-11-30,2025-10-31,2025-09-03,2024-09-05,2025-12-09,2024-09-16,2024-09-30,2025-05-22,2024-11-09,2024-09-03,2025-05-11,2024-03-31,2024-08-02,2025-12-04,2024-09-27,2024-10-11,2025-07-30,2025-08-28,2024-09-04,2024-07-04,2025-09-11,2025-10-09,2025-12-27,2025-10-17,2019-01-28,2025-06-07,2021-02-28,2019-11-21,2025-12-23,2021-03-17,2025-12-05,2019-04-02,2025-11-13,2025-12-30,2019-02-04,2025-12-07,2024-06-19,2024-12-30


**Servicio PARTICULAR (525,986 infracciones)**

- Las dos primeras ubicaciones son cámaras de mal parqueo (Sur y Norte), que en conjunto representan el **24.14%** de todas las infracciones de vehículos particulares.
- Los vehículos particulares aparecen en **52 ubicaciones** diferentes, lo que refleja una alta dispersión geográfica de sus infracciones.
- Operación continua desde 2018 hasta 2025 en la mayoría de ubicaciones.
- Se observan registros de particulares en cámaras Carril Bus (CARRERA 46 CON CALLE 34, CALLE 70 CON CARRERA 46, etc.), aunque con volúmenes muy bajos (menos de 400 comparendos en cada una).

**Servicio PÚBLICO (87,553 infracciones)**


- La ubicación "VIA 11 CON CARRERA 8" es la más activa para vehículos públicos (8.44%), mientras que para particulares ocupaba el tercer lugar.
- Aunque las cámaras de mal parqueo también aparecen en el top, su participación relativa es menor que en particulares (Sur: 8.08% vs 12.60% en particulares).
- Los vehículos públicos aparecen en **51 ubicaciones**, una menos que los particulares.
- También se registran infracciones de vehículos públicos en cámaras Carril Bus, con volúmenes bajos (Por ejemplo: CARRERA 46 CON CALLE 34 SENTIDO SUR-NORTE: 367 comparendos, 0.42%).

**Servicio OTROS (oficial y diplomático) (821 infracciones)**

- Solo 821 infracciones en todo el período, lo que representa el 0.13% del total.
- Las cámaras de mal parqueo (especialmente "Sur") concentran la mayor parte de las infracciones (Sur: 23.26%, Norte: 12.67%, Sur-Norte: 5.85%).
- Aparecen en **43 ubicaciones**, a pesar del bajo volumen, lo que indica que las infracciones de vehículos oficiales y diplomáticos ocurren en muchos puntos diferentes.
- Las fechas de registro son más irregulares que en los otros servicios, con operación discontinua en muchas ubicaciones.

---

**Hallazgos:**

- **Las cámaras de mal parqueo son críticas para todos los servicios**: Especialmente "Carro mal parqueo Sur" es la ubicación más importante para particulares (12.60%) y OTROS (23.26%), y la segunda para públicos (8.08%).

- **Los vehículos públicos tienen una distribución geográfica diferente**: Su ubicación más activa es "VIA 11 CON CARRERA 8" (8.44%), no una cámara de mal parqueo, lo que refleja que los vehículos públicos cometen proporcionalmente más infracciones de exceso de velocidad y semáforos en estos corredores.

- **Los vehículos OTROS tienen la mayor concentración geográfica**: Con el 41.78% de sus infracciones concentradas en solo 3 ubicaciones (todas de mal parqueo), los vehículos oficiales y diplomáticos tienen un patrón de infracciones más localizado.

- **Las cámaras Carril Bus tienen muy baja incidencia en todos los servicios**: Su implementación reciente (octubre 2025) explica los bajos volúmenes, pero se observa que tanto particulares como públicos ya están siendo detectados en estas ubicaciones.

---

**Notas:**

- Para reducir infracciones de vehículos particulares, se debe priorizar el mantenimiento de las cámaras de mal parqueo Sur y Norte, así como la VIA 11 CON CARRERA 8.
- Para vehículos públicos, además de las cámaras de mal parqueo, se debe prestar especial atención a corredores como VIA 11 CON CARRERA 8 y CARRERA 6 CON CALLE 72.
- Para vehículos oficiales y diplomáticos, las cámaras de mal parqueo Sur y Norte son las más estratégicas.

## Evolución Mensual de Comparendos Electrónicos por Servicio del Vehículo Infractor

Se examina la evolución mensual de los comparendos electrónicos desagregada por cada categoría de servicio del vehículo (PARTICULAR, PÚBLICO y OTROS), agrupando los registros por mes y sumando la columna `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones por cada período. Para cada servicio, se presentan tablas de frecuencias que muestran los meses con mayor y menor volumen, junto con sus porcentajes de participación dentro del total de ese servicio. Complementariamente, se generan gráficos de líneas interactivos que ilustran la tendencia temporal de cada categoría, incluyendo estadísticos como el promedio mensual, la desviación estándar, y los meses con valores máximo y mínimo, con un área sombreada que destaca el período crítico de la pandemia de COVID-19. Este análisis permite identificar patrones estacionales diferenciados por tipo de servicio vehicular, evaluar el impacto diferencial de eventos externos (como la pandemia o los cierres viales) en cada categoría, y comparar el comportamiento temporal de los conductores particulares frente a los de servicio público y oficial en la comisión de infracciones detectadas por el sistema de cámaras.

In [26]:
infracciones_por_mes_servicio = df_comparendos_electronicos_copy.groupby(['año_mes', 'SERVICIO_AGRUPADO'])['CANTIDAD_INFRACCIONES'].sum().reset_index()

for servicio in servicios:
    df_servicio_mensual = infracciones_por_mes_servicio[infracciones_por_mes_servicio['SERVICIO_AGRUPADO'] == servicio].copy()
    df_servicio_mensual = df_servicio_mensual[['año_mes', 'CANTIDAD_INFRACCIONES']].sort_values('año_mes').reset_index(drop=True)
    df_servicio_mensual.columns = ['Mes', 'Comparendos']
    
    total_servicio = df_servicio_mensual['Comparendos'].sum()
    

    df_servicio_mensual['Porcentaje'] = (df_servicio_mensual['Comparendos'] / total_servicio * 100).round(2)
    
    df_servicio_mensual = df_servicio_mensual.sort_values('Comparendos', ascending=False).reset_index(drop=True)
    
    print(f"Distribución de Comparendos Electrónicos por Mes - Servicio: {servicio}")
    pd.set_option('display.max_columns', None)  
    pd.set_option('display.width', None)        
    pd.set_option('display.max_colwidth', None) 
    display(df_servicio_mensual.T)
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')
    pd.reset_option('display.max_colwidth')
    
    display(HTML("<hr style='border: 2px solid #696969; margin: 20px 0;'>"))

for servicio in servicios:
    df_servicio = infracciones_por_mes_servicio[infracciones_por_mes_servicio['SERVICIO_AGRUPADO'] == servicio]
    
    total_servicio = df_servicio['CANTIDAD_INFRACCIONES'].sum()
    desviacion_servicio = df_servicio['CANTIDAD_INFRACCIONES'].std()
    promedio_servicio = df_servicio['CANTIDAD_INFRACCIONES'].mean()
    
    meses_servicio = sorted(df_servicio['año_mes'].unique())
    
    if len(meses_servicio) > 12:
        tick_positions = meses_servicio[::6]
    elif len(meses_servicio) > 6:
        tick_positions = meses_servicio[::3]
    else:
        tick_positions = meses_servicio
    
    max_valor = df_servicio['CANTIDAD_INFRACCIONES'].max()
    max_fila = df_servicio[df_servicio['CANTIDAD_INFRACCIONES'] == max_valor].iloc[0]
    max_mes = max_fila['año_mes']
    
    min_valor = df_servicio['CANTIDAD_INFRACCIONES'].min()
    min_fila = df_servicio[df_servicio['CANTIDAD_INFRACCIONES'] == min_valor].iloc[0]
    min_mes = min_fila['año_mes']
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=df_servicio['año_mes'],
        y=df_servicio['CANTIDAD_INFRACCIONES'],
        mode='lines+markers',
        name=servicio,
        line=dict(color='cornflowerblue', width=2),
        marker=dict(size=4, color='cornflowerblue'),
        hovertemplate=f'Servicio: {servicio}<br>Mes: %{{x}}<br>Infracciones: %{{y:,.0f}}<extra></extra>',
        showlegend=False
    ))
    
    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='none',
        name=f'Total: {int(total_servicio):,}',
        showlegend=True
    ))
    
    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='none',
        name=f'Std: {int(desviacion_servicio):,}',
        showlegend=True
    ))
    
    x_min = df_servicio['año_mes'].iloc[0]
    x_max = df_servicio['año_mes'].iloc[-1]
    fig.add_trace(go.Scatter(
        x=[x_min, x_max],
        y=[promedio_servicio, promedio_servicio],
        mode='lines',
        name=f'Promedio: {int(promedio_servicio):,}',
        line=dict(color='red', width=1.4, dash='dot'),
        showlegend=True,
        hoverinfo='none'
    ))
    
    fig.add_trace(go.Scatter(
        x=[max_mes],
        y=[max_valor],
        mode='markers',
        name=f'Máximo: {max_mes} ({int(max_valor):,})',
        marker=dict(size=8, color='red', symbol='circle'),
        hoverinfo='none'
    ))
    
    fig.add_trace(go.Scatter(
        x=[min_mes],
        y=[min_valor],
        mode='markers',
        name=f'Mínimo: {min_mes} ({int(min_valor):,})',
        marker=dict(size=8, color='green', symbol='circle'),
        hoverinfo='none'
    ))
    
    fig.add_vrect(
        x0="2020-03", x1="2020-12",
        fillcolor="red", opacity=0.1,
        line_width=0,
        annotation_text="COVID-19", 
        annotation_position="top left",
        annotation=dict(font_size=12, font_color="red")
    )
    
    fig.update_layout(
        title=dict(
            text=f'Evolución Mensual de Comparendos - Servicio: {servicio}',
            x=0.5,
            xanchor='center',
            font=dict(size=16, weight='bold')
        ),
        xaxis_title=dict(
            text='Año-Mes',
            font=dict(weight='bold')
        ),
        yaxis_title=dict(
            text='Cantidad de Comparendos',
            font=dict(weight='bold')
        ),
        template='plotly_white',
        hovermode='x unified',
        legend=dict(
            x=1, 
            y=1, 
            xanchor='center',
            yanchor='top',
            bgcolor='rgba(255,255,255,0.5)',
            font=dict(size=12)
        ),
        width=1055,
        height=500
    )
    
    fig.update_xaxes(
        tickangle=-45,
        tickvals=tick_positions,
        ticktext=tick_positions
    )
    
    fig.show()


Distribución de Comparendos Electrónicos por Mes - Servicio: PARTICULAR


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95
Mes,2019-05,2019-06,2019-07,2019-04,2019-03,2024-05,2024-04,2023-09,2023-07,2019-08,2023-12,2023-08,2023-10,2024-03,2024-06,2018-01,2022-06,2024-07,2023-06,2019-02,2018-11,2023-05,2022-12,2023-11,2024-09,2024-01,2020-01,2022-07,2023-03,2022-05,2018-03,2024-10,2019-10,2018-05,2018-10,2024-02,2022-11,2020-02,2018-04,2019-12,2018-12,2021-02,2020-10,2019-09,2023-04,2022-10,2024-08,2019-11,2022-09,2018-06,2019-01,2018-02,2018-07,2021-03,2018-09,2022-04,2020-12,2018-08,2025-10,2022-08,2021-01,2020-03,2020-11,2025-04,2024-11,2023-02,2025-01,2024-12,2025-03,2022-03,2025-11,2025-09,2021-07,2022-01,2025-05,2025-02,2025-07,2021-04,2021-12,2020-09,2025-06,2025-12,2021-08,2021-09,2020-04,2021-10,2020-08,2021-05,2021-11,2022-02,2025-08,2021-06,2020-06,2020-07,2023-01,2020-05
Comparendos,8466,8277,8011,7964,7860,7685,7491,7410,7375,7316,7287,7241,7158,7060,7029,7015,6917,6737,6736,6691,6544,6530,6473,6461,6437,6426,6424,6412,6398,6366,6344,6285,6219,6168,6164,6104,6056,6025,6019,6018,6006,5993,5987,5975,5958,5823,5777,5734,5655,5602,5532,5525,5514,5502,5455,5447,5432,5404,5396,5148,5147,5037,4887,4853,4794,4735,4703,4669,4579,4489,4311,4278,4272,4239,4223,4215,4174,4143,4129,4069,3772,3701,3700,3690,3690,3644,3550,3538,3495,3329,2958,2810,2483,2355,1722,1139
Porcentaje,1.61,1.57,1.52,1.51,1.49,1.46,1.42,1.41,1.4,1.39,1.39,1.38,1.36,1.34,1.34,1.33,1.32,1.28,1.28,1.27,1.24,1.24,1.23,1.23,1.22,1.22,1.22,1.22,1.22,1.21,1.21,1.19,1.18,1.17,1.17,1.16,1.15,1.15,1.14,1.14,1.14,1.14,1.14,1.14,1.13,1.11,1.1,1.09,1.08,1.07,1.05,1.05,1.05,1.05,1.04,1.04,1.03,1.03,1.03,0.98,0.98,0.96,0.93,0.92,0.91,0.9,0.89,0.89,0.87,0.85,0.82,0.81,0.81,0.81,0.8,0.8,0.79,0.79,0.79,0.77,0.72,0.7,0.7,0.7,0.7,0.69,0.67,0.67,0.66,0.63,0.56,0.53,0.47,0.45,0.33,0.22


Distribución de Comparendos Electrónicos por Mes - Servicio: PUBLICO


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95
Mes,2022-12,2019-03,2022-11,2022-06,2019-04,2018-11,2022-04,2019-05,2023-07,2022-05,2019-07,2023-06,2018-10,2023-12,2019-08,2023-09,2022-07,2019-06,2019-02,2023-03,2023-04,2023-05,2024-05,2024-09,2023-08,2023-10,2022-09,2022-10,2018-01,2024-04,2024-07,2025-10,2024-06,2018-12,2024-03,2024-10,2025-11,2018-05,2019-01,2018-03,2022-08,2023-11,2020-02,2023-02,2019-09,2020-01,2018-02,2019-12,2019-10,2018-06,2020-10,2024-01,2024-02,2018-04,2022-03,2019-11,2018-07,2018-09,2020-03,2024-08,2020-09,2018-08,2020-08,2020-11,2024-11,2025-09,2021-04,2020-12,2020-04,2021-01,2021-02,2025-04,2021-03,2025-05,2025-07,2022-01,2024-12,2025-01,2025-12,2025-03,2021-07,2021-11,2021-12,2021-09,2025-06,2021-10,2021-08,2025-02,2020-06,2020-07,2025-08,2021-05,2022-02,2023-01,2021-06,2020-05
Comparendos,1960,1637,1506,1493,1400,1400,1382,1368,1368,1361,1301,1299,1289,1276,1267,1263,1242,1233,1210,1183,1175,1171,1134,1121,1121,1111,1109,1103,1099,1098,1071,1068,1065,1056,1046,1012,1009,992,986,977,976,965,963,962,956,916,904,897,896,896,895,851,849,841,836,833,809,809,789,788,768,754,739,726,725,725,725,714,703,689,684,657,656,627,614,612,612,583,572,571,560,557,550,545,533,520,515,502,491,448,447,434,410,383,362,247
Porcentaje,2.24,1.87,1.72,1.71,1.6,1.6,1.58,1.56,1.56,1.55,1.49,1.48,1.47,1.46,1.45,1.44,1.42,1.41,1.38,1.35,1.34,1.34,1.3,1.28,1.28,1.27,1.27,1.26,1.26,1.25,1.22,1.22,1.22,1.21,1.19,1.16,1.15,1.13,1.13,1.12,1.11,1.1,1.1,1.1,1.09,1.05,1.03,1.02,1.02,1.02,1.02,0.97,0.97,0.96,0.95,0.95,0.92,0.92,0.9,0.9,0.88,0.86,0.84,0.83,0.83,0.83,0.83,0.82,0.8,0.79,0.78,0.75,0.75,0.72,0.7,0.7,0.7,0.67,0.65,0.65,0.64,0.64,0.63,0.62,0.61,0.59,0.59,0.57,0.56,0.51,0.51,0.5,0.47,0.44,0.41,0.28


Distribución de Comparendos Electrónicos por Mes - Servicio: OTROS


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92
Mes,2019-02,2018-12,2018-01,2025-02,2023-08,2024-09,2023-03,2025-12,2024-04,2018-10,2018-07,2024-03,2019-08,2024-12,2018-05,2025-07,2018-03,2024-07,2024-10,2025-05,2023-05,2025-04,2024-02,2021-02,2018-04,2018-09,2019-04,2018-06,2023-09,2025-11,2023-07,2019-01,2025-01,2019-06,2018-08,2023-10,2023-11,2018-02,2019-07,2022-05,2019-05,2022-12,2019-09,2018-11,2019-03,2025-06,2025-09,2023-12,2023-06,2022-11,2020-10,2020-12,2019-10,2024-05,2021-03,2024-06,2025-10,2020-04,2020-01,2019-12,2024-08,2025-03,2022-04,2022-06,2021-07,2022-09,2021-09,2020-09,2023-02,2024-01,2023-04,2022-10,2020-03,2022-07,2021-12,2022-01,2019-11,2021-11,2021-06,2021-04,2022-08,2024-11,2025-08,2022-03,2022-02,2020-07,2020-06,2021-08,2020-11,2021-10,2021-05,2020-02,2023-01
Comparendos,27,20,20,20,17,17,16,16,16,15,15,14,13,13,13,13,12,12,12,12,12,11,11,11,11,11,11,11,11,11,11,10,10,10,10,9,9,9,9,9,9,9,8,8,8,8,8,8,8,8,8,8,8,8,8,7,7,7,7,7,7,7,7,7,7,6,6,6,6,6,5,5,5,5,5,5,4,4,4,4,4,4,4,3,3,3,2,2,2,2,2,1,1
Porcentaje,3.29,2.44,2.44,2.44,2.07,2.07,1.95,1.95,1.95,1.83,1.83,1.71,1.58,1.58,1.58,1.58,1.46,1.46,1.46,1.46,1.46,1.34,1.34,1.34,1.34,1.34,1.34,1.34,1.34,1.34,1.34,1.22,1.22,1.22,1.22,1.1,1.1,1.1,1.1,1.1,1.1,1.1,0.97,0.97,0.97,0.97,0.97,0.97,0.97,0.97,0.97,0.97,0.97,0.97,0.97,0.85,0.85,0.85,0.85,0.85,0.85,0.85,0.85,0.85,0.85,0.73,0.73,0.73,0.73,0.73,0.61,0.61,0.61,0.61,0.61,0.61,0.49,0.49,0.49,0.49,0.49,0.49,0.49,0.37,0.37,0.37,0.24,0.24,0.24,0.24,0.24,0.12,0.12


**Posible explicación para los puntos máximos de los servicios (particular y otros) - períodos 2019-5 y 2019-02:** 

El punto máximo registrado en las infracciones cometidas por vehículos de servicio particular durante ciertos meses (como mayo de 2019) puede explicarse por los múltiples cierres viales que se presentaron en Barranquilla durante esos períodos. Los cierres viales generan alteraciones significativas en los patrones normales de circulación, desviando el tránsito hacia vías alternas y creando condiciones de congestión que incrementan la probabilidad de comisión de infracciones. Los conductores particulares, al enfrentarse a desvíos inesperados, demoras y falta de señalización temporal, tienden a cometer con mayor frecuencia infracciones como estacionamiento en sitios prohibidos (al buscar alternativas de parqueo cerca de sus destinos), exceso de velocidad (al intentar recuperar tiempo perdido en rutas alternas), paso de semáforos en rojo o señal de pare (por impaciencia ante la congestión), bloqueo de calzadas o intersecciones y acceder a carriles exclusivos de Transmetro.

En cuanto al servicio OTROS (vehículos oficiales y diplomáticos), su punto máximo registrado en febrero de 2019 probablemente se debió a los múltiples cierres viales asociados a las celebraciones del Carnaval de Barranquilla y su planificación logística. Durante este mes, los desfiles, eventos masivos y la adecuación de las vías para las festividades generaron una gran cantidad de cierres y restricciones de movilidad en distintos puntos de la ciudad. Los vehículos oficiales y diplomáticos, al tener que cumplir con sus funciones durante esta temporada de alta congestión y desvíos, se vieron expuestos a condiciones de circulación atípicas que incrementaron la probabilidad de cometer infracciones.

**Fuentes**:
- https://barranquilla.gov.co/transito/por-trabajos-cierres-de-vias-a-partir-del-viernes-3-de-mayo
- https://barranquilla.gov.co/transito/por-ampliacion-de-la-calle-30-cierres-de-vias-a-partir-del-martes-14-de-mayo-de-2019
- https://barranquilla.gov.co/transito/por-proyecto-mall-plaza-cierres-de-vias-a-partir-del-martes-21-de-mayo-de-2019
- https://barranquilla.gov.co/transito/por-reparacion-de-redes-cierres-de-vias-a-partir-del-martes-28-de-mayo
- https://barranquilla.gov.co/transito/cierres-de-vias-para-el-sabado-23-y-domingo-24-de-febrero-de-2019
- https://barranquilla.gov.co/transito/cierres-de-vias-para-el-sabado-2-y-domingo-3-de-febrero-de-2019
- https://barranquilla.gov.co/transito/por-obra-de-canalizacion-del-arroyo-de-la-carrera-65-cierre-de-vias-a-partir-del-lunes-4-de-febrero
- https://barranquilla.gov.co/transito/por-izaje-de-equipos-cierres-de-vias-desde-la-noche-del-viernes-8de-febrero-de-2019
- https://barranquilla.gov.co/transito/por-garabato-del-country-cierre-de-vias-para-el-sabado-9-de-febrero-de-2019
- https://barranquilla.gov.co/transito/cierres-de-vias-para-el-viernes-15sabado-16-y-domingo-17-de-febrero-de-2019

---

**Posible explicación para el punto máximo del servicio público - período 2022-12**: 

El punto máximo registrado en las infracciones cometidas por vehículos de servicio público durante este meses puede explicarse por una combinación de múltiples factores. En primer lugar, algunos cierres viales ocurridos durante ese mes pudieron derivar a los conductores a manejar en zonas donde operan las cámaras de detección, incrementando su exposición a la vigilancia y, por ende, la probabilidad de ser sancionados. En segundo lugar, la Secretaría de Tránsito y Seguridad Vial del Área Metropolitana de Barranquilla sancionó a 40 conductores y propietarios de vehículos que prestaban un servicio no autorizado, lo que pudo generar una limitación en la circulación de conductores de aplicaciones como InDrive o Uber, quienes al no poder operar con normalidad, redujeron su presencia en las vías, disminuyendo la competencia y permitiendo que los vehículos de servicio público autorizado tuvieran una mayor demanda y, por tanto, una mayor exposición en las calles. Por último, este comportamiento se combinó con la época de diciembre, que es un período de alta productividad y movimiento para los vehículos de servicio público debido al aumento de la demanda por compras, reuniones familiares y eventos de fin de año. 

**Fuentes**:
- https://barranquilla.gov.co/transito/por-carrera-atletica-cierres-de-vias-para-el-sabado-24-diciembre
- https://barranquilla.gov.co/mi-barranquilla/amb-sanciona-conductores-y-propietarios-de-vehiculos-que-prestaban-un-servicio-no-autorizado
- https://barranquilla.gov.co/transito/por-evento-gran-parada-de-luz-cierre-de-vias-para-el-jueves-8-de-diciembre-de-2022

---

**Explicación para los puntos mínimos de los servicios:** Los puntos mínimos registrados en los tres servicios vehiculares (PARTICULAR, PÚBLICO y OTROS) pueden explicarse principalmente por la pandemia de COVID-19 y las estrictas medidas de restricción de movilidad implementadas durante ese período. El confinamiento obligatorio, la suspensión del servicio de Transmetro, la restricción máxima de circulación mediante el calendario de 'pico y cédula', la ley seca y la limitación de salidas exclusivamente para trabajar o por emergencias médicas redujeron drásticamente el flujo vehicular en las vías de la ciudad para todos los servicios. Con menos vehículos en circulación, las oportunidades de cometer infracciones disminuyeron significativamente, lo que se reflejó en una caída sin precedentes en los comparendos detectados para los servicios particular, público y otros durante ese período. En el caso específico del servicio OTROS (oficial y diplomático), su punto mínimo se presentó antes de la crisis nacional de la pandemia, posiblemente debido a información previa obtenida sobre operativos de control o restricciones administrativas que limitaron la circulación de estos vehículos, generando una reducción anticipada en sus niveles de infracción.

**Fuente**: https://barranquilla.gov.co/coronavirus/60-dias-de-crisis-covid19-barranquilla

### Distribución Mensual y Anual de Comparendos Electrónicos por Servicio del Vehículo

Se construyen tablas pivote que cruzan los meses y los años para cada categoría de servicio del vehículo (PARTICULAR, PÚBLICO y OTROS), utilizando la suma ponderada de `CANTIDAD_INFRACCIONES` para visualizar el volumen de infracciones en cada combinación temporal. Los resultados se presentan mediante mapas de calor independientes para cada servicio, con los meses ordenados de diciembre a enero para facilitar la identificación de patrones estacionales específicos de cada categoría. Esta visualización permite comparar el comportamiento temporal de los diferentes servicios vehiculares, identificar períodos de alta o baja actividad particulares de cada grupo, detectar estacionalidades diferenciadas según el tipo de servicio, y evaluar cómo eventos externos como la pandemia o los cierres viales afectaron de manera distinta a conductores particulares, de servicio público y oficiales en la comisión de infracciones detectadas por el sistema de cámaras.

In [27]:
servicios_heatmap = ['PARTICULAR', 'PUBLICO', 'OTROS']

for servicio in servicios_heatmap:
    df_servicio = df_comparendos_electronicos_copy[df_comparendos_electronicos_copy['SERVICIO_AGRUPADO'] == servicio]
    
    df_servicio['año'] = df_servicio['fecha_comparendo'].dt.year
    df_servicio['mes_numero'] = df_servicio['fecha_comparendo'].dt.month
    
    tabla_heatmap = df_servicio.pivot_table(
        values='CANTIDAD_INFRACCIONES',
        index='mes_numero',
        columns='año',
        aggfunc='sum',
        fill_value=0
    )
    
    tabla_heatmap.index = nombres_meses
    tabla_heatmap = tabla_heatmap.iloc[::-1]
    
    fig = go.Figure(data=go.Heatmap(
        z=tabla_heatmap.values,
        x=tabla_heatmap.columns,
        y=tabla_heatmap.index,
        colorscale='blues',
        text=tabla_heatmap.values,
        texttemplate='%{text:,.0f}',
        hovertemplate=f'Servicio: {servicio}<br>Mes: %{{y}}<br>Año: %{{x}}<br>Comparendos: %{{z:,.0f}}<extra></extra>'
    ))
    
    fig.update_layout(
        title=dict(
            text=f'Distribución de Comparendos Electrónicos por Año y Mes - Servicio: {servicio}',
            x=0.5,
            xanchor='center',
            font=dict(size=16, weight='bold')
        ),
        xaxis_title=dict(
            text='Año',
            font=dict(weight='bold')
        ),
        yaxis_title=dict(
            text='Mes',
            font=dict(weight='bold')
        ),
        template='plotly_white',
        width=1055, 
        height=500
    )
    
    fig.show()

### Distribución Mensual de Comparendos Electrónicos por Servicio del Vehículo Infractor

Se examina la variabilidad mensual de los comparendos electrónicos desagregada por cada categoría de servicio del vehículo (PARTICULAR, PÚBLICO y OTROS), agrupando los registros por mes y año y sumando la columna `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones por cada combinación. Para cada servicio, se presentan estadísticos descriptivos (media, desviación estándar, mínimo, máximo y cuartiles) para cada mes, con el fin de caracterizar su comportamiento típico y su dispersión interanual de manera específica por tipo de servicio vehicular. Complementariamente, se generan diagramas de caja (boxplot) que visualizan la distribución de las infracciones para cada mes, incluyendo la media como referencia. Para validar si las diferencias observadas entre meses son estadísticamente significativas para cada servicio, se aplica un análisis de varianza que incluye la prueba de normalidad de Shapiro-Wilk para evaluar la distribución de los datos por mes, la prueba de Levene para verificar la homogeneidad de varianzas, y dependiendo del cumplimiento de los supuestos, se realiza un ANOVA paramétrico (con prueba post-hoc de Tukey HSD) o un Kruskal-Wallis no paramétrico (con prueba post-hoc de Dunn y corrección de Bonferroni). Este análisis permite identificar, de manera diferenciada por servicio, qué meses presentan mayor variabilidad en la detección de infracciones para cada categoría de vehículo, detectar meses con comportamientos atípicos específicos de cada servicio, comparar la estabilidad estacional entre conductores particulares, de servicio público y oficiales/diplomáticos, y determinar si existe estacionalidad estadísticamente significativa en el volumen de comparendos para cada tipo de servicio vehicular.

In [28]:
for servicio in servicios_heatmap:
    df_servicio = df_comparendos_electronicos_copy[df_comparendos_electronicos_copy['SERVICIO_AGRUPADO'] == servicio]
    
    df_boxplot_servicio = df_servicio.groupby(['mes_nombre', 'año'])['CANTIDAD_INFRACCIONES'].sum().reset_index()
    df_boxplot_servicio.columns = ['mes', 'año', 'infracciones']
    df_boxplot_servicio['mes_nombre'] = df_boxplot_servicio['mes'].apply(lambda x: nombres_meses[x-1])
    
    print(f"Estadísticas Descriptivas - Servicio: {servicio}")
    estadisticas = df_boxplot_servicio.groupby('mes_nombre')['infracciones'].describe().round(0).fillna(0).astype(int).reindex(nombres_meses)
    display(estadisticas.T)
    
    display(HTML("<hr style='border: 1px solid #696969; margin: 20px 0;'>"))
    
    fig = go.Figure()
    
    fig.add_trace(go.Box(
        x=df_boxplot_servicio['mes_nombre'],
        y=df_boxplot_servicio['infracciones'],
        marker_color='mediumpurple',
        name='',
        boxmean=True,
        hovertemplate='Mes: %{x}<br>Infracciones: %{y:,.0f}<extra></extra>'
    ))
    
    fig.update_layout(
        title=dict(
            text=f'Distribución de Comparendos Electrónicos por Mes - Servicio: {servicio}',
            x=0.5,
            xanchor='center',
            font=dict(size=16, weight='bold')
        ),
        xaxis_title=dict(
            text='Mes',
            font=dict(weight='bold')
        ),
        yaxis_title=dict(
            text='Cantidad de Comparendos',
            font=dict(weight='bold')
        ),
        template='plotly_white',
        width=1055,
        height=500
    )
    
    fig.show()
    

    df_anova = df_boxplot_servicio.copy()
    
    meses_grupos = [
        df_anova[df_anova['mes_nombre'] == mes]['infracciones'].values 
        for mes in nombres_meses
    ]
    
    print("Prueba de normalidad (Shapiro-Wilk) por mes:")
    
    normalidad = []
    for mes, grupo in zip(nombres_meses, meses_grupos):
        if len(grupo) >= 3:  
            stat, p = stats.shapiro(grupo)
            normalidad.append(p > 0.05)
            print(f"{mes}: p-valor = {p:.6f} -> {'Normal' if p > 0.05 else 'No normal'}")
        else:
            print(f"{mes}: No hay suficientes datos para evaluar normalidad")
            normalidad.append(False)
    
    cumple_normalidad = all(normalidad) if normalidad else False
    print(f"¿Se cumple normalidad en todos los meses? {'Sí' if cumple_normalidad else 'No'}")
    
    grupos_validos = [g for g in meses_grupos if len(g) > 0]
    
    if len(grupos_validos) > 1:
        stat_levene, p_levene = stats.levene(*grupos_validos)
        print(f"\nLevene p-valor: {p_levene:.6f}")
        print(f"¿Varianzas iguales? {'Sí' if p_levene > 0.05 else 'No'}")
        
        if cumple_normalidad and p_levene > 0.05:
            print("\nSe cumplen los supuestos → Se aplica ANOVA")
            
            f_stat, p_valor = stats.f_oneway(*meses_grupos)
            
            print(f"Estadístico F: {f_stat:.4f}")
            print(f"P-valor: {p_valor:.6f}")
            
            print(f"¿Hay diferencias entre meses? {'Sí' if p_valor < 0.05 else 'No'}")
            
            if p_valor < 0.05:
                print("\nPrueba post-hoc Tukey HSD:")
                
                tukey = pairwise_tukeyhsd(
                    endog=df_anova['infracciones'],
                    groups=df_anova['mes_nombre'],
                    alpha=0.05
                )
                
                print(tukey)
        
        else:
            print("\nNo se cumplen los supuestos → Se aplica Kruskal-Wallis")
            
            h_stat, p_valor = stats.kruskal(*grupos_validos)
            
            print(f"Estadístico H: {h_stat:.4f}")
            print(f"P-valor: {p_valor:.6f}")
            
            print(f"¿Hay diferencias entre meses? {'Sí' if p_valor < 0.05 else 'No'}")
            
            if p_valor < 0.05:
                print("\nPrueba post-hoc Dunn (con corrección Bonferroni):")
                
                import scikit_posthocs as sp
                
                df_dunn = df_anova[['mes_nombre', 'infracciones']].copy()
                
                dunn = sp.posthoc_dunn(
                    df_dunn,
                    val_col='infracciones',
                    group_col='mes_nombre',
                    p_adjust='bonferroni'
                )
                
                dunn = dunn.loc[nombres_meses, nombres_meses]
                display(dunn)
    
    display(HTML("<hr style='border: 2px solid #696969; margin: 20px 0;'>"))

Estadísticas Descriptivas - Servicio: PARTICULAR


mes_nombre,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
count,8,8,8,8,8,8,8,8,8,8,8,8
mean,5151,5327,5909,5696,5514,5453,5606,5137,5371,5834,5285,5464
std,1676,1135,1211,1500,2400,2168,1899,1647,1280,1016,1090,1223
min,1722,3329,4489,3690,1139,2483,2355,2958,3690,3644,3495,3701
25%,4587,4605,4922,4676,4052,3532,4248,3662,4226,5716,4673,4534
50%,5340,5759,5923,5702,6267,6169,5963,5276,5555,6076,5310,5719
75%,6424,6045,6564,6387,6819,6945,6896,6143,6090,6236,6157,6132
max,7015,6691,7860,7964,8466,8277,8011,7316,7410,7158,6544,7287


Prueba de normalidad (Shapiro-Wilk) por mes:
Enero: p-valor = 0.319509 -> Normal
Febrero: p-valor = 0.453260 -> Normal
Marzo: p-valor = 0.629646 -> Normal
Abril: p-valor = 0.761740 -> Normal
Mayo: p-valor = 0.621276 -> Normal
Junio: p-valor = 0.316269 -> Normal
Julio: p-valor = 0.786343 -> Normal
Agosto: p-valor = 0.440546 -> Normal
Septiembre: p-valor = 0.760100 -> Normal
Octubre: p-valor = 0.109225 -> Normal
Noviembre: p-valor = 0.596297 -> Normal
Diciembre: p-valor = 0.866121 -> Normal
¿Se cumple normalidad en todos los meses? Sí

Levene p-valor: 0.377027
¿Varianzas iguales? Sí

Se cumplen los supuestos → Se aplica ANOVA
Estadístico F: 0.1962
P-valor: 0.997442
¿Hay diferencias entre meses? No


Estadísticas Descriptivas - Servicio: PUBLICO


mes_nombre,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
count,8,8,8,8,8,8,8,8,8,8,8,8
mean,765,810,962,998,917,922,927,826,912,987,965,955
std,239,264,339,305,428,420,365,282,242,227,334,480
min,383,410,571,657,247,362,448,447,545,520,557,550
25%,605,638,756,720,579,522,600,683,757,896,726,602
50%,770,876,906,970,1063,980,940,771,882,1040,899,806
75%,934,962,1080,1227,1218,1250,1257,1012,1112,1105,1107,1111
max,1099,1210,1637,1400,1368,1493,1368,1267,1263,1289,1506,1960


Prueba de normalidad (Shapiro-Wilk) por mes:
Enero: p-valor = 0.923401 -> Normal
Febrero: p-valor = 0.740546 -> Normal
Marzo: p-valor = 0.514293 -> Normal
Abril: p-valor = 0.185176 -> Normal
Mayo: p-valor = 0.286509 -> Normal
Junio: p-valor = 0.509459 -> Normal
Julio: p-valor = 0.298030 -> Normal
Agosto: p-valor = 0.784263 -> Normal
Septiembre: p-valor = 0.849303 -> Normal
Octubre: p-valor = 0.332906 -> Normal
Noviembre: p-valor = 0.367947 -> Normal
Diciembre: p-valor = 0.074967 -> Normal
¿Se cumple normalidad en todos los meses? Sí

Levene p-valor: 0.525194
¿Varianzas iguales? Sí

Se cumplen los supuestos → Se aplica ANOVA
Estadístico F: 0.3834
P-valor: 0.959197
¿Hay diferencias entre meses? No


Estadísticas Descriptivas - Servicio: OTROS


mes_nombre,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
count,7,8,8,8,7,8,8,7,8,8,8,8
mean,8,11,9,9,9,7,9,8,9,8,6,11
std,6,9,4,4,4,3,4,5,4,4,3,5
min,1,1,3,4,2,2,3,2,6,2,2,5
25%,6,5,6,6,8,6,6,4,6,6,4,8
50%,7,10,8,9,9,8,10,7,8,8,6,8
75%,10,13,12,11,12,8,12,12,11,10,8,14
max,20,27,16,16,13,11,15,17,17,15,11,20


Prueba de normalidad (Shapiro-Wilk) por mes:
Enero: p-valor = 0.372054 -> Normal
Febrero: p-valor = 0.440490 -> Normal
Marzo: p-valor = 0.743492 -> Normal
Abril: p-valor = 0.468263 -> Normal
Mayo: p-valor = 0.164010 -> Normal
Junio: p-valor = 0.636703 -> Normal
Julio: p-valor = 0.892889 -> Normal
Agosto: p-valor = 0.579942 -> Normal
Septiembre: p-valor = 0.050434 -> Normal
Octubre: p-valor = 0.938431 -> Normal
Noviembre: p-valor = 0.355300 -> Normal
Diciembre: p-valor = 0.291992 -> Normal
¿Se cumple normalidad en todos los meses? Sí

Levene p-valor: 0.506496
¿Varianzas iguales? Sí

Se cumplen los supuestos → Se aplica ANOVA
Estadístico F: 0.6046
P-valor: 0.819741
¿Hay diferencias entre meses? No


**Servicio PARTICULAR (525,986 infracciones)**

- Los vehículos particulares presentan sus mayores volúmenes en marzo y octubre, mientras que agosto y enero son los meses de menor actividad. La variabilidad es especialmente alta en mayo, junio y julio, influenciada por eventos atípicos como cierres viales y la pandemia.
- Todos los meses cumplen con normalidad (p-valores > 0.05, el más bajo fue octubre con 0.1092) y las varianzas son homogéneas (p=0.3770). El ANOVA arroja un estadístico F de 0.1962 con un p-valor de 0.9974, extremadamente alto y cercano a 1. **No existen diferencias estadísticamente significativas entre las medias de los meses** para los vehículos particulares. Las fluctuaciones observadas en los promedios mensuales (marzo con mayor promedio, agosto con menor) no son consistentes a lo largo de los años.

**Servicio PÚBLICO (87,553 infracciones)**

- Los vehículos públicos tienen sus pico más altos en abril, a diferencia de los particulares cuyo pico es en marzo. Además, la variabilidad es notablemente alta en diciembre y mayo.

- Todos los meses cumplen con normalidad (p-valores > 0.05, el más bajo fue diciembre con 0.0750, aún por encima del umbral) y las varianzas son homogéneas (p=0.5252). El ANOVA arroja un estadístico F de 0.3834 con un p-valor de 0.9592. **No existen diferencias estadísticamente significativas entre las medias de los meses** para los vehículos públicos.

**Servicio OTROS (oficial y diplomático) (821 infracciones)**

- El servicio OTROS presenta menos años de registro (count=7) en algunos meses (enero, mayo, agosto), lo que indica que no hay registros de infracciones de vehículos oficiales y diplomáticos en ciertos años para esos meses.

- El volumen de infracciones para vehículos oficiales y diplomáticos es muy bajo, lo que dificulta identificar patrones estacionales claros. Febrero y diciembre son los meses con mayor actividad relativa, mientras que noviembre y junio presentan los valores más bajos.

- Todos los meses cumplen con normalidad (p-valores > 0.05, el más bajo fue septiembre con 0.0504, apenas por encima del umbral) y las varianzas son homogéneas (p=0.5065). El ANOVA arroja un estadístico F de 0.6046 con un p-valor de 0.8197. **No existen diferencias estadísticamente significativas entre las medias de los meses** para los vehículos oficiales y diplomáticos.

---

**Hallazgos:**

- **Estacionalidades diferenciadas por servicio**: Los vehículos particulares tienen sus picos en marzo y octubre, mientras que los vehículos públicos alcanzan sus máximos en abril, octubre y noviembre, con un pico adicional en diciembre. Esta diferencia sugiere que los patrones de movilidad y comisión de infracciones varían según el tipo de servicio.

- **Mayo como mes crítico para particulares**: Con la mayor variabilidad (std: 2,400) y la presencia tanto del máximo como del mínimo histórico, mayo es el mes más impredecible para los conductores particulares, influenciado por los cierres viales de 2019 y la pandemia de 2020.

- **Diciembre como mes crítico para públicos**: Los vehículos públicos presentan su mayor variabilidad en diciembre (std: 480) y su máximo histórico (1,960 infracciones), posiblemente relacionado con el aumento de la demanda de transporte público durante la temporada de fin de año.

- **Bajo volumen y alta estabilidad relativa de OTROS**: Aunque el volumen es marginal, los vehículos oficiales y diplomáticos muestran una variabilidad baja en comparación con su promedio, lo que indica un comportamiento más uniforme a lo largo del año.

- Ninguna de las tres categorías de servicio (PARTICULAR, PÚBLICO y OTROS) presenta diferencias estadísticamente significativas en el volumen de comparendos a lo largo de los meses del año. Esto indica que, independientemente del tipo de servicio, los patrones mensuales se mantienen constantes sin que ningún mes destaque consistentemente por encima de los demás.

- Estos resultados complementan el análisis de descomposición STL, donde se observó que PARTICULAR tenía una amplitud estacional de 2,823 comparendos y PÚBLICO de 598 comparendos, con variabilidades explicadas del 12.34% y 12.47% respectivamente. Sin embargo, al someter las diferencias mensuales a una prueba estadística (ANOVA), se confirma que las oscilaciones observadas entre meses no son lo suficientemente grandes ni consistentes a lo largo de los años como para ser consideradas estadísticamente significativas.

- OTROS presenta el p-valor más bajo pero aún no significativo con un p-valor de 0.8197, es el más cercano a 0.05 entre los tres servicios, lo que podría indicar una leve tendencia hacia diferencias mensuales, pero aún muy por encima del umbral de significancia.

---

**Notas:**

- Las campañas de control y educación vial para particulares deben intensificarse en marzo y octubre, con especial atención en mayo por su alta imprevisibilidad.
- Para vehículos públicos, se debe reforzar el control en abril, octubre y especialmente en diciembre, mes de alta demanda y alta variabilidad.
- La baja incidencia de vehículos OTROS sugiere que no requieren estrategias específicas de control, aunque febrero y diciembre son los meses de mayor actividad relativa.

## Distribución Geográfica de Comparendos Electrónicos por Clase del Vehículo Infractor

Se examina, para cada clase de vehículo (AUTOMÓVIL, CAMIONETA, MOTOCICLETA, CAMPERO, VEHÍCULOS PESADOS y OTROS), la distribución de los comparendos electrónicos según la ubicación específica de la cámara (variable `Camara_y_direccion`), utilizando la suma ponderada de `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones por cada punto de control. Para cada clase, se presentan tablas de frecuencias que muestran las ubicaciones con sus respectivos conteos, porcentajes dentro de la clase, y las fechas de primer y último registro, permitiendo identificar los puntos críticos de detección para cada tipo de vehículo, evaluar la antigüedad y continuidad operativa de las cámaras para cada clase, y comprender si existen patrones geográficos diferenciados en la comisión de infracciones según el tipo de vehículo.

In [29]:
clases = df_comparendos_electronicos_copy['CLASE_AGRUPADA'].unique()

for clase in clases:
    df_clase = df_comparendos_electronicos_copy[df_comparendos_electronicos_copy['CLASE_AGRUPADA'] == clase]
    
    total_clase = df_clase['CANTIDAD_INFRACCIONES'].sum()
    
    df_ubicaciones = df_clase.groupby('Camara_y_direccion')['CANTIDAD_INFRACCIONES'].sum().sort_values(ascending=False).reset_index()
    df_ubicaciones.columns = ['Cámara y Dirección', 'Comparendos']
    
    df_ubicaciones['Porcentaje'] = (df_ubicaciones['Comparendos'] / total_clase * 100).round(2)
    
    fechas = df_clase.groupby('Camara_y_direccion')['fecha_comparendo'].agg(['min', 'max']).reset_index()
    fechas.columns = ['Cámara y Dirección', 'Primer Registro', 'Último Registro']
    fechas['Primer Registro'] = pd.to_datetime(fechas['Primer Registro']).dt.date
    fechas['Último Registro'] = pd.to_datetime(fechas['Último Registro']).dt.date
    
    df_ubicaciones = df_ubicaciones.merge(fechas, on='Cámara y Dirección')
    
    print(f"Distribución de Comparendos Electrónicos - Clase: {clase}")
    pd.set_option('display.max_columns', None)  
    pd.set_option('display.width', None)        
    pd.set_option('display.max_colwidth', None) 
    display(df_ubicaciones.T)
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')
    pd.reset_option('display.max_colwidth')
    
    display(HTML("<hr style='border: 2px solid #696969; margin: 20px 0;'>"))

Distribución de Comparendos Electrónicos - Clase: AUTOMOVIL


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51
Cámara y Dirección,Carro mal parqueo Sur,Carro mal parqueo Norte,Carro mal parqueo Sur - Norte,VIA 11 CON CARRERA 8,CARRERA 51B CON CALLE 79,CALLE 82 CON CARRERA 51B,Carro mal parqueo alto norte,CARRERA 53 ENTRE CALLE 104 Y 106,CALLE 30 CON CARRERA 6B,CARRERA 6 CON CALLE 72,CALLE 30 CON CARRERA 8,AVENIDA CIRCUNVALAR CON CARRERA 9G,CALLE 45 CON CARRERA 20,Carro mal parqueo alto sur,CALLE 45B CON CARRERA 14,VIA 40 CON CALLE 73,CARRERA 51B CON CALLE 103,CALLE 45 CON CARRERA 1,CALLE 82 CON CARRERA 56,CALLE 45 CON CARRERA 21,CALLE 72 CON CARRERA 44,CALLE 53 CON CARRERA 45,CALLE 84 ENTRE CARRERA 59 Y 59B,CARRERA 46 CON CALLE 100,CALLE 76 CON CARRERA 38C-100,CARRERA 15 CON CALLE 21,CALLE 94 CON CARRERA 58,CALLE 70 CON CARRERA 46,CALLE 19 CON CARRERA 4C,CALLE 19 CON CARRERA 3D,CALLE 34 CON CARRERA 45,CALLE 56 CON CARRERA 14,CALLE 87 CON CARRERA 21,CARRERA 53 CON CALLE 86,AVENIDA CIRCUNVALAR CON CARRERA 31,CALLE 98 CON CARRERA 56,CALLE 45 CON CARRERA 21 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 21 SENTIDO NORTE-SUR,CALLE 61 CON CARRERA 35,CARRERA 46 CON CALLE 34 SENTIDO SUR-NORTE,CALLE 70 CON CARRERA 46 SENTIDO SUR-NORTE,CALLE 70 CON CARRERA 46 SENTIDO NORTE-SUR,CARRERA 27 CON CALLE 82C,CALLE 45 CON CARRERA 8 SENTIDO SUR-NORTE,AVENIDA CIRCUNVALAR (CALLE 110) ENTRE CARRERA 9G Y 12 SENTIDO NORTE A SUR,CALLE 45 CON CARRERA 33 SENTIDO NORTE-SUR,AVENIDA CIRCUNVALAR (CALLE 110) ENTRE CARRERA 9G Y 12 SENTIDO SUR A NORTE,CALLE 45 CON CARRERA 43 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 1E SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 43 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 1E SENTIDO NORTE-SUR,CARRERA 46 CON CALLE 34 SENTIDO NORTE-SUR
Comparendos,48261,41541,25869,21785,13015,11601,10089,9986,9766,8651,8425,8262,8171,7407,7286,7127,6856,6836,6626,6594,6064,4882,4752,3326,3047,2963,2910,2907,2294,2024,1938,1792,1612,1525,1144,1089,779,693,598,548,435,219,118,50,25,20,16,9,7,7,3,3
Porcentaje,14.99,12.9,8.04,6.77,4.04,3.6,3.13,3.1,3.03,2.69,2.62,2.57,2.54,2.3,2.26,2.21,2.13,2.12,2.06,2.05,1.88,1.52,1.48,1.03,0.95,0.92,0.9,0.9,0.71,0.63,0.6,0.56,0.5,0.47,0.36,0.34,0.24,0.22,0.19,0.17,0.14,0.07,0.04,0.02,0.01,0.01,0.0,0.0,0.0,0.0,0.0,0.0
Primer Registro,2018-01-02,2018-01-02,2018-09-29,2018-01-01,2018-01-01,2018-01-11,2023-06-28,2018-10-19,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2023-08-15,2018-01-04,2018-01-02,2018-01-04,2018-01-01,2018-01-09,2018-01-01,2018-01-10,2018-01-04,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-09,2018-01-01,2018-01-01,2018-01-03,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-10-14,2024-11-24,2024-11-22,2018-01-30,2025-10-23,2025-10-21,2025-10-21,2018-01-02,2025-10-23,2024-12-07,2025-10-23,2024-12-21,2025-10-25,2025-10-21,2025-10-24,2025-10-21,2025-10-21
Último Registro,2025-12-31,2025-12-31,2025-12-30,2025-12-29,2025-12-28,2025-12-30,2025-12-31,2025-12-30,2025-12-29,2025-12-25,2025-12-30,2024-11-18,2024-11-19,2025-12-31,2025-12-31,2025-12-20,2025-12-08,2025-12-30,2025-12-29,2025-10-05,2025-11-07,2025-12-30,2025-12-28,2025-12-31,2025-12-29,2025-12-16,2025-08-31,2025-12-29,2025-12-30,2025-12-25,2025-12-31,2021-05-02,2021-05-23,2022-07-18,2019-07-23,2024-08-05,2025-12-28,2025-12-31,2025-07-03,2025-11-17,2025-12-31,2025-12-11,2020-03-02,2025-12-29,2025-12-10,2025-12-29,2025-10-08,2025-11-05,2025-10-24,2025-11-01,2025-10-23,2025-10-25


Distribución de Comparendos Electrónicos - Clase: CAMIONETA


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50
Cámara y Dirección,Carro mal parqueo Sur,Carro mal parqueo Norte,VIA 11 CON CARRERA 8,Carro mal parqueo Sur - Norte,CARRERA 51B CON CALLE 79,CALLE 82 CON CARRERA 51B,Carro mal parqueo alto norte,CARRERA 53 ENTRE CALLE 104 Y 106,CARRERA 51B CON CALLE 103,VIA 40 CON CALLE 73,AVENIDA CIRCUNVALAR CON CARRERA 9G,CALLE 82 CON CARRERA 56,CALLE 30 CON CARRERA 6B,CARRERA 6 CON CALLE 72,Carro mal parqueo alto sur,CALLE 30 CON CARRERA 8,CALLE 84 ENTRE CARRERA 59 Y 59B,CALLE 45B CON CARRERA 14,CALLE 72 CON CARRERA 44,CALLE 45 CON CARRERA 21,CALLE 45 CON CARRERA 20,CARRERA 46 CON CALLE 100,CALLE 94 CON CARRERA 58,CALLE 53 CON CARRERA 45,CALLE 45 CON CARRERA 1,CALLE 76 CON CARRERA 38C-100,CARRERA 53 CON CALLE 86,CARRERA 15 CON CALLE 21,CALLE 70 CON CARRERA 46,CALLE 19 CON CARRERA 4C,CALLE 19 CON CARRERA 3D,CALLE 98 CON CARRERA 56,AVENIDA CIRCUNVALAR CON CARRERA 31,CALLE 56 CON CARRERA 14,CALLE 87 CON CARRERA 21,CALLE 34 CON CARRERA 45,CALLE 45 CON CARRERA 21 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 21 SENTIDO NORTE-SUR,CALLE 61 CON CARRERA 35,CARRERA 46 CON CALLE 34 SENTIDO SUR-NORTE,CALLE 70 CON CARRERA 46 SENTIDO SUR-NORTE,CALLE 70 CON CARRERA 46 SENTIDO NORTE-SUR,CARRERA 27 CON CALLE 82C,CALLE 45 CON CARRERA 8 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 33 SENTIDO NORTE-SUR,AVENIDA CIRCUNVALAR (CALLE 110) ENTRE CARRERA 9G Y 12 SENTIDO NORTE A SUR,AVENIDA CIRCUNVALAR (CALLE 110) ENTRE CARRERA 9G Y 12 SENTIDO SUR A NORTE,CALLE 45 CON CARRERA 43 SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 1E SENTIDO SUR-NORTE,CALLE 45 CON CARRERA 43 SENTIDO NORTE-SUR,CARRERA 46 CON CALLE 34 SENTIDO NORTE-SUR
Comparendos,18816,17064,11904,9155,5937,5667,5114,4828,3919,3854,3853,3688,3139,2989,2825,2575,2471,2091,1730,1717,1676,1563,1559,1252,1216,1148,988,953,935,917,821,706,673,417,397,309,223,176,171,140,96,67,47,11,9,9,6,3,1,1,1
Porcentaje,14.49,13.14,9.17,7.05,4.57,4.37,3.94,3.72,3.02,2.97,2.97,2.84,2.42,2.3,2.18,1.98,1.9,1.61,1.33,1.32,1.29,1.2,1.2,0.96,0.94,0.88,0.76,0.73,0.72,0.71,0.63,0.54,0.52,0.32,0.31,0.24,0.17,0.14,0.13,0.11,0.07,0.05,0.04,0.01,0.01,0.01,0.0,0.0,0.0,0.0,0.0
Primer Registro,2018-01-02,2018-01-02,2018-01-01,2018-09-29,2018-01-01,2018-01-10,2023-06-28,2018-10-18,2018-01-05,2018-01-02,2018-01-01,2018-01-09,2018-01-01,2018-01-02,2023-08-15,2018-01-01,2018-01-03,2018-01-04,2018-05-23,2018-01-01,2018-01-01,2018-01-01,2018-01-02,2019-02-21,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-27,2018-01-02,2018-01-01,2018-10-15,2018-01-01,2018-01-04,2018-01-06,2018-01-20,2024-11-29,2024-11-24,2018-01-31,2025-10-23,2025-10-21,2025-10-22,2018-02-07,2025-11-02,2025-11-19,2024-12-22,2024-12-08,2025-10-25,2025-10-21,2025-11-01,2025-10-21
Último Registro,2025-12-31,2025-12-30,2025-12-29,2025-12-30,2025-12-28,2025-12-29,2025-12-31,2025-12-31,2025-12-09,2025-12-19,2024-11-04,2025-12-30,2025-12-28,2025-12-21,2025-12-31,2025-12-28,2025-12-28,2025-12-30,2025-11-07,2025-10-05,2024-11-17,2025-12-22,2025-08-24,2025-12-27,2025-12-25,2025-12-27,2022-07-18,2025-10-27,2025-12-23,2025-12-17,2025-12-27,2024-08-05,2019-07-23,2021-04-28,2021-05-17,2025-12-30,2025-12-28,2025-12-31,2025-06-22,2025-11-17,2025-12-31,2025-12-11,2020-02-24,2025-12-27,2025-12-24,2025-12-29,2025-10-04,2025-11-01,2025-10-21,2025-11-01,2025-10-21


Distribución de Comparendos Electrónicos - Clase: CAMPERO


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34
Cámara y Dirección,CALLE 45 CON CARRERA 1,VIA 11 CON CARRERA 8,CARRERA 51B CON CALLE 79,CALLE 30 CON CARRERA 8,CALLE 30 CON CARRERA 6B,CARRERA 53 ENTRE CALLE 104 Y 106,CARRERA 15 CON CALLE 21,CALLE 56 CON CARRERA 14,CARRERA 51B CON CALLE 103,CALLE 87 CON CARRERA 21,CALLE 19 CON CARRERA 4C,CALLE 19 CON CARRERA 3D,CALLE 94 CON CARRERA 58,CARRERA 6 CON CALLE 72,CALLE 45 CON CARRERA 20,CARRERA 46 CON CALLE 100,AVENIDA CIRCUNVALAR CON CARRERA 9G,CALLE 76 CON CARRERA 38C-100,CARRERA 53 CON CALLE 86,CALLE 45 CON CARRERA 21,CALLE 61 CON CARRERA 35,CARRERA 27 CON CALLE 82C,CALLE 45 CON CARRERA 21 SENTIDO NORTE-SUR,CALLE 84 ENTRE CARRERA 59 Y 59B,CALLE 45 CON CARRERA 21 SENTIDO SUR-NORTE,AVENIDA CIRCUNVALAR (CALLE 110) ENTRE CARRERA 9G Y 12 SENTIDO SUR A NORTE,VIA 40 CON CALLE 73,AVENIDA CIRCUNVALAR (CALLE 110) ENTRE CARRERA 9G Y 12 SENTIDO NORTE A SUR,CALLE 72 CON CARRERA 44,CALLE 45B CON CARRERA 14,CALLE 82 CON CARRERA 51B,AVENIDA CIRCUNVALAR CON CARRERA 31,CALLE 53 CON CARRERA 45,CALLE 70 CON CARRERA 46,CALLE 82 CON CARRERA 56
Comparendos,13923,11862,11781,9394,9325,5716,5358,4800,4513,4010,2969,2690,2659,2485,2248,1987,1762,1630,1356,984,349,208,192,189,115,78,78,28,7,6,2,1,1,1,1
Porcentaje,13.56,11.55,11.47,9.15,9.08,5.57,5.22,4.67,4.39,3.9,2.89,2.62,2.59,2.42,2.19,1.93,1.72,1.59,1.32,0.96,0.34,0.2,0.19,0.18,0.11,0.08,0.08,0.03,0.01,0.01,0.0,0.0,0.0,0.0,0.0
Primer Registro,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-10-17,2018-01-01,2018-01-01,2018-01-02,2018-01-01,2018-01-01,2018-01-01,2018-01-03,2018-01-01,2018-01-01,2018-01-01,2018-01-01,2018-01-02,2018-01-07,2018-01-04,2018-01-30,2018-01-26,2024-11-24,2022-08-21,2024-11-24,2024-11-29,2024-11-23,2024-12-06,2023-06-20,2022-05-19,2024-09-20,2018-05-23,2024-09-07,2025-02-07,2024-02-10
Último Registro,2025-12-30,2025-12-30,2025-12-27,2025-12-31,2025-12-30,2025-12-31,2025-12-19,2021-05-03,2025-12-12,2021-05-24,2025-12-30,2025-12-28,2025-07-07,2025-11-24,2024-10-30,2025-12-30,2024-11-28,2025-12-31,2022-07-19,2025-10-05,2025-07-03,2020-02-20,2025-12-31,2025-12-30,2025-09-23,2025-10-07,2025-12-07,2025-12-23,2024-10-22,2025-07-14,2024-12-18,2018-05-23,2024-09-07,2025-02-07,2024-02-10


Distribución de Comparendos Electrónicos - Clase: VEHÍCULOS PESADOS


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44
Cámara y Dirección,VIA 11 CON CARRERA 8,CARRERA 6 CON CALLE 72,CALLE 30 CON CARRERA 6B,Carro mal parqueo Sur,CALLE 30 CON CARRERA 8,CALLE 45B CON CARRERA 14,CALLE 34 CON CARRERA 45,CALLE 82 CON CARRERA 51B,AVENIDA CIRCUNVALAR CON CARRERA 9G,CALLE 72 CON CARRERA 44,Carro mal parqueo Norte,Carro mal parqueo Sur - Norte,CALLE 45 CON CARRERA 1,Carro mal parqueo alto norte,CALLE 19 CON CARRERA 4C,CARRERA 53 ENTRE CALLE 104 Y 106,CALLE 82 CON CARRERA 56,CARRERA 15 CON CALLE 21,CALLE 53 CON CARRERA 45,CALLE 19 CON CARRERA 3D,VIA 40 CON CALLE 73,CALLE 45 CON CARRERA 21,CALLE 56 CON CARRERA 14,CALLE 45 CON CARRERA 20,CARRERA 51B CON CALLE 79,CALLE 70 CON CARRERA 46,Carro mal parqueo alto sur,CALLE 87 CON CARRERA 21,CARRERA 46 CON CALLE 100,CALLE 94 CON CARRERA 58,CALLE 84 ENTRE CARRERA 59 Y 59B,AVENIDA CIRCUNVALAR CON CARRERA 31,CARRERA 51B CON CALLE 103,CALLE 76 CON CARRERA 38C-100,CALLE 98 CON CARRERA 56,CALLE 45 CON CARRERA 21 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 21 SENTIDO SUR-NORTE,CARRERA 53 CON CALLE 86,CARRERA 46 CON CALLE 34 SENTIDO SUR-NORTE,CALLE 61 CON CARRERA 35,CALLE 45 CON CARRERA 43 SENTIDO SUR-NORTE,CARRERA 27 CON CALLE 82C,CALLE 45 CON CARRERA 43 SENTIDO NORTE-SUR,CALLE 45 CON CARRERA 8 SENTIDO SUR-NORTE,CALLE 70 CON CARRERA 46 SENTIDO NORTE-SUR
Comparendos,2900,1659,1197,1193,906,883,839,816,756,725,722,594,586,522,483,447,411,402,392,345,320,304,291,259,259,217,212,205,181,132,131,103,94,87,60,27,27,27,6,5,5,3,1,1,1
Porcentaje,14.69,8.41,6.07,6.04,4.59,4.47,4.25,4.13,3.83,3.67,3.66,3.01,2.97,2.64,2.45,2.26,2.08,2.04,1.99,1.75,1.62,1.54,1.47,1.31,1.31,1.1,1.07,1.04,0.92,0.67,0.66,0.52,0.48,0.44,0.3,0.14,0.14,0.14,0.03,0.03,0.03,0.02,0.01,0.01,0.01
Primer Registro,2018-01-10,2018-01-08,2018-01-01,2018-01-03,2018-01-01,2018-01-21,2018-01-03,2018-01-11,2018-01-02,2018-01-18,2018-01-10,2018-09-29,2018-01-02,2023-06-28,2018-01-05,2018-10-26,2018-01-25,2018-01-08,2018-01-04,2018-01-01,2018-01-05,2018-01-08,2018-01-11,2018-01-01,2018-01-08,2018-04-27,2023-08-15,2018-01-06,2018-01-08,2018-01-01,2018-01-17,2018-01-02,2018-01-07,2018-01-05,2018-11-02,2024-12-07,2024-11-25,2018-04-10,2025-10-25,2018-02-17,2025-10-25,2018-03-21,2025-11-02,2025-12-09,2025-10-31
Último Registro,2025-12-25,2025-11-30,2025-12-25,2025-12-31,2025-12-09,2025-12-18,2025-12-30,2025-12-19,2024-11-01,2025-11-07,2025-12-17,2025-12-29,2025-12-28,2025-12-27,2025-12-22,2025-12-30,2025-10-30,2025-12-19,2025-11-28,2025-12-25,2025-11-23,2025-09-22,2021-04-26,2024-10-29,2025-12-25,2025-12-06,2025-12-17,2021-05-19,2025-10-15,2025-08-24,2025-07-16,2019-07-13,2025-04-04,2025-12-04,2024-07-04,2025-12-11,2025-11-09,2021-12-20,2025-11-15,2024-11-16,2025-11-01,2018-06-15,2025-11-02,2025-12-09,2025-10-31


Distribución de Comparendos Electrónicos - Clase: OTROS


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24
Cámara y Dirección,CALLE 19 CON CARRERA 3D,CALLE 30 CON CARRERA 6B,VIA 11 CON CARRERA 8,CARRERA 51B CON CALLE 79,CALLE 30 CON CARRERA 8,CALLE 45 CON CARRERA 1,CALLE 87 CON CARRERA 21,CALLE 19 CON CARRERA 4C,CARRERA 6 CON CALLE 72,CARRERA 53 ENTRE CALLE 104 Y 106,Carro mal parqueo Sur,CARRERA 46 CON CALLE 100,CARRERA 15 CON CALLE 21,CALLE 76 CON CARRERA 38C-100,CALLE 45 CON CARRERA 20,CALLE 45 CON CARRERA 21,CALLE 56 CON CARRERA 14,CARRERA 53 CON CALLE 86,CARRERA 51B CON CALLE 103,CALLE 82 CON CARRERA 51B,CALLE 82 CON CARRERA 56,Carro mal parqueo alto norte,CALLE 45 CON CARRERA 21 SENTIDO SUR-NORTE,CARRERA 46 CON CALLE 34 SENTIDO SUR-NORTE,CARRERA 27 CON CALLE 82C
Comparendos,37,20,16,16,12,12,7,7,6,6,6,5,5,4,4,3,2,2,2,2,2,2,1,1,1
Porcentaje,20.44,11.05,8.84,8.84,6.63,6.63,3.87,3.87,3.31,3.31,3.31,2.76,2.76,2.21,2.21,1.66,1.1,1.1,1.1,1.1,1.1,1.1,0.55,0.55,0.55
Primer Registro,2018-01-11,2018-06-17,2018-01-04,2021-03-09,2019-04-05,2021-11-13,2019-04-01,2018-12-31,2018-01-14,2023-11-29,2022-06-24,2018-12-10,2019-12-14,2024-05-08,2019-04-22,2018-02-13,2019-07-14,2019-10-25,2020-09-23,2024-05-15,2023-08-01,2024-05-14,2025-02-16,2025-10-30,2019-03-13
Último Registro,2025-07-13,2025-05-27,2025-08-25,2025-01-30,2025-06-24,2025-12-26,2021-03-19,2025-04-14,2024-05-31,2025-09-02,2024-12-04,2025-05-03,2024-06-21,2025-02-20,2024-02-29,2022-08-25,2021-02-05,2020-02-04,2022-01-08,2025-10-08,2025-10-22,2025-05-22,2025-02-16,2025-10-30,2019-03-13


## Evolución Mensual de Comparendos Electrónicos por Clase del Vehículo Infractor

Se examina la evolución mensual de los comparendos electrónicos desagregada por cada categoría de clase del vehículo (AUTOMÓVIL, CAMIONETA, MOTOCICLETA, CAMPERO, VEHÍCULOS PESADOS y OTROS), agrupando los registros por mes y sumando la columna `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones por cada período. Para cada clase, se presentan tablas de frecuencias que muestran los meses con mayor y menor volumen, junto con sus porcentajes de participación dentro del total de esa clase. Complementariamente, se generan gráficos de líneas interactivos que ilustran la tendencia temporal de cada categoría, incluyendo estadísticos como el promedio mensual, la desviación estándar, y los meses con valores máximo y mínimo, con un área sombreada que destaca el período crítico de la pandemia de COVID-19. Este análisis permite identificar patrones estacionales diferenciados por tipo de vehículo, evaluar el impacto diferencial de eventos externos como cierres viales o la pandemia en cada clase de vehículo, y comparar la variabilidad temporal entre los distintos tipos de vehículos infractores detectados por el sistema de cámaras.

In [30]:
infracciones_por_mes_clase = df_comparendos_electronicos_copy.groupby(['año_mes', 'CLASE_AGRUPADA'])['CANTIDAD_INFRACCIONES'].sum().reset_index()

for clase in clases:
    df_clase_mensual = infracciones_por_mes_clase[infracciones_por_mes_clase['CLASE_AGRUPADA'] == clase].copy()
    df_clase_mensual = df_clase_mensual[['año_mes', 'CANTIDAD_INFRACCIONES']].sort_values('año_mes').reset_index(drop=True)
    df_clase_mensual.columns = ['Mes', 'Comparendos']
    
    total_clase = df_clase_mensual['Comparendos'].sum()
    
    df_clase_mensual['Porcentaje'] = (df_clase_mensual['Comparendos'] / total_clase * 100).round(2)
    
    df_clase_mensual = df_clase_mensual.sort_values('Comparendos', ascending=False).reset_index(drop=True)
    
    print(f"Distribución de Comparendos Electrónicos por Mes - Clase: {clase}")
    pd.set_option('display.max_columns', None)  
    pd.set_option('display.width', None)        
    pd.set_option('display.max_colwidth', None) 
    display(df_clase_mensual.T)
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')
    pd.reset_option('display.max_colwidth')
    
    display(HTML("<hr style='border: 2px solid #696969; margin: 20px 0;'>"))

for clase in clases:
    df_clase = infracciones_por_mes_clase[infracciones_por_mes_clase['CLASE_AGRUPADA'] == clase]
    
    total_clase = df_clase['CANTIDAD_INFRACCIONES'].sum()
    desviacion_clase = df_clase['CANTIDAD_INFRACCIONES'].std()
    promedio_clase = df_clase['CANTIDAD_INFRACCIONES'].mean()
    
    meses_clase = sorted(df_clase['año_mes'].unique())
    
    if len(meses_clase) > 12:
        tick_positions = meses_clase[::6]
    elif len(meses_clase) > 6:
        tick_positions = meses_clase[::3]
    else:
        tick_positions = meses_clase
    
    max_valor = df_clase['CANTIDAD_INFRACCIONES'].max()
    max_fila = df_clase[df_clase['CANTIDAD_INFRACCIONES'] == max_valor].iloc[0]
    max_mes = max_fila['año_mes']
    
    min_valor = df_clase['CANTIDAD_INFRACCIONES'].min()
    min_fila = df_clase[df_clase['CANTIDAD_INFRACCIONES'] == min_valor].iloc[0]
    min_mes = min_fila['año_mes']
    
    fig = go.Figure()
    
    # Línea principal
    fig.add_trace(go.Scatter(
        x=df_clase['año_mes'],
        y=df_clase['CANTIDAD_INFRACCIONES'],
        mode='lines+markers',
        name=clase,
        line=dict(color='cornflowerblue', width=2),
        marker=dict(size=4, color='cornflowerblue'),
        hovertemplate=f'Clase: {clase}<br>Mes: %{{x}}<br>Infracciones: %{{y:,.0f}}<extra></extra>',
        showlegend=False
    ))
    
    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='none',
        name=f'Total: {int(total_clase):,}',
        showlegend=True
    ))
    
    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='none',
        name=f'Std: {int(desviacion_clase):,}',
        showlegend=True
    ))
    
    x_min = df_clase['año_mes'].iloc[0]
    x_max = df_clase['año_mes'].iloc[-1]
    fig.add_trace(go.Scatter(
        x=[x_min, x_max],
        y=[promedio_clase, promedio_clase],
        mode='lines',
        name=f'Promedio: {int(promedio_clase):,}',
        line=dict(color='red', width=1.4, dash='dot'),
        showlegend=True,
        hoverinfo='none'
    ))
    
    fig.add_trace(go.Scatter(
        x=[max_mes],
        y=[max_valor],
        mode='markers',
        name=f'Máximo: {max_mes} ({int(max_valor):,})',
        marker=dict(size=8, color='red', symbol='circle'),
        hoverinfo='none'
    ))
    
    fig.add_trace(go.Scatter(
        x=[min_mes],
        y=[min_valor],
        mode='markers',
        name=f'Mínimo: {min_mes} ({int(min_valor):,})',
        marker=dict(size=8, color='green', symbol='circle'),
        hoverinfo='none'
    ))
    
    fig.add_vrect(
        x0="2020-03", x1="2020-12",
        fillcolor="red", opacity=0.1,
        line_width=0,
        annotation_text="COVID-19", 
        annotation_position="top left",
        annotation=dict(font_size=12, font_color="red")
    )
    
    fig.update_layout(
        title=dict(
            text=f'Evolución Mensual de Comparendos - Clase: {clase}',
            x=0.5,
            xanchor='center',
            font=dict(size=16, weight='bold')
        ),
        xaxis_title=dict(
            text='Año-Mes',
            font=dict(weight='bold')
        ),
        yaxis_title=dict(
            text='Cantidad de Comparendos',
            font=dict(weight='bold')
        ),
        template='plotly_white',
        hovermode='x unified',
        legend=dict(
            x=1, 
            y=1, 
            xanchor='center',
            yanchor='top',
            bgcolor='rgba(255,255,255,0.5)',
            font=dict(size=12)
        ),
        width=1055,
        height=500
    )
    
    fig.update_xaxes(
        tickangle=-45,
        tickvals=tick_positions,
        ticktext=tick_positions
    )
    
    fig.show()

Distribución de Comparendos Electrónicos por Mes - Clase: AUTOMOVIL


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95
Mes,2019-05,2019-06,2023-07,2019-03,2019-04,2023-09,2019-07,2022-12,2023-08,2023-10,2022-06,2023-06,2023-12,2024-05,2024-04,2018-01,2023-05,2019-08,2018-11,2023-03,2022-11,2022-07,2022-05,2023-11,2024-03,2018-03,2018-10,2024-06,2024-07,2024-09,2020-01,2023-04,2018-12,2019-02,2022-10,2020-02,2018-05,2024-10,2024-01,2020-10,2024-02,2018-04,2022-09,2019-12,2019-10,2021-02,2019-09,2022-04,2025-10,2018-02,2022-08,2018-06,2019-11,2018-07,2021-03,2018-09,2018-08,2024-08,2019-01,2020-12,2025-11,2020-11,2023-02,2021-01,2024-11,2025-04,2024-12,2025-01,2020-03,2025-03,2022-03,2025-09,2021-07,2025-02,2025-07,2022-01,2025-05,2020-09,2021-12,2025-12,2021-04,2021-08,2021-10,2021-09,2025-06,2021-11,2022-02,2021-05,2020-08,2021-06,2025-08,2020-04,2023-01,2020-06,2020-07,2020-05
Comparendos,5212,5175,5050,4915,4905,4892,4832,4770,4751,4701,4633,4602,4575,4567,4500,4342,4322,4317,4306,4245,4173,4172,4123,4118,4106,4067,4062,4035,3978,3961,3959,3937,3843,3817,3791,3746,3744,3741,3732,3723,3686,3661,3622,3601,3571,3569,3562,3530,3487,3437,3417,3412,3313,3297,3250,3232,3231,3208,3183,3158,3140,3029,2953,2916,2910,2849,2835,2831,2812,2760,2744,2689,2637,2555,2543,2506,2496,2400,2390,2265,2245,2227,2187,2179,2164,2097,1974,1863,1727,1695,1649,1503,999,953,934,430
Porcentaje,1.62,1.61,1.57,1.53,1.52,1.52,1.5,1.48,1.48,1.46,1.44,1.43,1.42,1.42,1.4,1.35,1.34,1.34,1.34,1.32,1.3,1.3,1.28,1.28,1.28,1.26,1.26,1.25,1.24,1.23,1.23,1.22,1.19,1.19,1.18,1.16,1.16,1.16,1.16,1.16,1.14,1.14,1.13,1.12,1.11,1.11,1.11,1.1,1.08,1.07,1.06,1.06,1.03,1.02,1.01,1.0,1.0,1.0,0.99,0.98,0.98,0.94,0.92,0.91,0.9,0.88,0.88,0.88,0.87,0.86,0.85,0.84,0.82,0.79,0.79,0.78,0.78,0.75,0.74,0.7,0.7,0.69,0.68,0.68,0.67,0.65,0.61,0.58,0.54,0.53,0.51,0.47,0.31,0.3,0.29,0.13


Distribución de Comparendos Electrónicos por Mes - Clase: CAMIONETA


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95
Mes,2019-05,2023-09,2023-12,2024-04,2023-07,2024-05,2019-06,2023-10,2019-04,2023-08,2024-03,2019-07,2024-09,2024-07,2024-06,2022-12,2024-10,2023-11,2023-06,2019-03,2025-10,2019-08,2022-06,2024-08,2023-05,2023-03,2022-11,2024-01,2019-02,2024-02,2018-01,2018-11,2020-01,2022-07,2022-05,2019-10,2022-10,2018-03,2023-04,2025-04,2021-02,2022-09,2019-09,2020-10,2019-12,2025-01,2024-11,2025-11,2018-10,2024-12,2020-02,2018-12,2021-03,2018-05,2019-11,2022-08,2018-04,2025-05,2019-01,2025-03,2020-12,2025-02,2025-09,2018-06,2018-07,2025-07,2018-02,2020-11,2022-04,2018-09,2021-01,2021-07,2023-02,2018-08,2025-06,2025-12,2022-03,2021-10,2022-01,2021-08,2021-12,2020-03,2021-09,2021-11,2025-08,2020-09,2021-04,2021-05,2022-02,2020-08,2021-06,2020-04,2023-01,2020-06,2020-07,2020-05
Comparendos,2091,2076,2043,2009,1992,1984,1973,1942,1902,1883,1818,1812,1802,1784,1783,1777,1762,1755,1747,1733,1719,1696,1677,1655,1654,1620,1596,1574,1546,1544,1523,1498,1494,1492,1479,1465,1463,1455,1451,1446,1426,1423,1416,1410,1408,1381,1381,1380,1375,1350,1344,1342,1338,1319,1318,1308,1293,1291,1282,1278,1277,1271,1254,1239,1208,1194,1186,1171,1170,1148,1133,1129,1126,1119,1095,1089,1049,995,995,991,984,982,921,878,862,853,832,784,738,721,689,597,455,399,358,157
Porcentaje,1.61,1.6,1.57,1.55,1.53,1.53,1.52,1.5,1.47,1.45,1.4,1.4,1.39,1.37,1.37,1.37,1.36,1.35,1.35,1.33,1.32,1.31,1.29,1.27,1.27,1.25,1.23,1.21,1.19,1.19,1.17,1.15,1.15,1.15,1.14,1.13,1.13,1.12,1.12,1.11,1.1,1.1,1.09,1.09,1.08,1.06,1.06,1.06,1.06,1.04,1.04,1.03,1.03,1.02,1.02,1.01,1.0,0.99,0.99,0.98,0.98,0.98,0.97,0.95,0.93,0.92,0.91,0.9,0.9,0.88,0.87,0.87,0.87,0.86,0.84,0.84,0.81,0.77,0.77,0.76,0.76,0.76,0.71,0.68,0.66,0.66,0.64,0.6,0.57,0.56,0.53,0.46,0.35,0.31,0.28,0.12


Distribución de Comparendos Electrónicos por Mes - Clase: CAMPERO


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95
Mes,2020-04,2019-03,2019-08,2019-02,2019-07,2019-04,2024-06,2022-04,2019-05,2024-03,2020-03,2024-05,2018-01,2020-08,2020-06,2019-10,2022-05,2021-04,2024-01,2024-07,2018-10,2024-04,2019-01,2019-06,2018-05,2022-06,2022-07,2019-11,2019-09,2018-09,2020-07,2018-11,2021-01,2019-12,2020-02,2018-08,2018-04,2020-01,2024-02,2018-02,2018-07,2023-12,2018-12,2018-06,2023-04,2020-09,2020-12,2018-03,2024-08,2023-02,2022-09,2021-02,2024-10,2020-10,2023-03,2024-09,2023-05,2022-12,2022-03,2022-10,2022-11,2021-03,2021-05,2023-07,2023-09,2023-06,2023-08,2023-10,2021-12,2022-01,2023-11,2020-11,2022-08,2021-09,2020-05,2024-11,2021-11,2022-02,2025-03,2025-06,2025-04,2025-01,2024-12,2021-07,2025-10,2025-07,2025-08,2025-12,2025-09,2025-05,2021-10,2021-08,2021-06,2025-02,2023-01,2025-11
Comparendos,1990,1894,1644,1636,1626,1591,1578,1527,1524,1492,1488,1482,1444,1436,1393,1385,1384,1363,1354,1339,1334,1329,1326,1324,1319,1304,1293,1289,1272,1271,1270,1252,1227,1207,1200,1196,1178,1161,1156,1149,1124,1119,1118,1110,1109,1102,1097,1090,1090,1089,1082,1077,1076,1067,1059,1054,1041,1035,1019,1013,991,958,943,936,934,927,895,892,876,865,860,849,811,724,708,706,687,672,672,660,656,628,618,607,606,593,592,586,576,571,566,551,511,475,425,383
Porcentaje,1.94,1.84,1.6,1.59,1.58,1.55,1.54,1.49,1.48,1.45,1.45,1.44,1.41,1.4,1.36,1.35,1.35,1.33,1.32,1.3,1.3,1.29,1.29,1.29,1.28,1.27,1.26,1.26,1.24,1.24,1.24,1.22,1.19,1.18,1.17,1.16,1.15,1.13,1.13,1.12,1.09,1.09,1.09,1.08,1.08,1.07,1.07,1.06,1.06,1.06,1.05,1.05,1.05,1.04,1.03,1.03,1.01,1.01,0.99,0.99,0.96,0.93,0.92,0.91,0.91,0.9,0.87,0.87,0.85,0.84,0.84,0.83,0.79,0.7,0.69,0.69,0.67,0.65,0.65,0.64,0.64,0.61,0.6,0.59,0.59,0.58,0.58,0.57,0.56,0.56,0.55,0.54,0.5,0.46,0.41,0.37


Distribución de Comparendos Electrónicos por Mes - Clase: VEHÍCULOS PESADOS


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95
Mes,2022-12,2022-11,2022-06,2023-12,2019-07,2019-02,2024-07,2019-08,2022-05,2023-06,2024-09,2019-05,2019-03,2023-07,2024-04,2022-09,2024-10,2022-04,2025-10,2023-08,2019-04,2022-07,2018-11,2023-05,2023-10,2023-09,2023-03,2022-10,2024-05,2024-06,2019-06,2024-03,2023-11,2024-08,2023-04,2018-01,2018-05,2022-08,2023-02,2020-10,2024-11,2025-04,2024-02,2024-01,2019-09,2018-06,2025-09,2019-01,2025-05,2020-02,2018-12,2025-07,2018-10,2018-03,2018-04,2019-12,2018-09,2018-02,2020-08,2019-10,2020-03,2025-03,2018-07,2024-12,2022-03,2018-08,2020-11,2021-04,2020-01,2019-11,2025-06,2020-12,2021-03,2021-01,2025-08,2025-02,2021-11,2020-09,2021-02,2025-01,2020-04,2025-11,2022-01,2021-12,2021-10,2021-09,2025-12,2021-07,2023-01,2020-06,2020-07,2022-02,2021-05,2021-08,2021-06,2020-05
Comparendos,491,382,345,330,326,325,317,316,316,308,304,300,298,295,295,284,283,283,280,275,275,272,272,271,267,266,265,264,264,259,259,249,249,246,237,227,222,213,207,207,205,204,204,197,196,195,193,193,187,187,187,187,185,182,179,177,173,172,169,166,165,163,162,161,159,158,157,157,155,154,152,152,145,144,142,141,138,137,136,134,133,129,126,124,113,112,111,111,109,100,97,91,90,89,64,43
Porcentaje,2.49,1.94,1.75,1.67,1.65,1.65,1.61,1.6,1.6,1.56,1.54,1.52,1.51,1.49,1.49,1.44,1.43,1.43,1.42,1.39,1.39,1.38,1.38,1.37,1.35,1.35,1.34,1.34,1.34,1.31,1.31,1.26,1.26,1.25,1.2,1.15,1.12,1.08,1.05,1.05,1.04,1.03,1.03,1.0,0.99,0.99,0.98,0.98,0.95,0.95,0.95,0.95,0.94,0.92,0.91,0.9,0.88,0.87,0.86,0.84,0.84,0.83,0.82,0.82,0.81,0.8,0.8,0.8,0.79,0.78,0.77,0.77,0.73,0.73,0.72,0.71,0.7,0.69,0.69,0.68,0.67,0.65,0.64,0.63,0.57,0.57,0.56,0.56,0.55,0.51,0.49,0.46,0.46,0.45,0.32,0.22


Distribución de Comparendos Electrónicos por Mes - Clase: OTROS


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72
Mes,2024-05,2019-04,2021-01,2022-11,2024-02,2019-07,2021-02,2022-03,2024-06,2023-05,2025-01,2025-02,2024-03,2025-04,2021-03,2023-11,2019-05,2018-01,2022-04,2024-04,2023-04,2025-05,2025-10,2019-08,2019-09,2020-12,2019-10,2022-08,2022-05,2023-09,2021-10,2022-10,2018-12,2025-07,2025-06,2024-10,2024-07,2022-01,2022-02,2020-06,2019-11,2021-08,2020-09,2020-02,2019-12,2020-10,2021-09,2023-03,2022-06,2023-12,2019-02,2018-10,2019-01,2018-03,2018-05,2018-02,2018-06,2021-04,2021-11,2019-03,2020-08,2019-06,2023-06,2023-07,2023-08,2023-10,2024-01,2024-08,2024-12,2025-03,2025-08,2025-09,2025-12
Comparendos,11,8,6,5,5,4,4,4,4,4,4,4,4,4,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
Porcentaje,6.08,4.42,3.31,2.76,2.76,2.21,2.21,2.21,2.21,2.21,2.21,2.21,2.21,2.21,1.66,1.66,1.66,1.66,1.66,1.66,1.66,1.66,1.66,1.66,1.66,1.66,1.66,1.66,1.66,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55


**Posible explicación para los puntos máximos en las clases (Automovil, camioneta y campero) - Períodos 2019-05 y 2019-06:** 

Los puntos máximos registrados en las infracciones cometidas por automóviles, camionetas y camperos durante ciertos meses pueden explicarse por los múltiples cierres viales que se presentaron en Barranquilla durante esos períodos. Los cierres viales generan alteraciones significativas en los patrones normales de circulación, desviando el tránsito hacia vías alternas y creando condiciones de congestión que incrementan la probabilidad de comisión de infracciones. Estos tres tipos de vehículos (automóviles, camionetas y camperos), al ser los más numerosos en el parque automotor y los que más circulan por las vías urbanas, son los más afectados por estas interrupciones viales. Ante los desvíos inesperados, las demoras y la falta de señalización temporal, los conductores de estas clases tienden a cometer con mayor frecuencia infracciones como exceso de velocidad (al intentar recuperar tiempo perdido en rutas alternas), estacionamiento en sitios prohibidos (al buscar alternativas de parqueo cerca de sus destinos), paso de semáforos en rojo (por impaciencia ante la congestión) y bloqueo de calzadas (al quedar detenidos en intersecciones). Dado que automóviles, camionetas y camperos representan en conjunto el 90.25% del total de infracciones, su comportamiento es el principal determinante de los picos generales en la serie temporal, reflejando directamente las condiciones anormales de movilidad generadas por los cierres viales en la ciudad.

**Fuentes**:
- https://barranquilla.gov.co/transito/por-trabajos-cierres-de-vias-a-partir-del-viernes-3-de-mayo
- https://barranquilla.gov.co/transito/por-ampliacion-de-la-calle-30-cierres-de-vias-a-partir-del-martes-14-de-mayo-de-2019
- https://barranquilla.gov.co/transito/por-proyecto-mall-plaza-cierres-de-vias-a-partir-del-martes-21-de-mayo-de-2019
- https://barranquilla.gov.co/transito/por-reparacion-de-redes-cierres-de-vias-a-partir-del-martes-28-de-mayo
- https://barranquilla.gov.co/transito/cierres-de-vias-para-el-sabado-1-y-domingo-2-de-junio-de-20
- https://barranquilla.gov.co/transito/por-reparacion-de-pavimento-cierres-de-vias-a-partir-del-miercoles-5-de-junio-de-2019
- https://barranquilla.gov.co/transito/por-reparacion-de-pavimento-cierres-de-vias-para-el-martes-18-de-junio-de-2019
- https://barranquilla.gov.co/transito/cierres-de-vias-a-partir-del-viernes-21-de-junio-de-2019

---

**Punto máximo de la clase motocicletas en abril de 2020**: 

El punto máximo registrado en las infracciones cometidas por motocicletas durante abril de 2020 representa un comportamiento atípico dentro del análisis, ya que contrasta significativamente con lo observado en las demás clases de vehículos. Mientras que automóviles, camionetas, camperos y vehículos pesados presentaron sus valores más bajos durante el período de confinamiento estricto por la pandemia de COVID-19, las motocicletas mostraron un pico inusualmente alto en abril de ese mismo año. 

---

**Posible explicación para el punto máximo en la clase vehículos pesados - período 2022-12:** 

El punto máximo registrado en las infracciones cometidas por vehículos pesados coincide temporalmente con el punto máximo del servicio público, lo cual no es casualidad dada la estrecha relación entre ambas categorías. Los vehículos pesados (camión, bus, microbús, buseta, volqueta y tractocamión) son predominantemente de servicio público, como se evidenció en el análisis de relación entre clase y servicio. Por lo tanto, el pico en vehículos pesados es un reflejo directo del comportamiento del servicio público. Este punto máximo puede explicarse por la combinación de varios factores. En primer lugar, algunos cierres viales ocurridos durante ese mes pudieron derivar a los conductores de vehículos pesados a manejar en zonas donde operan las cámaras de detección, incrementando su exposición a la vigilancia. En segundo lugar, la Secretaría de Tránsito y Seguridad Vial del Área Metropolitana de Barranquilla sancionó a 40 conductores y propietarios de vehículos que prestaban un servicio no autorizado, lo que pudo generar una limitación en la circulación de conductores de aplicaciones como InDrive o Uber, reduciendo la competencia y permitiendo que los vehículos de servicio público autorizado (incluyendo los pesados) tuvieran una mayor demanda y, por tanto, una mayor exposición en las calles. Por último, este comportamiento se combinó con la época de diciembre, que es un período de alta productividad y movimiento para los vehículos de servicio público debido al aumento de la demanda por compras, reuniones familiares y eventos de fin de año.

**Fuentes**:
- https://barranquilla.gov.co/transito/por-carrera-atletica-cierres-de-vias-para-el-sabado-24-diciembre
- https://barranquilla.gov.co/mi-barranquilla/amb-sanciona-conductores-y-propietarios-de-vehiculos-que-prestaban-un-servicio-no-autorizado
- https://barranquilla.gov.co/transito/por-evento-gran-parada-de-luz-cierre-de-vias-para-el-jueves-8-de-diciembre-de-2022

---

**Posible explicación para el punto máximo en la clase otros - período 2024-05:** 

El punto máximo registrado en las infracciones cometidas por la clase OTROS (que incluye motocarro, cuatrimoto, no reportado y sin clase) durante cierto mes puede explicarse por la combinación de múltiples factores. En primer lugar, los múltiples cierres viales ocurridos durante ese período generaron alteraciones en los patrones normales de circulación, desviando el tránsito hacia vías alternas y creando condiciones de congestión que incrementaron la exposición de estos vehículos a las cámaras de detección. En segundo lugar, la implementación de laboratorios viales durante ese mes, los cuales implican simulaciones, pruebas de infraestructura y cambios temporales en la señalización y el flujo vehicular, pudo haber generado confusión entre los conductores de estas clases de vehículos, aumentando la probabilidad de cometer infracciones. Por último, la temporada de fuertes lluvias que coincidió con este período agravó aún más la situación, ya que las precipitaciones intensas reducen la visibilidad, generan encharcamientos, deterioran la señalización vial y afectan la fluidez del tránsito, condiciones que favorecen la comisión de infracciones por parte de conductores que buscan sortear los obstáculos y las demoras. 

**Fuentes:**

- https://barranquilla.gov.co/mi-barranquilla/sin-descanso-distrito-mantiene-limpieza-de-arroyos-para-prevenir-emergencias-durante-epoca-de-lluvias
- https://barranquilla.gov.co/transito/por-trabajos-de-alcantarillados-cierres-de-vias-para-el-sabado-4-y-lunes-6-de-mayo-de-2024
- https://barranquilla.gov.co/transito/continua-contraflujo-del-laboratorio-vial-en-la-carrera-59-entre-calles-79-y-98
- https://barranquilla.gov.co/transito/a-partir-del-jueves-2-de-mayo-se-implementa-laboratorio-vial-en-la-carrera-59-entre-calles-79-y-98
- https://barranquilla.gov.co/transito/por-obras-en-la-avenida-cordialidad-cierres-de-vias-a-partir-del-viernes-10-de-mayo-de-2024
- https://barranquilla.gov.co/transito/por-instalacion-de-alcantarillado-y-resane-de-pavimento-cierres-de-vias-a-partir-del-martes-14-de-mayo
- https://barranquilla.gov.co/salud/alcaldia-medidas-prevencion-enfermedades-aumentan-lluvias
- https://barranquilla.gov.co/transito/cierres-de-vias-a-partir-del-sabado-18-de-mayo

---

**Explicación de los puntos mínimos en las clases (Automovil, camioneta, campero y vehículos pesados) - período 2020-05:** 

Los puntos mínimos registrados en las infracciones cometidas por automóviles, camionetas, camperos y vehículos pesados durante ciertos períodos se explican principalmente por las estrictas medidas de restricción de movilidad implementadas durante la pandemia de COVID-19. Estas medidas afectaron de manera directa a estas cuatro clases de vehículos, que en conjunto representan la gran mayoría del parque automotor que circula habitualmente por Barranquilla. Es importante destacar que, a diferencia de las motocicletas que presentaron un comportamiento atípico con un pico durante la pandemia, los automóviles, camionetas, camperos y vehículos pesados redujeron su movilidad de manera considerable durante el confinamiento, reflejándose en los valores mínimos observados en sus series temporales durante los meses más críticos de 2020.

**Fuente**: https://barranquilla.gov.co/coronavirus/60-dias-de-crisis-covid19-barranquilla

### Distribución Mensual y Anual de Comparendos Electrónicos por Clase del Vehículo Infractor

Se construyen tablas pivote que cruzan los meses y los años para cada clase del vehículo (AUTOMÓVIL, CAMIONETA, MOTOCICLETA, CAMPERO, VEHÍCULOS PESADOS y OTROS), utilizando la suma ponderada de `CANTIDAD_INFRACCIONES` para visualizar el volumen de infracciones en cada combinación temporal. Los resultados se presentan mediante mapas de calor independientes para cada clase, con los meses ordenados de diciembre a enero para facilitar la identificación de patrones estacionales específicos de cada tipo de vehículo. Esta visualización permite comparar el comportamiento temporal de las diferentes clases, identificar períodos de alta o baja actividad particulares de cada tipo de vehículo, detectar estacionalidades diferenciadas (como picos en ciertos meses para automóviles vs motocicletas), y evaluar cómo eventos externos como la pandemia o los cierres viales afectaron de manera distinta a cada clase del parque automotor.

In [31]:
clases_heatmap = ['AUTOMOVIL', 'CAMIONETA', 'MOTOCICLETA', 'CAMPERO', 'VEHÍCULOS PESADOS', 'OTROS']

nombres_meses = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 
                 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']

for clase in clases_heatmap:
    df_clase = df_comparendos_electronicos_copy[df_comparendos_electronicos_copy['CLASE_AGRUPADA'] == clase]
    
    df_clase['año'] = df_clase['fecha_comparendo'].dt.year
    df_clase['mes_numero'] = df_clase['fecha_comparendo'].dt.month
    
    tabla_heatmap = df_clase.pivot_table(
        values='CANTIDAD_INFRACCIONES',
        index='mes_numero',
        columns='año',
        aggfunc='sum',
        fill_value=0
    )
    
    tabla_heatmap.index = nombres_meses
    tabla_heatmap = tabla_heatmap.iloc[::-1]

    fig = go.Figure(data=go.Heatmap(
        z=tabla_heatmap.values,
        x=tabla_heatmap.columns,
        y=tabla_heatmap.index,
        colorscale='blues',
        text=tabla_heatmap.values,
        texttemplate='%{text:,.0f}',
        hovertemplate=f'Clase: {clase}<br>Mes: %{{y}}<br>Año: %{{x}}<br>Comparendos: %{{z:,.0f}}<extra></extra>'
    ))
    
    fig.update_layout(
        title=dict(
            text=f'Distribución de Comparendos Electrónicos por Año y Mes - Clase: {clase}',
            x=0.5,
            xanchor='center',
            font=dict(size=16, weight='bold')
        ),
        xaxis_title=dict(
            text='Año',
            font=dict(weight='bold')
        ),
        yaxis_title=dict(
            text='Mes',
            font=dict(weight='bold')
        ),
        template='plotly_white',
        width=1055, 
        height=500
    )
    
    fig.show()

### Distribución Mensual de Comparendos Electrónicos por Clase del Vehículo Infractor

Se examina la variabilidad mensual de los comparendos electrónicos desagregada por cada clase del vehículo (AUTOMÓVIL, CAMIONETA, MOTOCICLETA, CAMPERO, VEHÍCULOS PESADOS y OTROS), agrupando los registros por mes y año y sumando la columna `CANTIDAD_INFRACCIONES` para obtener el volumen real de infracciones por cada combinación. Para cada clase, se presentan estadísticos descriptivos (media, desviación estándar, mínimo, máximo y cuartiles) para cada mes, con el fin de caracterizar su comportamiento típico y su dispersión interanual de manera específica por tipo de vehículo. Complementariamente, se generan diagramas de caja (boxplot) que visualizan la distribución de las infracciones para cada mes, incluyendo la media como referencia. Para validar si las diferencias observadas entre meses son estadísticamente significativas para cada clase, se aplica un análisis de varianza que incluye la prueba de normalidad de Shapiro-Wilk para evaluar la distribución de los datos por mes, la prueba de Levene para verificar la homogeneidad de varianzas, y dependiendo del cumplimiento de los supuestos, se realiza un ANOVA paramétrico (con prueba post-hoc de Tukey HSD) o un Kruskal-Wallis no paramétrico (con prueba post-hoc de Dunn y corrección de Bonferroni). Este análisis permite identificar, de manera diferenciada por clase de vehículo, qué meses presentan mayor variabilidad en la detección de infracciones para cada tipo de vehículo, detectar meses con comportamientos atípicos específicos de cada clase, comparar la estabilidad estacional entre los distintos tipos de vehículos (automóviles, camionetas, motocicletas, camperos, vehículos pesados y otros), y determinar si existe estacionalidad estadísticamente significativa en el volumen de comparendos para cada clase del parque automotor.

In [32]:
clases_boxplot = ['AUTOMOVIL', 'CAMIONETA', 'MOTOCICLETA', 'CAMPERO', 'VEHÍCULOS PESADOS', 'OTROS']

for clase in clases_boxplot:
    df_clase = df_comparendos_electronicos_copy[df_comparendos_electronicos_copy['CLASE_AGRUPADA'] == clase]
    
    df_boxplot_clase = df_clase.groupby(['mes_nombre', 'año'])['CANTIDAD_INFRACCIONES'].sum().reset_index()
    df_boxplot_clase.columns = ['mes', 'año', 'infracciones']
    df_boxplot_clase['mes_nombre'] = df_boxplot_clase['mes'].apply(lambda x: nombres_meses[x-1])
    
    print(f"Estadísticas Descriptivas - Clase: {clase}")
    estadisticas = df_boxplot_clase.groupby('mes_nombre')['infracciones'].describe().round(0).fillna(0).astype(int).reindex(nombres_meses)
    display(estadisticas.T)
    
    display(HTML("<hr style='border: 1px solid #696969; margin: 20px 0;'>"))
    
    fig = go.Figure()
    
    fig.add_trace(go.Box(
        x=df_boxplot_clase['mes_nombre'],
        y=df_boxplot_clase['infracciones'],
        marker_color='mediumpurple',
        name='',
        boxmean=True,
        hovertemplate='Mes: %{x}<br>Comparendos: %{y:,.0f}<extra></extra>'
    ))
    
    fig.update_layout(
        title=dict(
            text=f'Distribución de Comparendos Electrónicos por Mes - Clase: {clase}',
            x=0.5,
            xanchor='center',
            font=dict(size=16, weight='bold')
        ),
        xaxis_title=dict(
            text='Mes',
            font=dict(weight='bold')
        ),
        yaxis_title=dict(
            text='Cantidad de Comparendos',
            font=dict(weight='bold')
        ),
        template='plotly_white',
        width=1055,
        height=500
    )
    
    fig.show()
    
    df_anova = df_boxplot_clase.copy()
    
    meses_grupos = [
        df_anova[df_anova['mes_nombre'] == mes]['infracciones'].values 
        for mes in nombres_meses
    ]
    
    print("Prueba de normalidad (Shapiro-Wilk) por mes:")
    
    normalidad = []
    for mes, grupo in zip(nombres_meses, meses_grupos):
        if len(grupo) >= 3:  
            stat, p = stats.shapiro(grupo)
            normalidad.append(p > 0.05)
            print(f"{mes}: p-valor = {p:.6f} -> {'Normal' if p > 0.05 else 'No normal'}")
        else:
            print(f"{mes}: No hay suficientes datos para evaluar normalidad")
            normalidad.append(False)
    
    cumple_normalidad = all(normalidad) if normalidad else False
    print(f"¿Se cumple normalidad en todos los meses? {'Sí' if cumple_normalidad else 'No'}")
    
    grupos_validos = [g for g in meses_grupos if len(g) > 0]
    
    if len(grupos_validos) > 1:
        stat_levene, p_levene = stats.levene(*grupos_validos)
        print(f"\nLevene p-valor: {p_levene:.6f}")
        print(f"¿Varianzas iguales? {'Sí' if p_levene > 0.05 else 'No'}")
        
        if cumple_normalidad and p_levene > 0.05:
            print("\nSe cumplen los supuestos → Se aplica ANOVA")
            
            f_stat, p_valor = stats.f_oneway(*meses_grupos)
            
            print(f"Estadístico F: {f_stat:.4f}")
            print(f"P-valor: {p_valor:.6f}")
            
            print(f"¿Hay diferencias entre meses? {'Sí' if p_valor < 0.05 else 'No'}")
            
            if p_valor < 0.05:
                print("\nPrueba post-hoc Tukey HSD:")
                
                tukey = pairwise_tukeyhsd(
                    endog=df_anova['infracciones'],
                    groups=df_anova['mes_nombre'],
                    alpha=0.05
                )
                
                print(tukey)
        
        else:
            print("\nNo se cumplen los supuestos → Se aplica Kruskal-Wallis")
            
            h_stat, p_valor = stats.kruskal(*grupos_validos)
            
            print(f"Estadístico H: {h_stat:.4f}")
            print(f"P-valor: {p_valor:.6f}")
            
            print(f"¿Hay diferencias entre meses? {'Sí' if p_valor < 0.05 else 'No'}")
            
            if p_valor < 0.05:
                print("\nPrueba post-hoc Dunn (con corrección Bonferroni):")
                
                import scikit_posthocs as sp
                
                df_dunn = df_anova[['mes_nombre', 'infracciones']].copy()
                
                dunn = sp.posthoc_dunn(
                    df_dunn,
                    val_col='infracciones',
                    group_col='mes_nombre',
                    p_adjust='bonferroni'
                )
                
                dunn = dunn.loc[nombres_meses, nombres_meses]
                display(dunn)
    
    display(HTML("<hr style='border: 2px solid #696969; margin: 20px 0;'>"))

Estadísticas Descriptivas - Clase: AUTOMOVIL


mes_nombre,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
count,8,8,8,8,8,8,8,8,8,8,8,8
mean,3058,3217,3612,3391,3345,3334,3430,3066,3317,3658,3386,3430
std,1038,664,828,1139,1609,1554,1368,1139,893,705,764,939
min,999,1974,2744,1503,430,953,934,1649,2179,2187,2097,2265
25%,2750,2854,2799,2698,2338,2047,2614,2102,2617,3550,2999,2724
50%,3050,3503,3658,3596,3934,3724,3638,3220,3397,3732,3226,3380
75%,3789,3701,4141,4078,4383,4610,4337,3642,3707,3859,4132,4026
max,4342,3817,4915,4905,5212,5175,5050,4751,4892,4701,4306,4770


Prueba de normalidad (Shapiro-Wilk) por mes:
Enero: p-valor = 0.555302 -> Normal
Febrero: p-valor = 0.116620 -> Normal
Marzo: p-valor = 0.191241 -> Normal
Abril: p-valor = 0.903985 -> Normal
Mayo: p-valor = 0.458931 -> Normal
Junio: p-valor = 0.437994 -> Normal
Julio: p-valor = 0.675391 -> Normal
Agosto: p-valor = 0.515785 -> Normal
Septiembre: p-valor = 0.822531 -> Normal
Octubre: p-valor = 0.159112 -> Normal
Noviembre: p-valor = 0.417717 -> Normal
Diciembre: p-valor = 0.606611 -> Normal
¿Se cumple normalidad en todos los meses? Sí

Levene p-valor: 0.248574
¿Varianzas iguales? Sí

Se cumplen los supuestos → Se aplica ANOVA
Estadístico F: 0.2201
P-valor: 0.995730
¿Hay diferencias entre meses? No


Estadísticas Descriptivas - Clase: CAMIONETA


mes_nombre,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
count,8,8,8,8,8,8,8,8,8,8,8,8
mean,1230,1273,1409,1338,1345,1325,1371,1279,1362,1516,1372,1409
std,371,265,305,482,633,567,522,427,418,291,267,348
min,455,738,982,597,157,399,358,721,853,995,878,984
25%,1098,1171,1221,1086,1164,994,1178,959,1091,1401,1281,1230
50%,1332,1308,1396,1370,1399,1458,1350,1214,1335,1464,1380,1346
75%,1501,1456,1648,1564,1736,1756,1791,1665,1518,1730,1522,1500
max,1574,1546,1818,2009,2091,1973,1992,1883,2076,1942,1755,2043


...

mes_nombre,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
count,8,8,8,8,8,8,8,8,8,8,8,8
mean,161,183,203,220,212,210,221,201,208,221,211,217
std,41,69,58,61,99,100,93,75,70,63,87,130
min,109,91,145,133,43,64,97,89,112,113,129,111
25%,132,140,162,174,163,139,149,154,164,180,150,145
50%,150,180,174,220,243,227,230,191,194,236,181,169
75%,194,205,253,277,278,271,300,253,270,270,255,223
max,227,325,298,295,316,345,326,316,304,283,382,491


Prueba de normalidad (Shapiro-Wilk) por mes:
Enero: p-valor = 0.633324 -> Normal
Febrero: p-valor = 0.404832 -> Normal
Marzo: p-valor = 0.094767 -> Normal
Abril: p-valor = 0.506567 -> Normal
Mayo: p-valor = 0.273880 -> Normal
Junio: p-valor = 0.782199 -> Normal
Julio: p-valor = 0.230036 -> Normal
Agosto: p-valor = 0.957678 -> Normal
Septiembre: p-valor = 0.591216 -> Normal
Octubre: p-valor = 0.242042 -> Normal
Noviembre: p-valor = 0.173427 -> Normal
Diciembre: p-valor = 0.015837 -> No normal
¿Se cumple normalidad en todos los meses? No

Levene p-valor: 0.673985
¿Varianzas iguales? Sí

No se cumplen los supuestos → Se aplica Kruskal-Wallis
Estadístico H: 5.3341
P-valor: 0.913915
¿Hay diferencias entre meses? No


Estadísticas Descriptivas - Clase: OTROS


mes_nombre,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
count,6,7,7,6,6,7,4,7,5,8,4,6
mean,3,3,2,4,4,2,2,2,2,2,3,2
std,2,2,1,2,3,1,1,1,1,1,2,1
min,1,1,1,1,1,1,1,1,1,1,1,1
25%,1,2,1,3,3,1,2,1,2,2,2,1
50%,2,2,2,3,3,2,2,1,2,2,2,2
75%,4,4,4,4,4,2,2,2,2,2,4,2
max,6,5,4,8,11,4,4,3,3,3,5,3


Prueba de normalidad (Shapiro-Wilk) por mes:
Enero: p-valor = 0.452205 -> Normal
Febrero: p-valor = 0.224390 -> Normal
Marzo: p-valor = 0.063730 -> Normal
Abril: p-valor = 0.121785 -> Normal
Mayo: p-valor = 0.015952 -> No normal
Junio: p-valor = 0.026145 -> No normal
Julio: p-valor = 0.406387 -> Normal
Agosto: p-valor = 0.008193 -> No normal
Septiembre: p-valor = 0.325430 -> Normal
Octubre: p-valor = 0.092878 -> Normal
Noviembre: p-valor = 0.849971 -> Normal
Diciembre: p-valor = 0.211705 -> Normal
¿Se cumple normalidad en todos los meses? No

Levene p-valor: 0.687965
¿Varianzas iguales? Sí

No se cumplen los supuestos → Se aplica Kruskal-Wallis
Estadístico H: 10.3374
P-valor: 0.500316
¿Hay diferencias entre meses? No


**Clase AUTOMÓVIL (321,953 infracciones)**

- Los automóviles presentan su mayor actividad en octubre y marzo, mientras que enero y agosto son los meses de menor volumen. La variabilidad es extremadamente alta en mayo, junio y julio, influenciada por los picos de 2019 y los valles de 2020.

- Todos los meses cumplen con normalidad (p-valores > 0.05, el más bajo fue febrero con 0.1166) y las varianzas son homogéneas (p=0.2486). El ANOVA arroja un estadístico F de 0.2201 con un p-valor de 0.9957. **No existen diferencias estadísticamente significativas entre las medias de los meses** para los automóviles.

**Clase CAMIONETA (129,827 infracciones)**

- Similar a los automóviles, las camionetas tienen su pico en octubre y muestran alta variabilidad en mayo, junio y julio.

- Todos los meses cumplen con normalidad (p-valores > 0.05) y las varianzas son homogéneas (p=0.2732). El ANOVA arroja un estadístico F de 0.2497 con un p-valor de 0.9926. **No existen diferencias estadísticamente significativas entre las medias de los meses** para las camionetas.

**Clase MOTOCICLETA (102,708 infracciones)**

- Las motocicletas presentan su pico máximo en abril, a diferencia de otras clases cuyo pico es en octubre o marzo. La variabilidad es más moderada en comparación con automóviles y camionetas.

- Todos los meses cumplen con normalidad (p-valores > 0.05, el más bajo fue diciembre con 0.0797, aún por encima del umbral) y las varianzas son homogéneas (p=0.9621). El ANOVA arroja un estadístico F de 0.9928 con un p-valor de 0.4598. **No existen diferencias estadísticamente significativas entre las medias de los meses** para las motocicletas.

**Clase CAMPERO (39,955 infracciones)**

- Los camperos tienen su pico en marzo y octubre, con variabilidad moderada y valores extremos en junio y julio.

- Todos los meses cumplen con normalidad (p-valores > 0.05) y las varianzas son homogéneas (p=0.6989). El ANOVA arroja un estadístico F de 0.0722 con un p-valor de 0.99998, prácticamente 1. **No existen diferencias estadísticamente significativas entre las medias de los meses** para los camperos.

**Clase VEHÍCULOS PESADOS (19,736 infracciones)**

- Los vehículos pesados tienen su pico en octubre y agosto y presentan su mínima actividad en enero.

- No se cumple el supuesto de normalidad en diciembre (p-valor = 0.0158 < 0.05), aunque los demás meses sí son normales. Las varianzas son homogéneas (p=0.6740). Dado que no se cumple la normalidad, se aplica la prueba no paramétrica de Kruskal-Wallis, que arroja un estadístico H de 5.3341 con un p-valor de 0.9139. **No existen diferencias estadísticamente significativas entre los meses** para los vehículos pesados.

**Clase OTROS (980 infracciones)** 

- La clase OTROS, que incluye motocarro, cuatrimoto, vehículos no reportados y sin clase, presenta un comportamiento particular debido a su bajo volumen de infracciones. Los meses con mayor actividad relativa son abril y mayo, con promedios de 4 comparendos cada uno, mientras que marzo, junio, julio, agosto, septiembre, octubre y diciembre presentan los promedios más bajos (2 comparendos). La variabilidad es baja en general, con desviaciones estándar que oscilan entre 1 y 3 comparendos. Cabe destacar que mayo registra el valor máximo absoluto (11 comparendos) y abril el segundo máximo (8 comparendos), lo que sugiere una ligera concentración de infracciones en estos meses. El número de años con registros varía considerablemente entre meses: octubre cuenta con 8 años de registro, mientras que julio solo tiene 4 años, lo que refleja la irregularidad en la operación o en la captura de datos para esta categoría.

- No se cumple el supuesto de normalidad en mayo (p-valor = 0.0160), junio (p-valor = 0.0261) y agosto (p-valor = 0.0082). Las varianzas son homogéneas (p=0.6880). Se aplica Kruskal-Wallis, que arroja un estadístico H de 10.3374 con un p-valor de 0.5003. **No existen diferencias estadísticamente significativas entre los meses** para la clase OTROS.

---

**Hallazgos:**

- **Estacionalidades diferenciadas**: Automóviles, camionetas y camperos tienen su pico en octubre, mientras que las motocicletas lo tienen en abril y los vehículos pesados en octubre. Esta diferencia refleja patrones de movilidad distintos según el tipo de vehículo.

- **Mayo como mes crítico para automóviles y camionetas**: Al igual que en análisis previos, mayo presenta la mayor variabilidad para estas clases, influenciada por los cierres viales de 2019 y la pandemia de 2020.

- **Comportamiento atípico de las motocicletas**: Su pico en abril de 2020 contrasta con el resto de clases que redujeron su actividad durante la pandemia, confirmando el comportamiento atípico identificado anteriormente.

- **Comportamiento de la clase OTROS**: A diferencia de las demás clases que presentan picos claramente definidos (octubre para automóviles, camionetas y camperos; abril para motocicletas), la clase OTROS muestra una actividad más dispersa, con una leve concentración en abril y mayo. Esto podría deberse a que estos vehículos (motocarros, cuatrimotos) tienen patrones de movilidad distintos, posiblemente asociados a actividades comerciales o recreativas que se intensifican en estos meses.

- **Contraste con otras clases**: Mientras que automóviles, camionetas y camperos tienen sus picos en octubre y marzo, y las motocicletas en abril, la clase OTROS no presenta un pico tan definido, lo que refleja que su comportamiento es más errático y menos predecible, consistente con su bajo volumen y la heterogeneidad de vehículos que la componen.

- Ninguna de las seis clases analizadas (AUTOMÓVIL, CAMIONETA, MOTOCICLETA, CAMPERO, VEHÍCULOS PESADOS y OTROS) presenta diferencias estadísticamente significativas en el volumen de comparendos a lo largo de los meses del año. Esto confirma que, independientemente del tipo de vehículo, los patrones mensuales se mantienen constantes sin que ningún mes destaque consistentemente por encima de los demás.

- Estos resultados complementan los análisis descriptivos donde se observaron picos en ciertos meses (octubre para automóviles y camionetas, abril para motocicletas). Sin embargo, al someter las diferencias mensuales a pruebas estadísticas rigurosas (ANOVA o Kruskal-Wallis), se confirma que las oscilaciones observadas entre meses no son lo suficientemente grandes ni consistentes a lo largo de los años como para ser consideradas estadísticamente significativas.

- VEHÍCULOS PESADOS (diciembre) y OTROS (mayo, junio, agosto) presentan meses donde no se cumple la normalidad. En el caso de OTROS, esto puede deberse al bajo volumen de infracciones (solo 980 en total), lo que genera distribuciones más erráticas y menos estables que en clases con mayor volumen. A pesar de ello, las pruebas no paramétricas confirman la ausencia de diferencias significativas.

- Con un p-valor de 0.99998, es la clase donde más claramente se rechaza cualquier diferencia estacional, indicando una uniformidad casi perfecta en el volumen de infracciones a lo largo de los meses.

---

**Notas:**

- Para reducir infracciones en automóviles, camionetas y camperos, se debe priorizar el control en octubre y marzo.
- Para motocicletas, se requiere especial atención en abril.
- Para la clase OTROS (motocarros, cuatrimotos, etc.), aunque el volumen es bajo, se sugiere prestar atención en abril y mayo, meses donde se concentra la mayor actividad relativa.
